# Scaling Crystallized Models

*Language modeling, quotient transfer, ontology acquisition, and symbolic certification*

This cleaned notebook preserves every nonempty experiment from `Untitled19.ipynb` in its original order. The archive was a sequence of standalone, single-cell programs rather than a top-to-bottom analysis. Each section below remains independently runnable after the setup cell.

## Reproducibility contract

- Original SHA-256: `de97b1c4df7ee3fd12417df9f1d61301d4ad44d61e97878852367da7a9090432`. Original cell indices are recorded in every code cell's metadata and section header.
- Recorded outputs were removed to reduce file size; all positive, negative, null, failed, and warning outcomes are preserved in the accompanying research record and website diary.
- Concrete `/content/...` and `outputs/...` paths were moved to unique `outputs/<experiment>/` directories. Algorithms, seeds, sample sizes, thresholds, and metrics are otherwise preserved except for the explicitly listed source repairs.
- Run one experiment section at a time. Several full settings require a T4/A100-class GPU or hours of CPU/GPU time; fast/dev environment variables are smoke tests, not replacements for registered full runs.
- The source archive is historical evidence, not a preregistration. An observed output may have been produced by an earlier source revision when the output and current cell disagree; such cases are called out in the research record.


In [ ]:
from pathlib import Path
import platform

_experiment_output_dirs = ['01-crystallized-nanogpt-v10-1', '02-quotient-mdl-active-learner-v11', '03-mdl-text8-scaling-v12-100-000-state-cap', '04-mdl-text8-scaling-v12-250-000-state-cap', '05-neural-quotient-active-learner-v13', '06-neural-quotient-active-learner-v13-1', '07-appended-older-v13-rerun-interrupted', '08-neural-quotient-v13-1-continuation', '09-ontology-free-active-learner-v14', '10-monotone-predictive-state-discovery-v16-core', '11-neural-counterexample-ranking-v16-1', '12-non-enumerable-entropy-ladder-v16-2', '13-v16-2-machine-readable-artifact-display']
for _experiment in _experiment_output_dirs:
    Path("outputs", _experiment).mkdir(parents=True, exist_ok=True)

print(f"Python {platform.python_version()}")
print(f"Artifact root: {Path('outputs').resolve()}")


## Experiment 1: Crystallized nanoGPT V10.1

Original cell `0`.


In [ ]:
"""
CRYSTALLIZED nanoGPT V10.1 — CALIBRATION-SELECTED, SINGLE-CELL T4 EXPERIMENT

Purpose
-------
Test the user's neural + crystallized-symbolic mechanism on the exact character-
level Tiny Shakespeare data and 90/10 split used by Karpathy's nanoGPT example.

Matched systems
---------------
1. nanoGPT-style causal Transformer.
2. Parameter-matched recurrent neural language model (GRU; no attention).
3. MDL context machine: a variable-order probabilistic finite-state machine.
4. Calibrated crystallized hybrid: GRU + context machine, with a gate fitted only
   on a held-out calibration tail from the official training partition.

What “crystallization” means here
---------------------------------
Each symbolic state is a character suffix. It is retained only when its measured
predictive gain pays its description-length penalty relative to the shorter suffix.
Its transition is exact: append the emitted character, then take the longest retained
suffix. Its emission is a calibrated distribution. This is a probabilistic symbolic
machine, because natural language is not a deterministic transition table.

Fairness
--------
- Same official text, vocabulary, and 90/10 train/validation split.
- The tail of the 90% training partition is calibration data and is unavailable to
  every learner's parameter/count fitting.
- Transformer and GRU see identical minibatches, token presentations, schedule
  shape, and evaluation batches. Their predeclared peak learning rates differ because
  the V10 GRU became overconfident and unstable under the Transformer's 1e-3 rate.
- The symbolic learner sees each fitted character once. Its learning curve is charged
  one observation per character; repeated neural minibatch tokens are charged again.
- The validation partition is never used to train models or choose the gate.
- Neural checkpoints are selected only on a dedicated slice of the calibration tail;
  the rest of that tail trains/selects/audits the gate, with no overlap.

Run modes
---------
On a CUDA runtime, the default uses Karpathy's published baby-GPT shape:
block=256, batch=64, 6 layers, 6 heads, width=384, dropout=.2. The calibrated
T4 protocol runs 2000 steps: the V10 follow-up showed both learners degrading
after their calibration-selected optima while consuming the remaining budget.
The GRU hidden width is selected to closely match its parameter count.

The T4 path keeps that effective batch but uses FP16 Tensor Cores, SDPA/Flash
attention, one physical batch of 64, fused AdamW, and GPU-resident corpus tensors.
Set CRYSTALLIZED_NANOGPT_V10_MICRO_BATCH_SIZE=16 if another notebook process has
claimed VRAM; set CRYSTALLIZED_NANOGPT_V10_COMPILE=0 if compilation is unavailable.
The compiler uses standard Inductor mode because PyTorch 2.11 CUDA graphs are
incompatible with this cell's gradient accumulation.

Set CRYSTALLIZED_NANOGPT_V10_FAST_DEV_RUN=1 for a short smoke test.
Set CRYSTALLIZED_NANOGPT_V10_OFFICIAL_SCALE=0 for the smaller research scale.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


FAST_DEV_RUN = os.environ.get("CRYSTALLIZED_NANOGPT_V10_FAST_DEV_RUN", "0") == "1"
OFFICIAL_SCALE_REQUESTED = os.environ.get("CRYSTALLIZED_NANOGPT_V10_OFFICIAL_SCALE", "1") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 1337
    data_url: str = (
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/"
        "data/tinyshakespeare/input.txt"
    )
    data_path: str = "outputs/01-crystallized-nanogpt-v10-1/tinyshakespeare.txt"
    output_dir: str = "outputs/01-crystallized-nanogpt-v10-1/crystallized_nanogpt_v10_1_results"
    calibration_characters: int = 50_000
    checkpoint_selection_characters: int = 10_000
    block_size: int = 256
    batch_size: int = 64  # effective batch; split into T4-safe microbatches
    micro_batch_size: int = 64
    evaluation_window_batch_size: int = 64
    # With one physical batch, reduce-overhead compilation is safe and fastest.
    # If a smaller microbatch is requested, the cell switches to standard mode;
    # CUDA graphs cannot reuse the same graph within one accumulated optimizer step.
    compile_transformer_training: bool = True
    max_steps: int = 2_000
    eval_batches: int = 40
    eval_checkpoints: Tuple[int, ...] = (0, 100, 250, 500, 1000, 1500, 2000)
    learning_rate: float = 1e-3
    min_learning_rate: float = 1e-4
    gru_learning_rate: float = 3e-4
    gru_min_learning_rate: float = 3e-5
    warmup_steps: int = 100
    weight_decay: float = 0.1
    dropout: float = 0.2
    transformer_layers: int = 6
    transformer_heads: int = 6
    transformer_width: int = 384
    gru_layers: int = 2
    gru_embedding: int = 384
    gru_hidden: int = 1024
    symbolic_max_order: int = 8
    symbolic_min_count: int = 3
    symbolic_max_states_per_order: int = 50_000
    symbolic_backoff_strength: float = 32.0
    symbolic_definition_nats: float = 2.0
    prefix_observations: Tuple[int, ...] = (
        10_000, 25_000, 50_000, 100_000, 250_000, 500_000
    )
    gate_train_fraction: float = 0.60
    gate_selection_fraction: float = 0.20
    gate_steps: int = 500
    generation_characters: int = 700
    temperature: float = 0.82
    top_k: int = 24


cfg = Config()
if not torch.cuda.is_available() or not OFFICIAL_SCALE_REQUESTED:
    cfg = replace(
        cfg,
        block_size=128,
        batch_size=32,
        micro_batch_size=16,
        evaluation_window_batch_size=32,
        compile_transformer_training=False,
        max_steps=2_000,
        eval_batches=20,
        eval_checkpoints=(0, 50, 100, 250, 500, 1000, 1500, 2000),
        warmup_steps=50,
        transformer_layers=4,
        transformer_heads=4,
        transformer_width=192,
        gru_embedding=192,
        gru_hidden=384,
    )
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        data_path=str(Path.cwd() / "work" / "tinyshakespeare" / "input.txt"),
        output_dir=str(Path.cwd() / "crystallized_nanogpt_v10_1_smoke"),
        calibration_characters=20_000,
        checkpoint_selection_characters=4_000,
        block_size=64,
        batch_size=16,
        micro_batch_size=16,
        evaluation_window_batch_size=16,
        compile_transformer_training=False,
        max_steps=1_000,
        eval_batches=5,
        eval_checkpoints=(0, 25, 50, 100, 250, 500, 750, 1000),
        warmup_steps=20,
        transformer_layers=2,
        transformer_heads=4,
        transformer_width=96,
        gru_embedding=96,
        gru_hidden=144,
        symbolic_max_states_per_order=20_000,
        prefix_observations=(10_000, 50_000, 200_000, 250_000, 500_000, 750_000),
        gate_steps=250,
        generation_characters=300,
    )
data_override = os.environ.get("CRYSTALLIZED_NANOGPT_V10_DATA_PATH")
output_override = os.environ.get("CRYSTALLIZED_NANOGPT_V10_OUTPUT_DIR")
micro_batch_override = os.environ.get("CRYSTALLIZED_NANOGPT_V10_MICRO_BATCH_SIZE")
compile_override = os.environ.get("CRYSTALLIZED_NANOGPT_V10_COMPILE")
if data_override:
    cfg = replace(cfg, data_path=data_override)
if output_override:
    cfg = replace(cfg, output_dir=output_override)
if micro_batch_override:
    cfg = replace(cfg, micro_batch_size=int(micro_batch_override))
if compile_override is not None:
    cfg = replace(cfg, compile_transformer_training=compile_override == "1")

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(True)
        torch.backends.cuda.enable_mem_efficient_sdp(True)
        torch.backends.cuda.enable_math_sdp(True)
torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = device.type == "cuda"
if cfg.batch_size % cfg.micro_batch_size != 0:
    raise RuntimeError("Effective batch size must be divisible by micro batch size")
gradient_accumulation_steps = cfg.batch_size // cfg.micro_batch_size
transformer_compile_mode = (
    "reduce-overhead" if gradient_accumulation_steps == 1 else "default"
)


def load_official_data() -> Tuple[str, np.ndarray, np.ndarray, np.ndarray, Dict[str, int], Dict[int, str]]:
    path = Path(cfg.data_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"downloading official Tiny Shakespeare -> {path}")
        urllib.request.urlretrieve(cfg.data_url, path)
    raw = path.read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    expected_digest = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
    if digest != expected_digest:
        raise RuntimeError(f"Tiny Shakespeare SHA-256 mismatch: {digest}")
    text = raw.decode("utf-8")
    if len(text) != 1_115_394:
        raise RuntimeError(
            f"Expected official 1,115,394-character corpus, found {len(text):,}."
        )
    characters = sorted(set(text))
    if len(characters) != 65:
        raise RuntimeError(f"Expected official 65-character vocabulary, found {len(characters)}.")
    stoi = {character: index for index, character in enumerate(characters)}
    itos = {index: character for character, index in stoi.items()}
    ids = np.asarray([stoi[character] for character in text], dtype=np.int64)
    split = int(0.9 * len(ids))
    official_train = ids[:split]
    official_validation = ids[split:]
    fit = official_train[: -cfg.calibration_characters]
    calibration = official_train[-cfg.calibration_characters :]
    return text, fit, calibration, official_validation, stoi, itos


@dataclass
class SymbolicLevel:
    codes: np.ndarray
    probabilities: np.ndarray
    totals: np.ndarray
    gains: np.ndarray


class MDLContextMachine:
    """Variable-order probabilistic automaton with MDL state admission."""

    def __init__(self, vocab_size: int) -> None:
        self.vocab_size = int(vocab_size)
        self.levels: Dict[int, SymbolicLevel] = {}
        self.global_probability = np.empty(0, dtype=np.float32)

    def _codes(self, data: np.ndarray, order: int) -> Tuple[np.ndarray, np.ndarray]:
        n = len(data) - order
        codes = np.zeros(n, dtype=np.uint64)
        base = np.uint64(self.vocab_size)
        for offset in range(order):
            codes = codes * base + data[offset : offset + n].astype(np.uint64)
        return codes, data[order:].astype(np.int64, copy=False)

    def _lookup(
        self, order: int, codes: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        n = len(codes)
        probabilities = np.repeat(self.global_probability[None, :], n, axis=0)
        supports = np.zeros(n, dtype=np.float32)
        used_orders = np.zeros(n, dtype=np.int16)
        unresolved = np.ones(n, dtype=bool)
        current_codes = codes.astype(np.uint64, copy=True)
        for current_order in range(order, 0, -1):
            level = self.levels.get(current_order)
            if level is not None and np.any(unresolved):
                rows = np.flatnonzero(unresolved)
                query = current_codes[rows]
                positions = np.searchsorted(level.codes, query)
                hit = positions < len(level.codes)
                safe = np.minimum(positions, max(0, len(level.codes) - 1))
                if len(level.codes):
                    hit &= level.codes[safe] == query
                hit_rows = rows[hit]
                hit_positions = positions[hit]
                probabilities[hit_rows] = level.probabilities[hit_positions]
                supports[hit_rows] = level.totals[hit_positions]
                used_orders[hit_rows] = current_order
                unresolved[hit_rows] = False
            if current_order > 1:
                current_codes %= np.uint64(self.vocab_size ** (current_order - 1))
        return probabilities, supports, used_orders

    def fit(self, data: np.ndarray, verbose: bool = False) -> "MDLContextMachine":
        data = np.asarray(data, dtype=np.int64)
        counts = np.bincount(data, minlength=self.vocab_size).astype(np.float64)
        self.global_probability = ((counts + 0.15) / (counts.sum() + 0.15 * self.vocab_size)).astype(
            np.float32
        )
        self.levels = {}
        for order in range(1, cfg.symbolic_max_order + 1):
            codes, targets = self._codes(data, order)
            unique_codes, totals = np.unique(codes, return_counts=True)
            eligible = totals >= cfg.symbolic_min_count
            unique_codes, totals = unique_codes[eligible], totals[eligible]
            if len(unique_codes) == 0:
                continue
            pre_cap = min(len(unique_codes), cfg.symbolic_max_states_per_order * 4)
            if len(unique_codes) > pre_cap:
                top = np.argpartition(totals, -pre_cap)[-pre_cap:]
                unique_codes, totals = unique_codes[top], totals[top]
                ordering = np.argsort(unique_codes)
                unique_codes, totals = unique_codes[ordering], totals[ordering]

            pair_keys = codes * np.uint64(self.vocab_size) + targets.astype(np.uint64)
            unique_pairs, pair_totals = np.unique(pair_keys, return_counts=True)
            pair_context = unique_pairs // np.uint64(self.vocab_size)
            pair_target = (unique_pairs % np.uint64(self.vocab_size)).astype(np.int64)
            positions = np.searchsorted(unique_codes, pair_context)
            hit = positions < len(unique_codes)
            safe = np.minimum(positions, len(unique_codes) - 1)
            hit &= unique_codes[safe] == pair_context
            dense = np.zeros((len(unique_codes), self.vocab_size), dtype=np.float32)
            np.add.at(
                dense,
                (positions[hit], pair_target[hit]),
                pair_totals[hit].astype(np.float32),
            )

            if order == 1:
                base = np.repeat(self.global_probability[None, :], len(unique_codes), axis=0)
            else:
                suffix_codes = unique_codes % np.uint64(self.vocab_size ** (order - 1))
                base, _, _ = self._lookup(order - 1, suffix_codes)
            probabilities = (dense + cfg.symbolic_backoff_strength * base) / (
                dense.sum(axis=1, keepdims=True) + cfg.symbolic_backoff_strength
            )
            gains = np.sum(
                dense
                * (
                    np.log(np.clip(probabilities, 1e-12, 1.0))
                    - np.log(np.clip(base, 1e-12, 1.0))
                ),
                axis=1,
            )
            nonzero = np.sum(dense > 0, axis=1)
            penalty = cfg.symbolic_definition_nats + 0.5 * np.maximum(nonzero - 1, 1) * np.log(
                totals + 1.0
            )
            keep = gains > penalty
            if not np.any(keep):
                continue
            kept = np.flatnonzero(keep)
            if len(kept) > cfg.symbolic_max_states_per_order:
                score = gains[kept] - penalty[kept]
                kept = kept[
                    np.argpartition(score, -cfg.symbolic_max_states_per_order)[
                        -cfg.symbolic_max_states_per_order :
                    ]
                ]
            kept = kept[np.argsort(unique_codes[kept])]
            self.levels[order] = SymbolicLevel(
                codes=unique_codes[kept],
                probabilities=probabilities[kept].astype(np.float32),
                totals=totals[kept].astype(np.float32),
                gains=gains[kept].astype(np.float32),
            )
            if verbose:
                print(
                    f" symbolic order={order} | states={len(kept):6,d} | "
                    f"median support={np.median(totals[kept]):6.1f}"
                )
        return self

    def predict_sequence(
        self, data: np.ndarray, reset_interval: int
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        probabilities_list: List[np.ndarray] = []
        supports_list: List[np.ndarray] = []
        orders_list: List[np.ndarray] = []
        targets_list: List[np.ndarray] = []
        for start in range(0, len(data) - 1, reset_interval):
            block = data[start : min(len(data), start + reset_interval + 1)]
            if len(block) < 2:
                continue
            n_targets = len(block) - 1
            probabilities = np.repeat(self.global_probability[None, :], n_targets, axis=0)
            supports = np.zeros(n_targets, dtype=np.float32)
            orders = np.zeros(n_targets, dtype=np.int16)
            for order in range(1, min(cfg.symbolic_max_order, len(block) - 1) + 1):
                codes, _ = self._codes(block, order)
                level = self.levels.get(order)
                if level is None:
                    continue
                positions = np.searchsorted(level.codes, codes)
                hit = positions < len(level.codes)
                safe = np.minimum(positions, len(level.codes) - 1)
                hit &= level.codes[safe] == codes
                rows = np.arange(order - 1, n_targets)[hit]
                probabilities[rows] = level.probabilities[positions[hit]]
                supports[rows] = level.totals[positions[hit]]
                orders[rows] = order
            probabilities_list.append(probabilities)
            supports_list.append(supports)
            orders_list.append(orders)
            targets_list.append(block[1:])
        return (
            np.concatenate(probabilities_list),
            np.concatenate(supports_list),
            np.concatenate(orders_list),
            np.concatenate(targets_list),
        )

    def predict_one(self, context: Sequence[int]) -> Tuple[np.ndarray, float, int]:
        usable = list(context)[-cfg.symbolic_max_order :]
        for order in range(min(len(usable), cfg.symbolic_max_order), 0, -1):
            code = np.uint64(0)
            for token in usable[-order:]:
                code = code * np.uint64(self.vocab_size) + np.uint64(token)
            level = self.levels.get(order)
            if level is None:
                continue
            position = int(np.searchsorted(level.codes, code))
            if position < len(level.codes) and level.codes[position] == code:
                return level.probabilities[position], float(level.totals[position]), order
        return self.global_probability, 0.0, 0

    @property
    def n_states(self) -> int:
        return 1 + sum(len(level.codes) for level in self.levels.values())


class CausalSelfAttention(nn.Module):
    def __init__(self, width: int, heads: int, dropout: float) -> None:
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.width = width
        self.dropout = dropout
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width, bias=False)

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        batch, length, width = hidden.shape
        q, k, v = self.qkv(hidden).chunk(3, dim=-1)
        q = q.view(batch, length, self.heads, width // self.heads).transpose(1, 2)
        k = k.view(batch, length, self.heads, width // self.heads).transpose(1, 2)
        v = v.view(batch, length, self.heads, width // self.heads).transpose(1, 2)
        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().view(batch, length, width)
        return self.projection(attended)


class TransformerBlock(nn.Module):
    def __init__(self, width: int, heads: int, dropout: float) -> None:
        super().__init__()
        self.norm_attention = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, dropout)
        self.norm_mlp = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, 4 * width, bias=False),
            nn.GELU(),
            nn.Linear(4 * width, width, bias=False),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        hidden = hidden + self.dropout(self.attention(self.norm_attention(hidden)))
        return hidden + self.mlp(self.norm_mlp(hidden))


class NanoGPT(nn.Module):
    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        width = cfg.transformer_width
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(cfg.block_size, width)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(width, cfg.transformer_heads, cfg.dropout)
                for _ in range(cfg.transformer_layers)
            ]
        )
        self.norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, vocab_size, bias=False)
        self.head.weight = self.token_embedding.weight
        self.apply(self._initialize)
        residual_std = 0.02 / math.sqrt(2 * cfg.transformer_layers)
        for block in self.blocks:
            nn.init.normal_(block.attention.projection.weight, mean=0.0, std=residual_std)
            nn.init.normal_(block.mlp[2].weight, mean=0.0, std=residual_std)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        length = tokens.shape[1]
        positions = torch.arange(length, device=tokens.device)
        hidden = self.dropout(self.token_embedding(tokens) + self.position_embedding(positions)[None])
        for block in self.blocks:
            hidden = block(hidden)
        return self.head(self.norm(hidden))


class RecurrentNeuralLM(nn.Module):
    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, cfg.gru_embedding)
        self.gru = nn.GRU(
            cfg.gru_embedding,
            cfg.gru_hidden,
            num_layers=cfg.gru_layers,
            dropout=cfg.dropout if cfg.gru_layers > 1 else 0.0,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(cfg.gru_hidden)
        self.head = nn.Linear(cfg.gru_hidden, vocab_size)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        logits, _ = self.step(tokens)
        return logits

    def step(
        self, tokens: torch.Tensor, state: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden, state = self.gru(self.embedding(tokens), state)
        return self.head(self.norm(hidden)), state


class DeviceSequence:
    """Small corpus resident on the accelerator; batches require no host transfer."""

    def __init__(self, data: np.ndarray) -> None:
        self.data = torch.as_tensor(data, dtype=torch.long, device=device)
        self.offsets = torch.arange(cfg.block_size, dtype=torch.long, device=device)

    def gather(self, starts: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        indices = starts[:, None] + self.offsets[None, :]
        return self.data[indices], self.data[indices + 1]

    def random_batch(
        self, generator: torch.Generator, batch_size: int
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        starts = torch.randint(
            0,
            len(self.data) - cfg.block_size - 1,
            (batch_size,),
            generator=generator,
        ).to(device, non_blocking=True)
        return self.gather(starts)


class ShuffledBlockStream:
    """Presents each non-overlapping training block once before reshuffling."""

    def __init__(self, data: np.ndarray, seed: int) -> None:
        self.sequence = DeviceSequence(data)
        self.generator = np.random.default_rng(seed)
        self.starts = np.arange(
            0,
            len(data) - cfg.block_size - 1,
            cfg.block_size,
            dtype=np.int64,
        )
        self.order = self.generator.permutation(len(self.starts))
        self.cursor = 0
        self.epochs = 0

    def next(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.cursor + cfg.batch_size > len(self.order):
            self.order = self.generator.permutation(len(self.starts))
            self.cursor = 0
            self.epochs += 1
        selected_cpu = self.starts[self.order[self.cursor : self.cursor + cfg.batch_size]]
        self.cursor += cfg.batch_size
        selected = torch.as_tensor(selected_cpu, dtype=torch.long, device=device)
        return self.sequence.gather(selected)


def learning_rate(step: int, maximum: float, minimum: float) -> float:
    if step < cfg.warmup_steps:
        return maximum * (step + 1) / cfg.warmup_steps
    ratio = min(1.0, (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps))
    coefficient = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return minimum + coefficient * (maximum - minimum)


@torch.inference_mode()
def evaluate_neural_models(
    models: Dict[str, nn.Module], source: DeviceSequence, seed: int, batches: Optional[int] = None
) -> Dict[str, float]:
    states = {name: model.training for name, model in models.items()}
    for model in models.values():
        model.eval()
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    losses = {name: [] for name in models}
    for _ in range(cfg.eval_batches if batches is None else batches):
        iteration_losses = {name: [] for name in models}
        for _micro_step in range(gradient_accumulation_steps):
            x, y = source.random_batch(generator, cfg.micro_batch_size)
            for name, model in models.items():
                with torch.autocast(
                    device_type=device.type, dtype=torch.float16, enabled=amp_enabled
                ):
                    logits = model(x)
                    loss = F.cross_entropy(
                        logits.reshape(-1, logits.shape[-1]), y.reshape(-1)
                    )
                iteration_losses[name].append(float(loss.item()))
        for name in models:
            losses[name].append(float(np.mean(iteration_losses[name])))
    for name, was_training in states.items():
        models[name].train(was_training)
    return {name: float(np.mean(values)) for name, values in losses.items()}


def train_neural_models(
    fit_data: np.ndarray,
    checkpoint_selection_data: np.ndarray,
    vocab_size: int,
) -> Tuple[Dict[str, nn.Module], List[Dict[str, float]], float, Dict[str, Dict[str, float]]]:
    models: Dict[str, nn.Module] = {
        "nanoGPT": NanoGPT(vocab_size).to(device),
        "recurrent-neural": RecurrentNeuralLM(vocab_size).to(device),
    }
    training_models = dict(models)
    if amp_enabled and cfg.compile_transformer_training and hasattr(torch, "compile"):
        print("compiling fixed-shape nanoGPT training graph (one-time cost)...")
        training_models["nanoGPT"] = torch.compile(
            models["nanoGPT"], mode=transformer_compile_mode, fullgraph=False
        )
    optimizers: Dict[str, torch.optim.Optimizer] = {}
    maximum_lrs = {
        "nanoGPT": cfg.learning_rate,
        "recurrent-neural": cfg.gru_learning_rate,
    }
    minimum_lrs = {
        "nanoGPT": cfg.min_learning_rate,
        "recurrent-neural": cfg.gru_min_learning_rate,
    }
    for name, model in models.items():
        decay = [parameter for parameter in model.parameters() if parameter.requires_grad and parameter.dim() >= 2]
        no_decay = [parameter for parameter in model.parameters() if parameter.requires_grad and parameter.dim() < 2]
        groups = [
            {"params": decay, "weight_decay": cfg.weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ]
        optimizer_kwargs = {
            "lr": maximum_lrs[name],
            "betas": (0.9, 0.99),
        }
        if amp_enabled:
            optimizer_kwargs["fused"] = True
        try:
            optimizers[name] = torch.optim.AdamW(groups, **optimizer_kwargs)
        except (TypeError, RuntimeError):
            optimizer_kwargs.pop("fused", None)
            optimizers[name] = torch.optim.AdamW(groups, **optimizer_kwargs)
    def new_scaler():
        try:
            return torch.amp.GradScaler(device.type, enabled=amp_enabled)
        except (AttributeError, TypeError):
            return torch.cuda.amp.GradScaler(enabled=amp_enabled)

    scalers = {name: new_scaler() for name in models}
    training_stream = ShuffledBlockStream(fit_data, cfg.seed + 10)
    selection_source = DeviceSequence(checkpoint_selection_data)
    checkpoint_set = set(cfg.eval_checkpoints)
    history: List[Dict[str, float]] = []
    best_selection = {name: float("inf") for name in models}
    best_steps = {name: 0 for name in models}
    best_states: Dict[str, Dict[str, torch.Tensor]] = {}
    if amp_enabled:
        torch.cuda.synchronize()
    started = time.perf_counter()

    for step in range(0, cfg.max_steps + 1):
        if step in checkpoint_set:
            selection_values = evaluate_neural_models(
                models,
                selection_source,
                cfg.seed + 2000 + step,
                batches=cfg.eval_batches,
            )
            for name in models:
                if selection_values[name] < best_selection[name]:
                    best_selection[name] = selection_values[name]
                    best_steps[name] = step
                    best_states[name] = {
                        key: value.detach().cpu().clone()
                        for key, value in models[name].state_dict().items()
                    }
                history.append(
                    {
                        "model": name,
                        "step": float(step),
                        "token_presentations": float(step * cfg.batch_size * cfg.block_size),
                        "checkpoint_selection_nll": selection_values[name],
                        "checkpoint_selection_bpc": selection_values[name] / math.log(2.0),
                    }
                )
            print(
                f"step {step:5d}/{cfg.max_steps} | tokens={step*cfg.batch_size*cfg.block_size:10,d} | "
                + " | ".join(
                    f"{name} select={selection_values[name]:.4f}"
                    for name in models
                )
            )
        if step == cfg.max_steps:
            break
        x, y = training_stream.next()
        for name, raw_model in models.items():
            model = training_models[name]
            raw_model.train()
            optimizer = optimizers[name]
            lr = learning_rate(step, maximum_lrs[name], minimum_lrs[name])
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.zero_grad(set_to_none=True)
            for micro_step in range(gradient_accumulation_steps):
                start = micro_step * cfg.micro_batch_size
                stop = start + cfg.micro_batch_size
                with torch.autocast(
                    device_type=device.type, dtype=torch.float16, enabled=amp_enabled
                ):
                    logits = model(x[start:stop])
                    loss = F.cross_entropy(
                        logits.reshape(-1, vocab_size), y[start:stop].reshape(-1)
                    ) / gradient_accumulation_steps
                scalers[name].scale(loss).backward()
            scalers[name].unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
            scalers[name].step(optimizer)
            scalers[name].update()
    if amp_enabled:
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    print(
        f"completed shuffled training epochs="
        f"{training_stream.epochs + training_stream.cursor / len(training_stream.order):.2f}"
    )
    checkpoint_metadata: Dict[str, Dict[str, float]] = {}
    for name, model in models.items():
        model.load_state_dict(best_states[name])
        checkpoint_metadata[name] = {
            "step": float(best_steps[name]),
            "token_presentations": float(
                best_steps[name] * cfg.batch_size * cfg.block_size
            ),
            "selection_nll": float(best_selection[name]),
        }
        print(
            f"restored {name} checkpoint | step={best_steps[name]} | "
            f"selection nll={best_selection[name]:.4f}"
        )
    del training_models, optimizers, scalers, training_stream, selection_source
    if amp_enabled:
        torch.cuda.empty_cache()
    return models, history, elapsed, checkpoint_metadata


@torch.inference_mode()
def sequential_neural_probabilities(
    model: nn.Module, data: np.ndarray, vocab_size: int
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    n_blocks = (len(data) - 1) // cfg.block_size
    source = DeviceSequence(data)
    starts = torch.arange(n_blocks, dtype=torch.long, device=device) * cfg.block_size
    outputs: List[np.ndarray] = []
    targets: List[np.ndarray] = []
    window_batch = cfg.evaluation_window_batch_size
    for start in range(0, n_blocks, window_batch):
        tokens, labels = source.gather(starts[start : start + window_batch])
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(tokens)
        outputs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        targets.append(labels.cpu().numpy())
    return (
        np.concatenate(outputs, axis=0).reshape(-1, vocab_size),
        np.concatenate(targets, axis=0).reshape(-1),
    )


def distribution_metrics(probabilities: np.ndarray, targets: np.ndarray) -> Dict[str, float]:
    true_probability = np.clip(probabilities[np.arange(len(targets)), targets], 1e-12, 1.0)
    nll = float(-np.mean(np.log(true_probability)))
    return {
        "nll": nll,
        "bits_per_character": nll / math.log(2.0),
        "perplexity": float(math.exp(nll)),
        "next_character_accuracy": float(np.mean(np.argmax(probabilities, axis=1) == targets)),
    }


def gate_features(
    symbolic: np.ndarray,
    neural: np.ndarray,
    supports: np.ndarray,
    orders: np.ndarray,
) -> np.ndarray:
    eps = 1e-12
    symbolic_entropy = -np.sum(symbolic * np.log(np.clip(symbolic, eps, 1.0)), axis=1)
    neural_entropy = -np.sum(neural * np.log(np.clip(neural, eps, 1.0)), axis=1)
    midpoint = 0.5 * (symbolic + neural)
    js = 0.5 * np.sum(
        symbolic * (np.log(np.clip(symbolic, eps, 1.0)) - np.log(np.clip(midpoint, eps, 1.0)))
        + neural * (np.log(np.clip(neural, eps, 1.0)) - np.log(np.clip(midpoint, eps, 1.0))),
        axis=1,
    )
    symbolic_top = np.partition(symbolic, -2, axis=1)[:, -2:]
    neural_top = np.partition(neural, -2, axis=1)[:, -2:]
    return np.column_stack(
        [
            np.log1p(supports),
            orders / max(1, cfg.symbolic_max_order),
            symbolic_entropy / math.log(symbolic.shape[1]),
            neural_entropy / math.log(neural.shape[1]),
            symbolic_top[:, 1] - symbolic_top[:, 0],
            neural_top[:, 1] - neural_top[:, 0],
            js,
        ]
    ).astype(np.float32)


@dataclass
class CalibratedGate:
    feature_mean: np.ndarray
    feature_std: np.ndarray
    weights: np.ndarray
    bias: float
    global_alpha: float
    feature_scale: float
    selected_mode: str
    train_nll: float
    selection_nll: float
    audit_nll: float
    audit_best_component_nll: float

    def alpha(self, features: np.ndarray) -> np.ndarray:
        if self.selected_mode == "symbolic-only":
            return np.ones(len(features), dtype=np.float32)
        if self.selected_mode == "neural-only":
            return np.zeros(len(features), dtype=np.float32)
        normalized = (features - self.feature_mean) / self.feature_std
        raw = 1.0 / (1.0 + np.exp(-(normalized @ self.weights + self.bias)))
        if self.selected_mode == "global-mixture":
            return np.full(len(features), self.global_alpha, dtype=np.float32)
        return (
            self.global_alpha + self.feature_scale * (raw - self.global_alpha)
        ).clip(0.0, 1.0).astype(np.float32)


def mixture_nll(alpha: np.ndarray, symbolic_true: np.ndarray, neural_true: np.ndarray) -> float:
    probability = alpha * symbolic_true + (1.0 - alpha) * neural_true
    return float(-np.mean(np.log(np.clip(probability, 1e-12, 1.0))))


def fit_calibrated_gate(
    symbolic: np.ndarray,
    neural: np.ndarray,
    supports: np.ndarray,
    orders: np.ndarray,
    targets: np.ndarray,
) -> CalibratedGate:
    features = gate_features(symbolic, neural, supports, orders)
    train_end = int(cfg.gate_train_fraction * len(targets))
    selection_end = int(
        (cfg.gate_train_fraction + cfg.gate_selection_fraction) * len(targets)
    )
    if not 0 < train_end < selection_end < len(targets):
        raise RuntimeError("Gate train/selection/audit split is invalid")
    feature_mean = features[:train_end].mean(axis=0)
    feature_std = features[:train_end].std(axis=0) + 1e-6
    normalized = (features[:train_end] - feature_mean) / feature_std
    symbolic_true = symbolic[np.arange(len(targets)), targets]
    neural_true = neural[np.arange(len(targets)), targets]

    x = torch.as_tensor(normalized, dtype=torch.float32, device=device)
    s = torch.as_tensor(symbolic_true[:train_end], dtype=torch.float32, device=device)
    n = torch.as_tensor(neural_true[:train_end], dtype=torch.float32, device=device)
    linear = nn.Linear(features.shape[1], 1).to(device)
    nn.init.zeros_(linear.weight)
    nn.init.zeros_(linear.bias)
    optimizer = torch.optim.Adam(linear.parameters(), lr=0.05, weight_decay=1e-3)
    for _ in range(cfg.gate_steps):
        optimizer.zero_grad(set_to_none=True)
        alpha = torch.sigmoid(linear(x)).squeeze(-1)
        loss = -torch.log(torch.clamp(alpha * s + (1.0 - alpha) * n, min=1e-12)).mean()
        loss.backward()
        optimizer.step()
    weights = linear.weight.detach().cpu().numpy().reshape(-1)
    bias = float(linear.bias.detach().cpu().item())

    grid = np.linspace(0.0, 1.0, 41)
    global_losses = [
        mixture_nll(
            np.full(train_end, alpha, dtype=np.float32),
            symbolic_true[:train_end],
            neural_true[:train_end],
        )
        for alpha in grid
    ]
    global_alpha = float(grid[int(np.argmin(global_losses))])
    selection_features = features[train_end:selection_end]
    selection_symbolic = symbolic_true[train_end:selection_end]
    selection_neural = neural_true[train_end:selection_end]
    normalized_selection = (selection_features - feature_mean) / feature_std
    raw_selection = 1.0 / (1.0 + np.exp(-(normalized_selection @ weights + bias)))
    candidates: List[Tuple[str, float, np.ndarray]] = [
        ("neural-only", 0.0, np.zeros(len(selection_features), dtype=np.float32)),
        ("symbolic-only", 0.0, np.ones(len(selection_features), dtype=np.float32)),
        (
            "global-mixture",
            0.0,
            np.full(len(selection_features), global_alpha, dtype=np.float32),
        ),
    ]
    for scale in (0.25, 0.5, 0.75, 1.0):
        alpha = np.clip(global_alpha + scale * (raw_selection - global_alpha), 0.0, 1.0)
        candidates.append(("feature-gate", scale, alpha.astype(np.float32)))
    losses = [
        mixture_nll(alpha, selection_symbolic, selection_neural)
        for _, _, alpha in candidates
    ]
    selected = int(np.argmin(losses))
    selected_mode, feature_scale, _ = candidates[selected]

    train_raw = 1.0 / (1.0 + np.exp(-(normalized @ weights + bias)))
    if selected_mode == "neural-only":
        train_alpha = np.zeros(train_end, dtype=np.float32)
    elif selected_mode == "symbolic-only":
        train_alpha = np.ones(train_end, dtype=np.float32)
    elif selected_mode == "global-mixture":
        train_alpha = np.full(train_end, global_alpha, dtype=np.float32)
    else:
        train_alpha = np.clip(
            global_alpha + feature_scale * (train_raw - global_alpha), 0.0, 1.0
        )

    audit_features = features[selection_end:]
    audit_symbolic = symbolic_true[selection_end:]
    audit_neural = neural_true[selection_end:]
    provisional = CalibratedGate(
        feature_mean=feature_mean,
        feature_std=feature_std,
        weights=weights,
        bias=bias,
        global_alpha=global_alpha,
        feature_scale=float(feature_scale),
        selected_mode=selected_mode,
        train_nll=0.0,
        selection_nll=0.0,
        audit_nll=0.0,
        audit_best_component_nll=0.0,
    )
    audit_alpha = provisional.alpha(audit_features)
    return CalibratedGate(
        feature_mean=feature_mean,
        feature_std=feature_std,
        weights=weights,
        bias=bias,
        global_alpha=global_alpha,
        feature_scale=float(feature_scale),
        selected_mode=selected_mode,
        train_nll=mixture_nll(
            train_alpha, symbolic_true[:train_end], neural_true[:train_end]
        ),
        selection_nll=float(losses[selected]),
        audit_nll=mixture_nll(audit_alpha, audit_symbolic, audit_neural),
        audit_best_component_nll=min(
            mixture_nll(np.ones(len(audit_features)), audit_symbolic, audit_neural),
            mixture_nll(np.zeros(len(audit_features)), audit_symbolic, audit_neural),
        ),
    )


def decode_context(code: int, order: int, itos: Dict[int, str], vocab_size: int) -> str:
    tokens = [0] * order
    value = int(code)
    for position in range(order - 1, -1, -1):
        tokens[position] = value % vocab_size
        value //= vocab_size
    return "".join(itos[token] for token in tokens)


def symbolic_state_rows(
    machine: MDLContextMachine, itos: Dict[int, str], limit: int = 80
) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for order, level in machine.levels.items():
        for index in range(len(level.codes)):
            probability = level.probabilities[index]
            top = np.argsort(probability)[-3:][::-1]
            rows.append(
                {
                    "order": order,
                    "context": decode_context(int(level.codes[index]), order, itos, machine.vocab_size),
                    "support": float(level.totals[index]),
                    "predictive_gain_nats": float(level.gains[index]),
                    "entropy_nats": float(-np.sum(probability * np.log(np.clip(probability, 1e-12, 1.0)))),
                    "top_predictions": " | ".join(
                        f"{itos[int(token)]!r}:{probability[int(token)]:.3f}" for token in top
                    ),
                }
            )
    selected: List[Dict[str, object]] = []
    per_order = max(1, limit // max(1, len(machine.levels)))
    for order in sorted(machine.levels):
        group = [row for row in rows if int(row["order"]) == order]
        group.sort(key=lambda row: -float(row["predictive_gain_nats"]))
        selected.extend(group[:per_order])
    selected.sort(key=lambda row: (int(row["order"]), -float(row["predictive_gain_nats"])))
    return selected[:limit]


@torch.inference_mode()
def next_neural_probability(model: nn.Module, context: Sequence[int]) -> np.ndarray:
    tokens = torch.as_tensor(
        np.asarray(context[-cfg.block_size :], dtype=np.int64)[None, :],
        dtype=torch.long,
        device=device,
    )
    model.eval()
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
        logits = model(tokens)[:, -1]
    return torch.softmax(logits.float(), dim=-1)[0].cpu().numpy()


def sample_model(
    model: nn.Module,
    prompt: str,
    stoi: Dict[str, int],
    itos: Dict[int, str],
    generator: np.random.Generator,
    machine: Optional[MDLContextMachine] = None,
    gate: Optional[CalibratedGate] = None,
) -> str:
    tokens = [stoi[character] for character in prompt]
    recurrent_state: Optional[torch.Tensor] = None
    recurrent_logits: Optional[torch.Tensor] = None
    if isinstance(model, RecurrentNeuralLM):
        model.eval()
        warmup = torch.as_tensor(tokens, dtype=torch.long, device=device)[None, :]
        with torch.inference_mode(), torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=amp_enabled
        ):
            logits, recurrent_state = model.step(warmup)
        recurrent_logits = logits[:, -1]
    for _ in range(cfg.generation_characters):
        if recurrent_logits is None:
            neural = next_neural_probability(model, tokens)
        else:
            neural = torch.softmax(recurrent_logits.float(), dim=-1)[0].cpu().numpy()
        if machine is not None and gate is not None:
            symbolic, support, order = machine.predict_one(tokens)
            features = gate_features(
                symbolic[None, :],
                neural[None, :],
                np.asarray([support], dtype=np.float32),
                np.asarray([order], dtype=np.int16),
            )
            alpha = float(gate.alpha(features)[0])
            probability = alpha * symbolic + (1.0 - alpha) * neural
        else:
            probability = neural
        logits = np.log(np.clip(probability, 1e-12, 1.0)) / cfg.temperature
        if cfg.top_k < len(logits):
            threshold = np.partition(logits, -cfg.top_k)[-cfg.top_k]
            logits[logits < threshold] = -np.inf
        probability = np.exp(logits - np.max(logits))
        probability /= probability.sum()
        next_token = int(generator.choice(len(probability), p=probability))
        tokens.append(next_token)
        if recurrent_logits is not None:
            token_tensor = torch.as_tensor([[next_token]], dtype=torch.long, device=device)
            with torch.inference_mode(), torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=amp_enabled
            ):
                logits, recurrent_state = model.step(token_tensor, recurrent_state)
            recurrent_logits = logits[:, -1]
    return "".join(itos[token] for token in tokens)


print("=" * 118)
print("CRYSTALLIZED nanoGPT V10.1 — calibration-selected Transformer versus recurrent + symbolic")
print("=" * 118)
print(
    f"device={device} | torch={torch.__version__} | fast_dev={FAST_DEV_RUN} | "
    f"official_scale={cfg.transformer_layers == 6 and cfg.transformer_width == 384}"
)
if amp_enabled:
    gpu_properties = torch.cuda.get_device_properties(0)
    print(
        f"GPU={torch.cuda.get_device_name(0)} | VRAM={gpu_properties.total_memory / 2**30:.1f} GiB | "
        f"capability={gpu_properties.major}.{gpu_properties.minor} | precision=FP16 AMP"
    )
text, fit_data, calibration_data, validation_data, stoi, itos = load_official_data()
vocab_size = len(stoi)
checkpoint_selection_data = calibration_data[: cfg.checkpoint_selection_characters]
gate_calibration_data = calibration_data[cfg.checkpoint_selection_characters :]
if len(checkpoint_selection_data) <= cfg.block_size or len(gate_calibration_data) <= cfg.block_size:
    raise RuntimeError("Calibration data is too short for disjoint checkpoint and gate splits")
print(
    f"official corpus={len(text):,} chars | vocab={vocab_size} | "
    f"fit={len(fit_data):,} | checkpoint-select={len(checkpoint_selection_data):,} | "
    f"gate-calibration={len(gate_calibration_data):,} | "
    f"validation={len(validation_data):,}"
)
print(
    f"block={cfg.block_size} | batch={cfg.batch_size} | steps={cfg.max_steps} | "
    f"tokens/learner={cfg.max_steps*cfg.batch_size*cfg.block_size:,}"
)
print(
    f"T4 execution: microbatch={cfg.micro_batch_size} x accumulation={gradient_accumulation_steps} | "
    f"fused AdamW={amp_enabled} | compiled nanoGPT train={amp_enabled and cfg.compile_transformer_training} "
    f"({transformer_compile_mode})"
)

total_started = time.perf_counter()
if amp_enabled:
    torch.cuda.reset_peak_memory_stats()
models, neural_history, neural_train_seconds, checkpoint_metadata = train_neural_models(
    fit_data, checkpoint_selection_data, vocab_size
)
tokens_per_model = cfg.max_steps * cfg.batch_size * cfg.block_size
aggregate_training_tokens_per_second = 2.0 * tokens_per_model / max(neural_train_seconds, 1e-9)
parameter_counts = {
    name: sum(parameter.numel() for parameter in model.parameters())
    for name, model in models.items()
}
parameter_ratio = parameter_counts["recurrent-neural"] / parameter_counts["nanoGPT"]
assert 0.90 <= parameter_ratio <= 1.10
print("parameter counts:", ", ".join(f"{name}={count:,}" for name, count in parameter_counts.items()))
print(f"recurrent/Transformer parameter ratio={parameter_ratio:.4f}")
print(
    f"neural wall time={neural_train_seconds:.1f}s | "
    f"aggregate throughput={aggregate_training_tokens_per_second:,.0f} tokens/s"
)
peak_gpu_gib = torch.cuda.max_memory_allocated() / 2**30 if amp_enabled else 0.0
if amp_enabled:
    print(f"peak allocated VRAM={peak_gpu_gib:.2f} GiB")

symbolic_started = time.perf_counter()
machine = MDLContextMachine(vocab_size).fit(fit_data, verbose=True)
symbolic_fit_seconds = time.perf_counter() - symbolic_started
print(f"symbolic machine states={machine.n_states:,} | fit seconds={symbolic_fit_seconds:.2f}")

symbolic_learning_curve: List[Dict[str, float]] = []
prefixes = sorted(set(min(len(fit_data), value) for value in cfg.prefix_observations) | {len(fit_data)})
for observations in prefixes:
    prefix_machine = machine if observations == len(fit_data) else MDLContextMachine(vocab_size).fit(
        fit_data[:observations]
    )
    probabilities, _, _, targets = prefix_machine.predict_sequence(
        checkpoint_selection_data, cfg.block_size
    )
    usable = ((len(checkpoint_selection_data) - 1) // cfg.block_size) * cfg.block_size
    probabilities, targets = probabilities[:usable], targets[:usable]
    metrics = distribution_metrics(probabilities, targets)
    symbolic_learning_curve.append(
        {
            "model": "mdl-context-machine",
            "character_observations": float(observations),
            "states": float(prefix_machine.n_states),
            **metrics,
        }
    )
    print(
        f"symbolic observations={observations:9,d} | states={prefix_machine.n_states:7,d} | "
        f"selection nll={metrics['nll']:.4f}"
    )

calibration_neural, calibration_targets = sequential_neural_probabilities(
    models["recurrent-neural"], gate_calibration_data, vocab_size
)
calibration_symbolic, calibration_support, calibration_order, symbolic_calibration_targets = (
    machine.predict_sequence(gate_calibration_data, cfg.block_size)
)
calibration_usable = len(calibration_targets)
calibration_symbolic = calibration_symbolic[:calibration_usable]
calibration_support = calibration_support[:calibration_usable]
calibration_order = calibration_order[:calibration_usable]
symbolic_calibration_targets = symbolic_calibration_targets[:calibration_usable]
assert np.array_equal(calibration_targets, symbolic_calibration_targets)
gate = fit_calibrated_gate(
    calibration_symbolic,
    calibration_neural,
    calibration_support,
    calibration_order,
    calibration_targets,
)
print(
    f"gate mode={gate.selected_mode} | global alpha={gate.global_alpha:.3f} | "
    f"feature scale={gate.feature_scale:.2f} | audit nll={gate.audit_nll:.4f}"
)

validation_probabilities: Dict[str, np.ndarray] = {}
validation_targets_reference: Optional[np.ndarray] = None
for name, model in models.items():
    probability, targets = sequential_neural_probabilities(model, validation_data, vocab_size)
    validation_probabilities[name] = probability
    if validation_targets_reference is None:
        validation_targets_reference = targets
    else:
        assert np.array_equal(validation_targets_reference, targets)
assert validation_targets_reference is not None
validation_targets = validation_targets_reference
symbolic_probability, validation_support, validation_order, symbolic_targets = machine.predict_sequence(
    validation_data, cfg.block_size
)
symbolic_probability = symbolic_probability[: len(validation_targets)]
validation_support = validation_support[: len(validation_targets)]
validation_order = validation_order[: len(validation_targets)]
symbolic_targets = symbolic_targets[: len(validation_targets)]
assert np.array_equal(validation_targets, symbolic_targets)
validation_probabilities["mdl-context-machine"] = symbolic_probability
validation_features = gate_features(
    symbolic_probability,
    validation_probabilities["recurrent-neural"],
    validation_support,
    validation_order,
)
validation_alpha = gate.alpha(validation_features)
validation_probabilities["crystallized-hybrid"] = (
    validation_alpha[:, None] * symbolic_probability
    + (1.0 - validation_alpha[:, None]) * validation_probabilities["recurrent-neural"]
)

final_metrics: List[Dict[str, object]] = []
for name, probability in validation_probabilities.items():
    uses_neural = name in ("nanoGPT", "recurrent-neural", "crystallized-hybrid")
    uses_symbolic = name in ("mdl-context-machine", "crystallized-hybrid")
    final_metrics.append(
        {
            "model": name,
            "parameters": (
                parameter_counts["recurrent-neural"]
                if name == "crystallized-hybrid"
                else parameter_counts.get(name, 0)
            ),
            "symbolic_states": machine.n_states if uses_symbolic else 0,
            "fit_character_observations": len(fit_data),
            "gradient_token_presentations": (
                cfg.max_steps * cfg.batch_size * cfg.block_size if uses_neural else 0
            ),
            "selected_checkpoint_step": (
                checkpoint_metadata["recurrent-neural"]["step"]
                if name == "crystallized-hybrid"
                else checkpoint_metadata.get(name, {}).get("step", 0)
            ),
            "selected_checkpoint_token_presentations": (
                checkpoint_metadata["recurrent-neural"]["token_presentations"]
                if name == "crystallized-hybrid"
                else checkpoint_metadata.get(name, {}).get("token_presentations", 0)
            ),
            "symbolic_character_observations": len(fit_data) if uses_symbolic else 0,
            **distribution_metrics(probability, validation_targets),
        }
    )

metrics_map = {str(row["model"]): row for row in final_metrics}
neural_final = float(metrics_map["recurrent-neural"]["nll"])
symbolic_final = float(metrics_map["mdl-context-machine"]["nll"])
hybrid_final = float(metrics_map["crystallized-hybrid"]["nll"])
transformer_final = float(metrics_map["nanoGPT"]["nll"])


def first_observations_at_nll(rows: List[Dict[str, float]], threshold: float) -> Optional[int]:
    hits = [
        int(row["character_observations"])
        for row in rows
        if float(row["nll"]) <= threshold
    ]
    return min(hits) if hits else None


reference_threshold = 2.0
symbolic_k2 = first_observations_at_nll(symbolic_learning_curve, reference_threshold)
neural_k2: Dict[str, Optional[int]] = {}
for name in ("nanoGPT", "recurrent-neural"):
    hits = [
        int(row["token_presentations"])
        for row in neural_history
        if row["model"] == name and float(row["checkpoint_selection_nll"]) <= reference_threshold
    ]
    neural_k2[name] = min(hits) if hits else None

predictions = {
    "P1_MDL_crystallization_builds_nontrivial_finite_machine": machine.n_states > vocab_size,
    "P2_hybrid_beats_matched_nanoGPT_validation_NLL": hybrid_final < transformer_final,
    "P3_hybrid_beats_each_component_validation_NLL": hybrid_final < min(neural_final, symbolic_final),
    "P4_gate_no_harm_on_heldout_calibration_audit": gate.audit_nll <= gate.audit_best_component_nll + 1e-6,
    "P5_symbolic_reaches_NLL2_with_fewer_observations_than_nanoGPT": bool(
        symbolic_k2 is not None
        and (neural_k2["nanoGPT"] is None or symbolic_k2 < neural_k2["nanoGPT"])
    ),
}

print("\nFINAL VALIDATION")
print("-" * 118)
print(" model                    | params       | states    | nll    | bpc    | ppl   | top-1")
for row in final_metrics:
    print(
        f" {row['model']:24s} | {int(row['parameters']):12,d} | {int(row['symbolic_states']):9,d} | "
        f"{row['nll']:.4f} | {row['bits_per_character']:.4f} | {row['perplexity']:.3f} | "
        f"{100*row['next_character_accuracy']:5.1f}%"
    )
print("\nSAMPLE EFFICIENCY")
print("-" * 118)
print(f"first character observations/presentations reaching selection NLL <= {reference_threshold:.1f}")
print(f" MDL context machine: {symbolic_k2}")
print(f" recurrent neural:    {neural_k2['recurrent-neural']}")
print(f" nanoGPT:             {neural_k2['nanoGPT']}")
print(
    f"gate symbolic use: mean alpha={validation_alpha.mean():.3f} | "
    f"alpha>0.5 on {100*np.mean(validation_alpha > 0.5):.1f}% of validation characters"
)
print("\nPREDECLARED PREDICTIONS")
print("-" * 118)
for name, passed in predictions.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

generation_rng = np.random.default_rng(cfg.seed + 999)
prompt = "ROMEO:\n"
samples = {
    "nanoGPT": sample_model(models["nanoGPT"], prompt, stoi, itos, generation_rng),
    "recurrent-neural": sample_model(
        models["recurrent-neural"], prompt, stoi, itos, generation_rng
    ),
    "crystallized-hybrid": sample_model(
        models["recurrent-neural"],
        prompt,
        stoi,
        itos,
        generation_rng,
        machine=machine,
        gate=gate,
    ),
}

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
for name, model in models.items():
    torch.save(model.state_dict(), out_dir / f"{name}.pt")
state_rows = symbolic_state_rows(machine, itos)
with (out_dir / "config.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            **asdict(cfg),
            "device": str(device),
            "torch_version": torch.__version__,
            "corpus_characters": len(text),
            "vocab_size": vocab_size,
            "fit_characters": len(fit_data),
            "calibration_characters": len(calibration_data),
            "checkpoint_selection_characters_actual": len(checkpoint_selection_data),
            "gate_calibration_characters": len(gate_calibration_data),
            "validation_characters": len(validation_data),
            "parameter_counts": parameter_counts,
            "parameter_ratio": parameter_ratio,
            "peak_allocated_gpu_gib": peak_gpu_gib,
            "aggregate_training_tokens_per_second": aggregate_training_tokens_per_second,
            "data_sha256": "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
        },
        file,
        indent=2,
    )
with (out_dir / "results.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "predictions": predictions,
            "checkpoint_selection": checkpoint_metadata,
            "final_metrics": final_metrics,
            "sample_efficiency_nll_2": {
                "mdl_context_machine": symbolic_k2,
                "recurrent_neural": neural_k2["recurrent-neural"],
                "nanoGPT": neural_k2["nanoGPT"],
            },
            "gate": {
                "selected_mode": gate.selected_mode,
                "global_alpha": gate.global_alpha,
                "feature_scale": gate.feature_scale,
                "train_nll": gate.train_nll,
                "selection_nll": gate.selection_nll,
                "audit_nll": gate.audit_nll,
                "audit_best_component_nll": gate.audit_best_component_nll,
                "mean_validation_alpha": float(validation_alpha.mean()),
                "validation_symbolic_majority_rate": float(np.mean(validation_alpha > 0.5)),
            },
            "timing": {
                "neural_training_seconds": neural_train_seconds,
                "symbolic_fit_seconds": symbolic_fit_seconds,
                "peak_allocated_gpu_gib": peak_gpu_gib,
                "aggregate_training_tokens_per_second": aggregate_training_tokens_per_second,
                "total_seconds": time.perf_counter() - total_started,
            },
        },
        file,
        indent=2,
    )
with (out_dir / "samples.txt").open("w", encoding="utf-8") as file:
    for name, sample in samples.items():
        file.write("=" * 80 + f"\n{name}\n" + "=" * 80 + "\n" + sample + "\n\n")

for filename, rows in (
    ("neural_learning_curve.csv", neural_history),
    ("symbolic_learning_curve.csv", symbolic_learning_curve),
    ("final_metrics.csv", final_metrics),
    ("symbolic_states.csv", state_rows),
):
    with (out_dir / filename).open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
colors = {
    "nanoGPT": "#E45756",
    "recurrent-neural": "#4C78A8",
    "mdl-context-machine": "#F2CF5B",
    "crystallized-hybrid": "#54A24B",
}

ax = axes[0, 0]
for name in ("nanoGPT", "recurrent-neural"):
    rows = [row for row in neural_history if row["model"] == name]
    ax.plot(
        [max(1, row["token_presentations"]) for row in rows],
        [row["checkpoint_selection_nll"] for row in rows],
        marker="o",
        color=colors[name],
        label=name,
    )
ax.set_xscale("log")
ax.set_title("Neural learning versus charged token presentations")
ax.set_xlabel("character-label presentations")
ax.set_ylabel("checkpoint-selection NLL (nats/character)")
ax.grid(alpha=0.25)
ax.legend()

ax = axes[0, 1]
ax.plot(
    [row["character_observations"] for row in symbolic_learning_curve],
    [row["nll"] for row in symbolic_learning_curve],
    marker="o",
    color=colors["mdl-context-machine"],
    label="MDL context machine (one pass)",
)
for name in ("nanoGPT", "recurrent-neural"):
    rows = [row for row in neural_history if row["model"] == name]
    ax.plot(
        [max(1, row["token_presentations"]) for row in rows],
        [row["checkpoint_selection_nll"] for row in rows],
        marker=".", alpha=0.7, color=colors[name], label=name,
    )
ax.set_xscale("log")
ax.set_title("Sample efficiency: one observation is one next-character label")
ax.set_xlabel("charged character observations/presentations")
ax.set_ylabel("checkpoint-selection NLL")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 0]
names = [str(row["model"]) for row in final_metrics]
ax.bar(
    np.arange(len(names)),
    [float(row["nll"]) for row in final_metrics],
    color=[colors[name] for name in names],
)
ax.set_xticks(np.arange(len(names)), names, rotation=20, ha="right")
ax.set_title("Final official validation loss (lower is better)")
ax.set_ylabel("nats/character")
ax.grid(axis="y", alpha=0.25)

ax = axes[1, 1]
orders_to_plot = sorted(set(int(value) for value in validation_order))
means = [
    float(validation_alpha[validation_order == order].mean())
    if np.any(validation_order == order)
    else float("nan")
    for order in orders_to_plot
]
counts = [int(np.sum(validation_order == order)) for order in orders_to_plot]
ax.bar(orders_to_plot, means, color="#B279A2")
for order, mean, count in zip(orders_to_plot, means, counts):
    ax.text(order, min(0.98, mean + 0.025), f"n={count:,}", ha="center", fontsize=7, rotation=90)
ax.set_ylim(0, 1.05)
ax.set_title("Calibrated symbolic responsibility by machine order")
ax.set_xlabel("longest retained symbolic suffix")
ax.set_ylabel("mean gate weight on symbolic machine")
ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    "Crystallized nanoGPT V10.1 — calibration-selected, official validation untouched",
    fontsize=15,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / "summary.png", dpi=180, bbox_inches="tight")
plt.show()

target_bundle = out_dir / "run_bundle.zip"
target_bundle.unlink(missing_ok=True)
archive_path = shutil.make_archive(str(out_dir.parent / f"{out_dir.name}_bundle"), "zip", root_dir=out_dir)
shutil.copy2(archive_path, target_bundle)
Path(archive_path).unlink(missing_ok=True)

print("\n" + "=" * 118)
print("FINAL V10.1 SUMMARY")
print("=" * 118)
print(
    f"validation NLL: nanoGPT={transformer_final:.4f} | GRU={neural_final:.4f} | "
    f"machine={symbolic_final:.4f} | hybrid={hybrid_final:.4f}"
)
print(
    f"hybrid improvement over nanoGPT={100*(transformer_final-hybrid_final)/transformer_final:+.2f}% | "
    f"machine states={machine.n_states:,} | gate={gate.selected_mode}"
)
print(f"artifacts: {out_dir}")
print(f"bundle: {target_bundle}")


## Experiment 2: Quotient-MDL active learner V11

Original cell `1`.


In [ ]:
"""
QUOTIENT-MDL ACTIVE LEARNER V11 — structural transfer across unseen representations

Core test
---------
Prior worlds may use arbitrary state names while sharing the same abstract finite
dynamics. The learner canonicalizes complete prior transition tables up to state
relabeling, uses MDL to store one template per isomorphism class, and then actively
queries an unseen relabeling to identify both its abstract law and its coordinate map.

This isolates a missing kind of generalization: the same rule must be recognized
when its surface representation is new. A persistent out-of-library regime is added
only when repeated residual structure makes a new template cheaper than exceptions;
one-off perturbations must not reorganize the library.
"""

from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("QUOTIENT_MDL_V11_SEED", "20260802"))
    n_states: int = 8
    n_actions: int = 3
    n_families: int = 4
    prior_tasks_per_family: int = 6
    test_tasks_per_family: int = 8
    random_query_repeats: int = 3
    novel_test_tasks: int = 24
    transient_tasks: int = 12
    novel_transition_changes: int = 4
    audit_period: int = 3
    report_budgets: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24)
    output_dir: str = os.environ.get(
        "QUOTIENT_MDL_V11_OUTPUT_DIR",
        "outputs/02-quotient-mdl-active-learner-v11/quotient_mdl_active_v11_results"
        if Path("/content").exists()
        else "outputs/02-quotient-mdl-active-learner-v11/quotient_mdl_active_v11_results",
    )


cfg = Config()
rng = np.random.default_rng(cfg.seed)
random.seed(cfg.seed)
N, A = cfg.n_states, cfg.n_actions
CELLS = N * A
PERMUTATIONS = np.asarray(list(itertools.permutations(range(N))), dtype=np.int16)
INVERSES = np.argsort(PERMUTATIONS, axis=1).astype(np.int16)
N_PERMUTATIONS = len(PERMUTATIONS)


def strongly_connected(table: np.ndarray) -> bool:
    for source in range(N):
        seen = {source}
        frontier = [source]
        while frontier:
            state = frontier.pop()
            for successor in table[state]:
                successor = int(successor)
                if successor not in seen:
                    seen.add(successor)
                    frontier.append(successor)
        if len(seen) != N:
            return False
    return True


def all_relabelings(table: np.ndarray) -> np.ndarray:
    """Every table under every old-label -> new-label permutation."""
    old_successors = table[INVERSES]
    rows = np.arange(N_PERMUTATIONS)[:, None, None]
    return PERMUTATIONS[rows, old_successors]


def canonicalize(table: np.ndarray) -> Tuple[np.ndarray, np.ndarray, int]:
    relabeled = all_relabelings(np.asarray(table, dtype=np.int16))
    flat = relabeled.reshape(N_PERMUTATIONS, CELLS)
    first = int(np.lexsort(flat[:, ::-1].T)[0])
    canonical = relabeled[first].copy()
    automorphisms = int(np.sum(np.all(flat == canonical.reshape(1, -1), axis=1)))
    return canonical, PERMUTATIONS[first].copy(), automorphisms


def relabel_table(table: np.ndarray, old_to_new: np.ndarray) -> np.ndarray:
    old_to_new = np.asarray(old_to_new, dtype=np.int16)
    new_to_old = np.argsort(old_to_new)
    return old_to_new[table[new_to_old]]


def generate_asymmetric_machine(excluded: set[bytes]) -> np.ndarray:
    attempts = 0
    while True:
        attempts += 1
        candidate = rng.integers(0, N, size=(N, A), dtype=np.int16)
        if not strongly_connected(candidate):
            continue
        canonical, _, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
        if attempts > 20_000:
            raise RuntimeError("Could not generate an asymmetric strongly connected machine")


def mutate_machine(base: np.ndarray, changes: int, excluded: set[bytes]) -> np.ndarray:
    for _ in range(20_000):
        candidate = base.copy()
        cells = rng.choice(CELLS, size=changes, replace=False)
        for cell in cells:
            state, action = divmod(int(cell), A)
            old = int(candidate[state, action])
            draw = int(rng.integers(0, N - 1))
            candidate[state, action] = draw + (draw >= old)
        if not strongly_connected(candidate):
            continue
        canonical, _, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not construct a novel asymmetric machine")


def mine_quotient_library(prior_tables: Sequence[np.ndarray]) -> Tuple[List[np.ndarray], List[int]]:
    representatives: Dict[bytes, np.ndarray] = {}
    assignments: List[int] = []
    for table in prior_tables:
        canonical, _, _ = canonicalize(table)
        key = canonical.astype(np.int8).tobytes()
        if key not in representatives:
            representatives[key] = canonical
        ordered_keys = list(representatives)
        assignments.append(ordered_keys.index(key))
    library = sorted(representatives.values(), key=lambda x: x.astype(np.int8).tobytes())
    key_to_index = {table.astype(np.int8).tobytes(): i for i, table in enumerate(library)}
    assignments = [key_to_index[canonicalize(table)[0].astype(np.int8).tobytes()] for table in prior_tables]
    return library, assignments


def library_description_bits(n_tasks: int, n_templates: int) -> Dict[str, float]:
    table_bits = CELLS * math.log2(N)
    raw = n_tasks * table_bits
    quotient = (
        n_templates * table_bits
        + n_tasks * (math.log2(max(1, n_templates)) + math.log2(math.factorial(N)))
    )
    return {"raw_bits": raw, "quotient_bits": quotient, "compression": raw / quotient}


def hypothesis_predictions(library: Sequence[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    predictions: List[np.ndarray] = []
    family_ids: List[np.ndarray] = []
    row_index = np.arange(N_PERMUTATIONS)[:, None, None]
    action_index = np.arange(A)[None, None, :]
    for family, table in enumerate(library):
        canonical_next = table[PERMUTATIONS[:, :, None], action_index]
        observed_next = INVERSES[row_index, canonical_next]
        predictions.append(observed_next.reshape(N_PERMUTATIONS, CELLS).astype(np.int8))
        family_ids.append(np.full(N_PERMUTATIONS, family, dtype=np.int16))
    return np.concatenate(predictions), np.concatenate(family_ids)


def entropy_from_counts(counts: np.ndarray) -> float:
    positive = counts[counts > 0].astype(np.float64)
    probability = positive / positive.sum()
    return float(-np.sum(probability * np.log2(probability)))


def choose_query(
    candidates: np.ndarray,
    predictions: np.ndarray,
    unqueried: np.ndarray,
    strategy: str,
    generator: np.random.Generator,
    query_number: int,
) -> int:
    available = np.flatnonzero(unqueried)
    if strategy == "random" or (
        strategy == "dual" and query_number > 0 and query_number % cfg.audit_period == 0
    ):
        return int(generator.choice(available))
    entropies = np.asarray(
        [
            entropy_from_counts(np.bincount(predictions[candidates, cell], minlength=N))
            for cell in available
        ]
    )
    best = available[np.isclose(entropies, entropies.max(), atol=1e-12)]
    return int(generator.choice(best))


def posterior_metrics(
    candidates: np.ndarray, predictions: np.ndarray, truth: np.ndarray
) -> Dict[str, float]:
    if len(candidates) == 0:
        return {
            "posterior_nll": float("nan"),
            "transition_accuracy": float("nan"),
            "resolved_fraction": float("nan"),
            "exact_table": 0.0,
            "candidates": 0.0,
        }
    correct_probabilities: List[float] = []
    top_correct = 0
    resolved = 0
    for cell in range(CELLS):
        counts = np.bincount(predictions[candidates, cell], minlength=N)
        correct_probabilities.append(float(counts[int(truth[cell])]) / len(candidates))
        top_correct += int(np.argmax(counts) == truth[cell])
        resolved += int(np.count_nonzero(counts) == 1)
    probability = np.clip(np.asarray(correct_probabilities), 1e-15, 1.0)
    resolved_fraction = resolved / CELLS
    return {
        "posterior_nll": float(-np.mean(np.log(probability))),
        "transition_accuracy": top_correct / CELLS,
        "resolved_fraction": resolved_fraction,
        "exact_table": float(resolved == CELLS),
        "candidates": float(len(candidates)),
    }


def inference_trace(
    truth_table: np.ndarray,
    predictions: np.ndarray,
    strategy: str,
    seed: int,
    stop_on_rejection: bool = True,
) -> Tuple[List[Dict[str, float]], Optional[int]]:
    truth = truth_table.reshape(-1).astype(np.int16)
    candidates = np.arange(len(predictions), dtype=np.int32)
    unqueried = np.ones(CELLS, dtype=bool)
    generator = np.random.default_rng(seed)
    rows: List[Dict[str, float]] = []
    rejected_at: Optional[int] = None
    for query_count in range(CELLS + 1):
        rows.append({"queries": float(query_count), **posterior_metrics(candidates, predictions, truth)})
        if query_count == CELLS or len(candidates) == 0:
            if len(candidates) == 0 and rejected_at is None:
                rejected_at = query_count
            if stop_on_rejection:
                break
            continue
        cell = choose_query(candidates, predictions, unqueried, strategy, generator, query_count)
        unqueried[cell] = False
        observed = truth[cell]
        candidates = candidates[predictions[candidates, cell] == observed]
        if len(candidates) == 0:
            rejected_at = query_count + 1
    return rows, rejected_at


def local_trace() -> List[Dict[str, float]]:
    rows: List[Dict[str, float]] = []
    for queries in range(CELLS + 1):
        unresolved = CELLS - queries
        rows.append(
            {
                "queries": float(queries),
                "posterior_nll": unresolved / CELLS * math.log(N),
                "transition_accuracy": queries / CELLS + unresolved / CELLS / N,
                "resolved_fraction": queries / CELLS,
                "exact_table": float(queries == CELLS),
                "candidates": float(N**unresolved) if unresolved <= 100 else float("inf"),
            }
        )
    return rows


def summarize_rows(rows: Sequence[Dict[str, object]], group: Sequence[str]) -> List[Dict[str, object]]:
    grouped: Dict[Tuple[object, ...], List[Dict[str, object]]] = {}
    for row in rows:
        key = tuple(row[column] for column in group)
        grouped.setdefault(key, []).append(row)
    summary: List[Dict[str, object]] = []
    for key, values in sorted(grouped.items()):
        item = {column: value for column, value in zip(group, key)}
        for metric in ("posterior_nll", "transition_accuracy", "resolved_fraction", "exact_table"):
            data = np.asarray([float(value[metric]) for value in values], dtype=np.float64)
            item[f"mean_{metric}"] = float(np.nanmean(data))
            item[f"sem_{metric}"] = float(np.nanstd(data) / math.sqrt(max(1, np.sum(~np.isnan(data)))))
        summary.append(item)
    return summary


def first_exact(rows: Sequence[Dict[str, float]]) -> int:
    hits = [int(row["queries"]) for row in rows if row["exact_table"] == 1.0]
    return min(hits) if hits else CELLS + 1


def write_csv(path: Path, rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


print("=" * 112)
print("QUOTIENT-MDL ACTIVE LEARNER V11 — same rule, unseen representation")
print("=" * 112)
print(
    f"states={N} | actions={A} | transitions/world={CELLS} | "
    f"candidate relabelings={N_PERMUTATIONS:,}"
)
started = time.perf_counter()

excluded: set[bytes] = set()
base_library = [generate_asymmetric_machine(excluded) for _ in range(cfg.n_families)]

prior_tables: List[np.ndarray] = []
prior_families: List[int] = []
for family, table in enumerate(base_library):
    for _ in range(cfg.prior_tasks_per_family):
        prior_tables.append(relabel_table(table, rng.permutation(N)))
        prior_families.append(family)

mined_library, assignments = mine_quotient_library(prior_tables)
description = library_description_bits(len(prior_tables), len(mined_library))
expected_keys = {table.astype(np.int8).tobytes() for table in base_library}
mined_keys = {table.astype(np.int8).tobytes() for table in mined_library}
library_recovery = len(expected_keys & mined_keys) / len(expected_keys)
print(
    f"prior worlds={len(prior_tables)} | mined templates={len(mined_library)} | "
    f"template recovery={100*library_recovery:.1f}%"
)
print(
    f"description: raw={description['raw_bits']:.1f} bits | quotient={description['quotient_bits']:.1f} bits | "
    f"compression={description['compression']:.2f}x"
)

predictions, hypothesis_families = hypothesis_predictions(mined_library)
print(f"test hypothesis space={len(predictions):,} template×coordinate hypotheses")

positive_rows: List[Dict[str, object]] = []
exact_queries: Dict[str, List[int]] = {"quotient-active": [], "quotient-random": [], "local-tabular": []}
test_id = 0
for family, table in enumerate(base_library):
    for task_in_family in range(cfg.test_tasks_per_family):
        observed = relabel_table(table, rng.permutation(N))
        active_rows, rejected = inference_trace(
            observed, predictions, "active", cfg.seed + 10_000 + test_id
        )
        assert rejected is None
        exact_queries["quotient-active"].append(first_exact(active_rows))
        for row in active_rows:
            if int(row["queries"]) in cfg.report_budgets:
                positive_rows.append(
                    {"method": "quotient-active", "task": test_id, "repeat": 0, "family": family, **row}
                )
        for repeat in range(cfg.random_query_repeats):
            random_rows, rejected = inference_trace(
                observed,
                predictions,
                "random",
                cfg.seed + 100_000 + 101 * test_id + repeat,
            )
            assert rejected is None
            exact_queries["quotient-random"].append(first_exact(random_rows))
            for row in random_rows:
                if int(row["queries"]) in cfg.report_budgets:
                    positive_rows.append(
                        {
                            "method": "quotient-random",
                            "task": test_id,
                            "repeat": repeat,
                            "family": family,
                            **row,
                        }
                    )
        local_rows = local_trace()
        exact_queries["local-tabular"].append(first_exact(local_rows))
        for row in local_rows:
            if int(row["queries"]) in cfg.report_budgets:
                positive_rows.append(
                    {"method": "local-tabular", "task": test_id, "repeat": 0, "family": family, **row}
                )
        test_id += 1

positive_summary = summarize_rows(positive_rows, ("method", "queries"))
summary_lookup = {(row["method"], int(row["queries"])): row for row in positive_summary}
for method in exact_queries:
    values = np.asarray(exact_queries[method])
    print(
        f"{method:18s} | median queries to exact={np.median(values):4.1f} | "
        f"exact@8={100*summary_lookup[(method, 8)]['mean_exact_table']:5.1f}%"
    )

# Novel-family validity and persistent-regime promotion.
novel_machine = mutate_machine(base_library[0], cfg.novel_transition_changes, excluded)
novel_tables = [relabel_table(novel_machine, rng.permutation(N)) for _ in range(cfg.novel_test_tasks)]
negative_rows: List[Dict[str, object]] = []
detection: Dict[str, List[int]] = {"active": [], "dual": [], "random": []}
for task_id, observed in enumerate(novel_tables):
    for strategy in detection:
        _, rejected = inference_trace(
            observed,
            predictions,
            strategy,
            cfg.seed + 500_000 + 997 * task_id + {"active": 1, "dual": 2, "random": 3}[strategy],
        )
        latency = CELLS + 1 if rejected is None else rejected
        detection[strategy].append(latency)
        negative_rows.append({"strategy": strategy, "task": task_id, "detection_queries": latency})
for strategy, values in detection.items():
    values_array = np.asarray(values)
    print(
        f"novel detection {strategy:6s} | median={np.median(values_array):4.1f} queries | "
        f"within 12={100*np.mean(values_array <= 12):5.1f}%"
    )

novel_truth = novel_tables[0].reshape(-1)
nearest_mismatches = int(np.min(np.sum(predictions != novel_truth[None, :], axis=1)))
permutation_bits = math.log2(math.factorial(N))
existing_task_bits = (
    math.log2(len(mined_library))
    + permutation_bits
    + math.log2(math.comb(CELLS, nearest_mismatches))
    + nearest_mismatches * math.log2(N - 1)
)
new_definition_bits = CELLS * math.log2(N)
new_task_bits = math.log2(len(mined_library) + 1) + permutation_bits
mdl_rows: List[Dict[str, object]] = []
promotion_at: Optional[int] = None
for repeats in range(1, cfg.novel_test_tasks + 1):
    exceptions_bits = repeats * existing_task_bits
    promote_bits = new_definition_bits + repeats * new_task_bits
    if promotion_at is None and promote_bits < exceptions_bits:
        promotion_at = repeats
    mdl_rows.append(
        {
            "repeated_worlds": repeats,
            "encode_as_exceptions_bits": exceptions_bits,
            "add_template_bits": promote_bits,
        }
    )

transient_canonical_counts: Dict[bytes, int] = {}
for _ in range(cfg.transient_tasks):
    transient = mutate_machine(base_library[int(rng.integers(0, len(base_library)))], cfg.novel_transition_changes, excluded)
    key = transient.astype(np.int8).tobytes()
    transient_canonical_counts[key] = transient_canonical_counts.get(key, 0) + 1
false_promotions = sum(count >= (promotion_at or CELLS + 1) for count in transient_canonical_counts.values())
print(
    f"persistent regime nearest mismatches={nearest_mismatches} | MDL promotion={promotion_at} repeated worlds | "
    f"transient false promotions={false_promotions}"
)

expanded_library = list(mined_library) + [novel_machine]
expanded_predictions, _ = hypothesis_predictions(expanded_library)
future_exact_queries: List[int] = []
future_rows: List[Dict[str, object]] = []
for task_id, observed in enumerate(novel_tables):
    trace, rejected = inference_trace(
        observed, expanded_predictions, "active", cfg.seed + 800_000 + task_id
    )
    assert rejected is None
    future_exact_queries.append(first_exact(trace))
    for row in trace:
        if int(row["queries"]) in cfg.report_budgets:
            future_rows.append({"task": task_id, **row})
future_median = float(np.median(future_exact_queries))
print(f"after promotion: median active queries to exact={future_median:.1f} (local={CELLS})")

predictions_declared = {
    "P1_quotient_MDL_compresses_prior_worlds_over_1_5x": description["compression"] > 1.5,
    "P2_active_recovers_unseen_relabelings_with_median_at_most_10_queries": float(
        np.median(exact_queries["quotient-active"])
    )
    <= 10,
    "P3_active_exact_at_8_exceeds_random_by_at_least_10_points": float(
        summary_lookup[("quotient-active", 8)]["mean_exact_table"]
        - summary_lookup[("quotient-random", 8)]["mean_exact_table"]
    )
    >= 0.10,
    "P4_dual_validity_detects_at_least_90pct_novel_worlds_by_12_queries": float(
        np.mean(np.asarray(detection["dual"]) <= 12)
    )
    >= 0.90,
    "P5_MDL_promotes_repeated_regime_but_not_unique_transients": bool(
        promotion_at is not None and promotion_at <= 8 and false_promotions == 0
    ),
    "P6_promoted_template_cuts_future_exact_identification_below_half_table": future_median
    < CELLS / 2,
}

print("\nPREDECLARED PREDICTIONS")
print("-" * 112)
for name, passed in predictions_declared.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(out_dir / "positive_trials.csv", positive_rows)
write_csv(out_dir / "positive_summary.csv", positive_summary)
write_csv(out_dir / "novel_detection.csv", negative_rows)
write_csv(out_dir / "mdl_promotion.csv", mdl_rows)
write_csv(out_dir / "future_family_trials.csv", future_rows)

result = {
    "predictions": predictions_declared,
    "library": {
        "prior_worlds": len(prior_tables),
        "templates": len(mined_library),
        "template_recovery": library_recovery,
        **description,
    },
    "positive": {
        method: {
            "median_queries_to_exact": float(np.median(values)),
            "exact_at_8": float(summary_lookup[(method, 8)]["mean_exact_table"]),
        }
        for method, values in exact_queries.items()
    },
    "validity": {
        strategy: {
            "median_detection_queries": float(np.median(values)),
            "detected_by_12": float(np.mean(np.asarray(values) <= 12)),
        }
        for strategy, values in detection.items()
    },
    "reorganization": {
        "nearest_template_mismatches": nearest_mismatches,
        "promotion_at_repeated_worlds": promotion_at,
        "transient_false_promotions": false_promotions,
        "future_median_queries_to_exact": future_median,
    },
    "elapsed_seconds": time.perf_counter() - started,
}
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
(out_dir / "results.json").write_text(json.dumps(result, indent=2))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
colors = {"quotient-active": "#54A24B", "quotient-random": "#F2CF5B", "local-tabular": "#4C78A8"}

ax = axes[0, 0]
for method in ("local-tabular", "quotient-random", "quotient-active"):
    rows = [row for row in positive_summary if row["method"] == method]
    ax.plot(
        [row["queries"] for row in rows],
        [100 * row["mean_exact_table"] for row in rows],
        marker="o",
        color=colors[method],
        label=method,
    )
ax.set_title("Exact unseen-world recovery")
ax.set_xlabel("queried transitions")
ax.set_ylabel("exact full transition table (%)")
ax.set_ylim(-2, 102)
ax.grid(alpha=0.25)
ax.legend()

ax = axes[0, 1]
for method in ("local-tabular", "quotient-random", "quotient-active"):
    rows = [row for row in positive_summary if row["method"] == method]
    ax.plot(
        [row["queries"] for row in rows],
        [row["mean_posterior_nll"] for row in rows],
        marker="o",
        color=colors[method],
        label=method,
    )
ax.set_title("Posterior prediction on all transitions")
ax.set_xlabel("queried transitions")
ax.set_ylabel("NLL (nats / transition)")
ax.grid(alpha=0.25)
ax.legend()

ax = axes[1, 0]
for strategy, color in (("active", "#E45756"), ("dual", "#B279A2"), ("random", "#9D755D")):
    values = np.asarray(detection[strategy])
    budgets = np.arange(1, CELLS + 1)
    ax.step(budgets, [100 * np.mean(values <= budget) for budget in budgets], where="post", color=color, label=strategy)
ax.set_title("Detecting an out-of-library law")
ax.set_xlabel("queried transitions")
ax.set_ylabel("rejected as novel (%)")
ax.set_ylim(-2, 102)
ax.grid(alpha=0.25)
ax.legend()

ax = axes[1, 1]
ax.plot(
    [row["repeated_worlds"] for row in mdl_rows],
    [row["encode_as_exceptions_bits"] for row in mdl_rows],
    marker="o",
    label="keep as exceptions",
    color="#E45756",
)
ax.plot(
    [row["repeated_worlds"] for row in mdl_rows],
    [row["add_template_bits"] for row in mdl_rows],
    marker="o",
    label="promote new template",
    color="#54A24B",
)
if promotion_at is not None:
    ax.axvline(promotion_at, color="black", linestyle="--", label=f"promotion at {promotion_at}")
ax.set_title("Persistent residuals trigger reorganization")
ax.set_xlabel("repeated structurally identical novel worlds")
ax.set_ylabel("cumulative description length (bits)")
ax.grid(alpha=0.25)
ax.legend()

fig.suptitle("Quotient-MDL active learning: transfer the law, relearn only the coordinates", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / "summary.png", dpi=180, bbox_inches="tight")
plt.close(fig)

bundle = shutil.make_archive(str(out_dir.parent / f"{out_dir.name}_bundle"), "zip", root_dir=out_dir)
print("\n" + "=" * 112)
print("FINAL V11 SUMMARY")
print("=" * 112)
print(
    f"library compression={description['compression']:.2f}x | active median exact={np.median(exact_queries['quotient-active']):.1f} "
    f"queries | random={np.median(exact_queries['quotient-random']):.1f} | local={CELLS}"
)
print(
    f"novel dual detection median={np.median(detection['dual']):.1f} | promotion after={promotion_at} repeats | "
    f"future active exact={future_median:.1f} queries"
)
print(f"elapsed={result['elapsed_seconds']:.1f}s | artifacts={out_dir} | bundle={bundle}")


## Experiment 3: MDL text8 scaling V12 — 100,000-state cap

Original cell `3`, split part `1`.

**Cleanup note.** Two complete programs were concatenated in the source cell; they are separated here so each has valid future imports and independent artifacts.


In [ ]:
"""
MDL TEXT8 V12 — T4-ACCELERATED, SINGLE-CELL EXPERIMENT

Purpose
-------
Move the finite-state MDL mechanism beyond Tiny Shakespeare and test it by
itself on text8: 100,000,000 Wikipedia characters with the conventional
90M/5M/5M chronological split.

This is not a neural comparison.  It asks whether a one-pass variable-order
probabilistic state machine still (1) scales, (2) compresses its candidate
contexts, (3) benefits from deeper symbolic states, and (4) generalizes to a
far-away held-out test segment.  GPU sorting/counting is used when CUDA is
available; the learned object remains an explicit finite-state machine.

Environment overrides
---------------------
MDL_TEXT8_FAST_DEV_RUN=1     small smoke test
MDL_TEXT8_MAX_FIT=10000000   largest training prefix
MDL_TEXT8_STATE_CAP=100000   maximum retained states per order
MDL_TEXT8_MAX_ORDER=12       maximum symbolic suffix length
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch


FAST_DEV_RUN = os.environ.get("MDL_TEXT8_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 20260802
    data_url: str = "https://mattmahoney.net/dc/text8.zip"
    archive_path: str = "outputs/03-mdl-text8-scaling-v12-100-000-state-cap/text8.zip"
    data_path: str = "outputs/03-mdl-text8-scaling-v12-100-000-state-cap/text8"
    output_dir: str = "outputs/03-mdl-text8-scaling-v12-100-000-state-cap/mdl_text8_v12_results"
    expected_characters: int = 100_000_000
    train_end: int = 90_000_000
    validation_start: int = 90_000_000
    validation_characters: int = 1_000_000
    test_start: int = 95_000_000
    test_characters: int = 1_000_000
    fit_prefixes: Tuple[int, ...] = (1_000_000, 3_000_000, 10_000_000)
    max_order: int = 12
    min_count: int = 3
    max_states_per_order: int = 100_000
    backoff_strength: float = 32.0
    definition_nats: float = 2.0
    evaluation_block: int = 65_536
    generation_characters: int = 700
    generation_temperature: float = 0.90
    gpu_unique_threshold: int = 250_000


cfg = Config()
cfg = replace(
    cfg,
    fit_prefixes=tuple(
        sorted(
            {
                min(int(os.environ.get("MDL_TEXT8_MAX_FIT", cfg.fit_prefixes[-1])), n)
                for n in cfg.fit_prefixes
            }
        )
    ),
    max_states_per_order=int(
        os.environ.get("MDL_TEXT8_STATE_CAP", cfg.max_states_per_order)
    ),
    max_order=int(os.environ.get("MDL_TEXT8_MAX_ORDER", cfg.max_order)),
)
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        output_dir=str(Path.cwd() / "mdl_text8_v12_smoke"),
        expected_characters=0,
        train_end=700_000,
        validation_start=700_000,
        validation_characters=100_000,
        test_start=800_000,
        test_characters=100_000,
        fit_prefixes=(100_000, 300_000, 600_000),
        max_order=8,
        max_states_per_order=10_000,
        evaluation_block=16_384,
        generation_characters=250,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)


def download_text8() -> bytes:
    data_path = Path(cfg.data_path)
    archive_path = Path(cfg.archive_path)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    if not data_path.exists():
        if not archive_path.exists():
            print(f"downloading text8 -> {archive_path}")
            urllib.request.urlretrieve(cfg.data_url, archive_path)
        with zipfile.ZipFile(archive_path, "r") as archive:
            member = archive.getinfo("text8")
            if member.file_size != 100_000_000:
                raise RuntimeError(f"Unexpected text8 member size: {member.file_size:,}")
            archive.extract("text8", data_path.parent)
    raw = data_path.read_bytes()
    if not FAST_DEV_RUN and len(raw) != cfg.expected_characters:
        raise RuntimeError(f"Expected 100,000,000 characters, found {len(raw):,}")
    return raw


def encode_text8(raw: bytes) -> Tuple[np.ndarray, List[str]]:
    byte_values = np.frombuffer(raw, dtype=np.uint8)
    alphabet_bytes = np.unique(byte_values)
    expected = np.asarray([32] + list(range(97, 123)), dtype=np.uint8)
    if not np.array_equal(alphabet_bytes, expected):
        raise RuntimeError(f"Unexpected text8 alphabet: {alphabet_bytes.tolist()}")
    lookup = np.full(256, -1, dtype=np.int16)
    lookup[expected] = np.arange(len(expected), dtype=np.int16)
    ids = lookup[byte_values].astype(np.int64)
    alphabet = [chr(int(value)) for value in expected]
    return ids, alphabet


@dataclass
class SymbolicLevel:
    codes: np.ndarray
    probabilities: np.ndarray
    totals: np.ndarray
    gains: np.ndarray
    raw_unique: int
    eligible: int


class MDLContextMachine:
    """Explicit variable-order probabilistic automaton with MDL state admission."""

    def __init__(self, vocab_size: int, max_order: Optional[int] = None) -> None:
        self.vocab_size = int(vocab_size)
        self.max_order = int(max_order or cfg.max_order)
        if self.vocab_size ** (self.max_order + 1) >= 2**63:
            raise ValueError("Context encoding would overflow signed int64")
        self.levels: Dict[int, SymbolicLevel] = {}
        self.global_probability = np.empty(0, dtype=np.float32)
        self.fit_seconds = 0.0

    def _codes(self, data: np.ndarray, order: int) -> Tuple[np.ndarray, np.ndarray]:
        n = len(data) - order
        codes = np.zeros(n, dtype=np.uint64)
        base = np.uint64(self.vocab_size)
        for offset in range(order):
            codes = codes * base + data[offset : offset + n].astype(np.uint64)
        return codes, data[order:].astype(np.int64, copy=False)

    def _unique_counts(self, values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if device.type == "cuda" and len(values) >= cfg.gpu_unique_threshold:
            try:
                tensor = torch.from_numpy(values.view(np.int64)).to(device)
                unique, counts = torch.unique(tensor, sorted=True, return_counts=True)
                result_unique = unique.cpu().numpy().view(np.uint64)
                result_counts = counts.cpu().numpy().astype(np.int64, copy=False)
                del tensor, unique, counts
                torch.cuda.empty_cache()
                return result_unique, result_counts
            except RuntimeError as error:
                print(f" GPU unique fallback to CPU: {str(error).splitlines()[0]}")
                torch.cuda.empty_cache()
        return np.unique(values, return_counts=True)

    def _lookup(
        self, order: int, codes: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        n = len(codes)
        probabilities = np.repeat(self.global_probability[None, :], n, axis=0)
        supports = np.zeros(n, dtype=np.float32)
        used_orders = np.zeros(n, dtype=np.int16)
        unresolved = np.ones(n, dtype=bool)
        current_codes = codes.astype(np.uint64, copy=True)
        for current_order in range(order, 0, -1):
            level = self.levels.get(current_order)
            if level is not None and np.any(unresolved):
                rows = np.flatnonzero(unresolved)
                query = current_codes[rows]
                positions = np.searchsorted(level.codes, query)
                hit = positions < len(level.codes)
                if len(level.codes):
                    safe = np.minimum(positions, len(level.codes) - 1)
                    hit &= level.codes[safe] == query
                hit_rows = rows[hit]
                hit_positions = positions[hit]
                probabilities[hit_rows] = level.probabilities[hit_positions]
                supports[hit_rows] = level.totals[hit_positions]
                used_orders[hit_rows] = current_order
                unresolved[hit_rows] = False
            if current_order > 1:
                current_codes %= np.uint64(self.vocab_size ** (current_order - 1))
        return probabilities, supports, used_orders

    def fit(self, data: np.ndarray, verbose: bool = True) -> "MDLContextMachine":
        started = time.perf_counter()
        data = np.asarray(data, dtype=np.int64)
        counts = np.bincount(data, minlength=self.vocab_size).astype(np.float64)
        self.global_probability = (
            (counts + 0.15) / (counts.sum() + 0.15 * self.vocab_size)
        ).astype(np.float32)
        self.levels = {}
        for order in range(1, self.max_order + 1):
            order_started = time.perf_counter()
            codes, targets = self._codes(data, order)
            unique_codes, totals = self._unique_counts(codes)
            raw_unique = len(unique_codes)
            eligible_mask = totals >= cfg.min_count
            unique_codes, totals = unique_codes[eligible_mask], totals[eligible_mask]
            eligible_count = len(unique_codes)
            if eligible_count == 0:
                continue

            pre_cap = min(eligible_count, cfg.max_states_per_order * 4)
            if eligible_count > pre_cap:
                top = np.argpartition(totals, -pre_cap)[-pre_cap:]
                unique_codes, totals = unique_codes[top], totals[top]
                ordering = np.argsort(unique_codes)
                unique_codes, totals = unique_codes[ordering], totals[ordering]

            pair_keys = codes * np.uint64(self.vocab_size) + targets.astype(np.uint64)
            unique_pairs, pair_totals = self._unique_counts(pair_keys)
            del pair_keys, codes
            pair_context = unique_pairs // np.uint64(self.vocab_size)
            pair_target = (unique_pairs % np.uint64(self.vocab_size)).astype(np.int64)
            positions = np.searchsorted(unique_codes, pair_context)
            hit = positions < len(unique_codes)
            safe = np.minimum(positions, len(unique_codes) - 1)
            hit &= unique_codes[safe] == pair_context
            dense = np.zeros((len(unique_codes), self.vocab_size), dtype=np.float32)
            np.add.at(
                dense,
                (positions[hit], pair_target[hit]),
                pair_totals[hit].astype(np.float32),
            )
            del unique_pairs, pair_totals, pair_context, pair_target, positions, hit, safe

            if order == 1:
                base = np.repeat(self.global_probability[None, :], len(unique_codes), axis=0)
            else:
                suffix_codes = unique_codes % np.uint64(self.vocab_size ** (order - 1))
                base, _, _ = self._lookup(order - 1, suffix_codes)
            probabilities = (dense + cfg.backoff_strength * base) / (
                dense.sum(axis=1, keepdims=True) + cfg.backoff_strength
            )
            gains = np.sum(
                dense
                * (
                    np.log(np.clip(probabilities, 1e-12, 1.0))
                    - np.log(np.clip(base, 1e-12, 1.0))
                ),
                axis=1,
            )
            nonzero = np.sum(dense > 0, axis=1)
            penalty = cfg.definition_nats + 0.5 * np.maximum(nonzero - 1, 1) * np.log(
                totals + 1.0
            )
            keep = gains > penalty
            if not np.any(keep):
                continue
            kept = np.flatnonzero(keep)
            if len(kept) > cfg.max_states_per_order:
                score = gains[kept] - penalty[kept]
                kept = kept[
                    np.argpartition(score, -cfg.max_states_per_order)[
                        -cfg.max_states_per_order :
                    ]
                ]
            kept = kept[np.argsort(unique_codes[kept])]
            self.levels[order] = SymbolicLevel(
                codes=unique_codes[kept],
                probabilities=probabilities[kept].astype(np.float32),
                totals=totals[kept].astype(np.float32),
                gains=gains[kept].astype(np.float32),
                raw_unique=raw_unique,
                eligible=eligible_count,
            )
            if verbose:
                print(
                    f" order={order:2d} | raw={raw_unique:9,d} | eligible={eligible_count:9,d} | "
                    f"retained={len(kept):7,d} | median support={np.median(totals[kept]):7.1f} | "
                    f"{time.perf_counter() - order_started:6.1f}s"
                )
            del dense, base, probabilities, gains, penalty, nonzero
        self.fit_seconds = time.perf_counter() - started
        return self

    def predict_block(
        self, data: np.ndarray, maximum_order: Optional[int] = None
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        n_targets = len(data) - 1
        probabilities = np.repeat(self.global_probability[None, :], n_targets, axis=0)
        orders = np.zeros(n_targets, dtype=np.int16)
        supports = np.zeros(n_targets, dtype=np.float32)
        if maximum_order is None:
            maximum_order = self.max_order
        maximum_order = min(maximum_order, self.max_order, len(data) - 1)
        for order in range(1, maximum_order + 1):
            level = self.levels.get(order)
            if level is None:
                continue
            codes, _ = self._codes(data, order)
            positions = np.searchsorted(level.codes, codes)
            hit = positions < len(level.codes)
            safe = np.minimum(positions, len(level.codes) - 1)
            hit &= level.codes[safe] == codes
            rows = np.arange(order - 1, n_targets)[hit]
            probabilities[rows] = level.probabilities[positions[hit]]
            supports[rows] = level.totals[positions[hit]]
            orders[rows] = order
        return probabilities, supports, orders

    def predict_one(self, context: Sequence[int]) -> Tuple[np.ndarray, int]:
        usable = list(context)[-self.max_order :]
        for order in range(min(len(usable), self.max_order), 0, -1):
            code = np.uint64(0)
            for token in usable[-order:]:
                code = code * np.uint64(self.vocab_size) + np.uint64(token)
            level = self.levels.get(order)
            if level is None:
                continue
            position = int(np.searchsorted(level.codes, code))
            if position < len(level.codes) and level.codes[position] == code:
                return level.probabilities[position], order
        return self.global_probability, 0

    @property
    def state_count(self) -> int:
        return int(sum(len(level.codes) for level in self.levels.values()))

    @property
    def eligible_count(self) -> int:
        return int(sum(level.eligible for level in self.levels.values()))

    @property
    def model_bytes(self) -> int:
        total = self.global_probability.nbytes
        for level in self.levels.values():
            total += sum(
                array.nbytes
                for array in (level.codes, level.probabilities, level.totals, level.gains)
            )
        return int(total)


def evaluate(
    machine: MDLContextMachine,
    data: np.ndarray,
    maximum_order: Optional[int] = None,
) -> Dict[str, object]:
    total_nll = 0.0
    correct = 0
    count = 0
    order_hist = np.zeros(machine.max_order + 1, dtype=np.int64)
    for start in range(0, len(data) - 1, cfg.evaluation_block):
        stop = min(len(data), start + cfg.evaluation_block + 1)
        block = data[start:stop]
        if len(block) < 2:
            continue
        probabilities, _, orders = machine.predict_block(block, maximum_order=maximum_order)
        targets = block[1:]
        selected = probabilities[np.arange(len(targets)), targets]
        total_nll -= float(np.log(np.clip(selected, 1e-12, 1.0)).sum())
        correct += int((probabilities.argmax(axis=1) == targets).sum())
        count += len(targets)
        order_hist += np.bincount(orders, minlength=len(order_hist))
    nll = total_nll / count
    return {
        "nll": nll,
        "bpc": nll / math.log(2.0),
        "ppl": math.exp(nll),
        "top1": correct / count,
        "characters": count,
        "order_histogram": order_hist.tolist(),
    }


def transition_audit(machine: MDLContextMachine, trials: int = 2_000) -> float:
    """Audit that state evolution equals append-symbol then longest retained suffix."""
    rng = np.random.default_rng(cfg.seed + 11)
    levels = [order for order, level in machine.levels.items() if len(level.codes)]
    passed = 0
    for _ in range(trials):
        order = int(rng.choice(levels))
        level = machine.levels[order]
        code = int(level.codes[int(rng.integers(len(level.codes)))])
        symbols = [0] * order
        remaining = code
        for index in range(order - 1, -1, -1):
            symbols[index] = remaining % machine.vocab_size
            remaining //= machine.vocab_size
        emitted = int(rng.integers(machine.vocab_size))
        _, actual_order = machine.predict_one(symbols + [emitted])
        expected_order = 0
        for candidate_order in range(min(machine.max_order, order + 1), 0, -1):
            suffix = symbols[-(candidate_order - 1) :] + [emitted] if candidate_order > 1 else [emitted]
            suffix_code = 0
            for token in suffix:
                suffix_code = suffix_code * machine.vocab_size + token
            candidate_level = machine.levels.get(candidate_order)
            if candidate_level is None:
                continue
            position = int(np.searchsorted(candidate_level.codes, np.uint64(suffix_code)))
            if position < len(candidate_level.codes) and int(candidate_level.codes[position]) == suffix_code:
                expected_order = candidate_order
                break
        passed += int(actual_order == expected_order)
    return passed / trials


def generate(machine: MDLContextMachine, alphabet: List[str]) -> str:
    rng = np.random.default_rng(cfg.seed + 23)
    seed_text = "the meaning of"
    lookup = {character: index for index, character in enumerate(alphabet)}
    tokens = [lookup[character] for character in seed_text]
    for _ in range(cfg.generation_characters):
        probability, _ = machine.predict_one(tokens)
        logits = np.log(np.clip(probability, 1e-12, 1.0)) / cfg.generation_temperature
        probability = np.exp(logits - logits.max())
        probability /= probability.sum()
        tokens.append(int(rng.choice(len(alphabet), p=probability)))
    return "".join(alphabet[token] for token in tokens)


print("=" * 112)
print("MDL TEXT8 V12 — finite-state language model at the next scale")
print("=" * 112)
print(
    f"device={device} | torch={torch.__version__} | fast_dev={FAST_DEV_RUN} | "
    f"max_order={cfg.max_order} | cap/order={cfg.max_states_per_order:,}"
)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU={props.name} | VRAM={props.total_memory / 2**30:.1f} GiB | GPU counting=True")

raw = download_text8()
raw_digest = hashlib.sha256(raw).hexdigest()
data, alphabet = encode_text8(raw)
validation = data[
    cfg.validation_start : cfg.validation_start + cfg.validation_characters
]
test = data[cfg.test_start : cfg.test_start + cfg.test_characters]
print(
    f"corpus={len(data):,} chars | vocab={len(alphabet)} | conventional split=90M/5M/5M | "
    f"evaluated validation/test={len(validation):,}/{len(test):,}"
)

results: List[Dict[str, object]] = []
final_machine: Optional[MDLContextMachine] = None
for observations in cfg.fit_prefixes:
    print("\n" + "-" * 112)
    print(f"FIT PREFIX: {observations:,} one-pass character observations")
    machine = MDLContextMachine(len(alphabet)).fit(data[:observations], verbose=True)
    validation_metrics = evaluate(machine, validation)
    test_metrics = evaluate(machine, test)
    row = {
        "observations": observations,
        "states": machine.state_count,
        "eligible_contexts": machine.eligible_count,
        "compression_ratio": machine.eligible_count / max(machine.state_count, 1),
        "model_mib": machine.model_bytes / 2**20,
        "fit_seconds": machine.fit_seconds,
        "validation_nll": validation_metrics["nll"],
        "validation_bpc": validation_metrics["bpc"],
        "validation_top1": validation_metrics["top1"],
        "test_nll": test_metrics["nll"],
        "test_bpc": test_metrics["bpc"],
        "test_top1": test_metrics["top1"],
        "test_order_histogram": test_metrics["order_histogram"],
    }
    results.append(row)
    print(
        f"RESULT | states={machine.state_count:,} | eligible/retained={row['compression_ratio']:.2f}x | "
        f"model={row['model_mib']:.1f} MiB | fit={machine.fit_seconds:.1f}s | "
        f"val bpc={validation_metrics['bpc']:.4f} | test bpc={test_metrics['bpc']:.4f} | "
        f"test top1={100 * test_metrics['top1']:.2f}%"
    )
    final_machine = machine

assert final_machine is not None
unigram_test = evaluate(final_machine, test, maximum_order=0)
order4_test = evaluate(final_machine, test, maximum_order=min(4, cfg.max_order))
full_test = evaluate(final_machine, test)
audit_accuracy = transition_audit(final_machine)
sample = generate(final_machine, alphabet)

test_hist = np.asarray(full_test["order_histogram"], dtype=np.float64)
test_usage = test_hist / max(test_hist.sum(), 1)
monotone_scaling = all(
    results[index + 1]["test_bpc"] <= results[index]["test_bpc"] + 0.01
    for index in range(len(results) - 1)
)
predictions = {
    "P1_nontrivial_finite_machine": final_machine.state_count >= 10_000,
    "P2_one_pass_scaling_improves_test_bpc": monotone_scaling
    and results[-1]["test_bpc"] < results[0]["test_bpc"],
    "P3_deep_symbolic_states_beat_order4": full_test["bpc"] < order4_test["bpc"],
    "P4_machine_beats_unigram_by_20pct": full_test["bpc"] <= 0.80 * unigram_test["bpc"],
    "P5_mdl_compresses_candidate_contexts": results[-1]["compression_ratio"] >= 2.0,
    "P6_exact_symbolic_transition_audit": audit_accuracy == 1.0,
}

print("\n" + "=" * 112)
print("FINAL HELD-OUT TEST — MDL MACHINE ONLY")
print("=" * 112)
print(f"unigram       | bpc={unigram_test['bpc']:.4f} | top1={100 * unigram_test['top1']:.2f}%")
print(f"MDL order<=4  | bpc={order4_test['bpc']:.4f} | top1={100 * order4_test['top1']:.2f}%")
print(f"MDL order<={cfg.max_order:<2d} | bpc={full_test['bpc']:.4f} | top1={100 * full_test['top1']:.2f}%")
print(
    f"states={final_machine.state_count:,} | model={final_machine.model_bytes / 2**20:.1f} MiB | "
    f"eligible/retained={results[-1]['compression_ratio']:.2f}x | transition audit={100*audit_accuracy:.2f}%"
)
print("\nPREDECLARED PREDICTIONS")
for name, passed in predictions.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")
print("\nGENERATED SAMPLE")
print(sample)

with (out_dir / "scaling.csv").open("w", newline="") as handle:
    fieldnames = [key for key in results[0] if key != "test_order_histogram"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for row in results:
        writer.writerow({key: row[key] for key in fieldnames})

state_rows = []
for order, level in final_machine.levels.items():
    state_rows.append(
        {
            "order": order,
            "raw_unique": level.raw_unique,
            "eligible": level.eligible,
            "retained": len(level.codes),
            "median_support": float(np.median(level.totals)),
            "median_gain_nats": float(np.median(level.gains)),
            "test_usage": float(test_usage[order]),
        }
    )
with (out_dir / "states_by_order.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(state_rows[0]))
    writer.writeheader()
    writer.writerows(state_rows)

summary = {
    "config": asdict(cfg),
    "corpus_sha256": raw_digest,
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "scaling": results,
    "final": {
        "unigram": unigram_test,
        "order4": order4_test,
        "full": full_test,
        "states": final_machine.state_count,
        "eligible_contexts": final_machine.eligible_count,
        "model_bytes": final_machine.model_bytes,
        "transition_audit": audit_accuracy,
    },
    "predictions": predictions,
    "generated_sample": sample,
}
(out_dir / "results.json").write_text(json.dumps(summary, indent=2))
(out_dir / "generated_sample.txt").write_text(sample)
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

figure, axes = plt.subplots(2, 2, figsize=(15, 10))
observations = np.asarray([row["observations"] for row in results])
test_bpc = np.asarray([row["test_bpc"] for row in results])
validation_bpc = np.asarray([row["validation_bpc"] for row in results])
states = np.asarray([row["states"] for row in results])

axes[0, 0].plot(observations, validation_bpc, "o-", label="validation")
axes[0, 0].plot(observations, test_bpc, "o-", label="test")
axes[0, 0].set_xscale("log")
axes[0, 0].set_xlabel("one-pass character observations")
axes[0, 0].set_ylabel("bits per character")
axes[0, 0].set_title("Held-out scaling")
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend()

axes[0, 1].plot(observations, states, "o-", color="#59a14f")
axes[0, 1].set_xscale("log")
axes[0, 1].set_xlabel("one-pass character observations")
axes[0, 1].set_ylabel("retained finite states")
axes[0, 1].set_title("MDL state growth")
axes[0, 1].grid(alpha=0.25)

orders = np.arange(len(test_usage))
axes[1, 0].bar(orders, test_usage, color="#b07aa1")
axes[1, 0].set_xlabel("longest retained suffix used")
axes[1, 0].set_ylabel("fraction of held-out characters")
axes[1, 0].set_title("Which symbolic depths actually fire")

labels = ["unigram", "order<=4", f"order<={cfg.max_order}"]
values = [unigram_test["bpc"], order4_test["bpc"], full_test["bpc"]]
axes[1, 1].bar(labels, values, color=["#9c9c9c", "#f2c14e", "#59a14f"])
axes[1, 1].set_ylabel("test bits per character")
axes[1, 1].set_title("Depth ablation (lower is better)")
axes[1, 1].grid(axis="y", alpha=0.25)

figure.suptitle(
    f"MDL finite-state language model on text8 — {cfg.fit_prefixes[-1]:,} observations, "
    f"{final_machine.state_count:,} retained states",
    fontsize=15,
)
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(out_dir / "summary.png", dpi=170)
plt.show()

for source_name in ("results.json", "config.json", "scaling.csv", "states_by_order.csv", "generated_sample.txt", "summary.png"):
    if not (out_dir / source_name).exists():
        raise RuntimeError(f"Missing expected artifact: {source_name}")
shutil.make_archive(str(out_dir / "run_bundle"), "zip", root_dir=out_dir)
print("\n" + "=" * 112)
print("V12 COMPLETE")
print("=" * 112)
print(f"artifacts: {out_dir}")
print(f"bundle: {out_dir / 'run_bundle.zip'}")
"""
MDL TEXT8 V12 — A100/T4-ACCELERATED, SINGLE-CELL EXPERIMENT

Purpose
-------
Move the finite-state MDL mechanism beyond Tiny Shakespeare and test it by
itself on text8: 100,000,000 Wikipedia characters with the conventional
90M/5M/5M chronological split.

This is not a neural comparison.  It asks whether a one-pass variable-order
probabilistic state machine still (1) scales, (2) compresses its candidate
contexts, (3) benefits from deeper symbolic states, and (4) generalizes to a
far-away held-out test segment.  GPU sorting/counting is used when CUDA is
available; the learned object remains an explicit finite-state machine.

Environment overrides
---------------------
MDL_TEXT8_FAST_DEV_RUN=1     small smoke test
MDL_TEXT8_MAX_FIT=90000000   largest training prefix (A100 default)
MDL_TEXT8_STATE_CAP=250000   maximum retained states per order (A100 default)
MDL_TEXT8_MAX_ORDER=12       maximum symbolic suffix length
"""


## Experiment 4: MDL text8 scaling V12 — 250,000-state cap

Original cell `3`, split part `2`.

**Cleanup note.** Two complete programs were concatenated in the source cell; they are separated here so each has valid future imports and independent artifacts.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch


FAST_DEV_RUN = os.environ.get("MDL_TEXT8_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 20260802
    data_url: str = "https://mattmahoney.net/dc/text8.zip"
    archive_path: str = "outputs/04-mdl-text8-scaling-v12-250-000-state-cap/text8.zip"
    data_path: str = "outputs/04-mdl-text8-scaling-v12-250-000-state-cap/text8"
    output_dir: str = "outputs/04-mdl-text8-scaling-v12-250-000-state-cap/mdl_text8_v12_results"
    expected_characters: int = 100_000_000
    train_end: int = 90_000_000
    validation_start: int = 90_000_000
    validation_characters: int = 1_000_000
    test_start: int = 95_000_000
    test_characters: int = 1_000_000
    fit_prefixes: Tuple[int, ...] = (1_000_000, 3_000_000, 10_000_000)
    max_order: int = 12
    min_count: int = 3
    max_states_per_order: int = 100_000
    backoff_strength: float = 32.0
    definition_nats: float = 2.0
    evaluation_block: int = 65_536
    generation_characters: int = 700
    generation_temperature: float = 0.90
    gpu_unique_threshold: int = 250_000


cfg = Config()
cfg = replace(
    cfg,
    fit_prefixes=tuple(
        sorted(
            {
                min(int(os.environ.get("MDL_TEXT8_MAX_FIT", cfg.fit_prefixes[-1])), n)
                for n in cfg.fit_prefixes
            }
        )
    ),
    max_states_per_order=int(
        os.environ.get("MDL_TEXT8_STATE_CAP", cfg.max_states_per_order)
    ),
    max_order=int(os.environ.get("MDL_TEXT8_MAX_ORDER", cfg.max_order)),
)
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        output_dir=str(Path.cwd() / "mdl_text8_v12_smoke"),
        expected_characters=0,
        train_end=700_000,
        validation_start=700_000,
        validation_characters=100_000,
        test_start=800_000,
        test_characters=100_000,
        fit_prefixes=(100_000, 300_000, 600_000),
        max_order=8,
        max_states_per_order=10_000,
        evaluation_block=16_384,
        generation_characters=250,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
a100_mode = "A100" in gpu_name.upper() and not FAST_DEV_RUN
if a100_mode:
    requested_max_fit = int(os.environ.get("MDL_TEXT8_MAX_FIT", "90000000"))
    a100_prefixes = tuple(
        sorted({min(requested_max_fit, value) for value in (1_000_000, 10_000_000, 30_000_000, 90_000_000)})
    )
    cfg = replace(
        cfg,
        fit_prefixes=a100_prefixes,
        validation_characters=5_000_000,
        test_characters=5_000_000,
        max_states_per_order=int(os.environ.get("MDL_TEXT8_STATE_CAP", "250000")),
        evaluation_block=262_144,
        gpu_unique_threshold=0,
    )
out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)


def download_text8() -> bytes:
    data_path = Path(cfg.data_path)
    archive_path = Path(cfg.archive_path)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    if not data_path.exists():
        if not archive_path.exists():
            print(f"downloading text8 -> {archive_path}")
            urllib.request.urlretrieve(cfg.data_url, archive_path)
        with zipfile.ZipFile(archive_path, "r") as archive:
            member = archive.getinfo("text8")
            if member.file_size != 100_000_000:
                raise RuntimeError(f"Unexpected text8 member size: {member.file_size:,}")
            archive.extract("text8", data_path.parent)
    raw = data_path.read_bytes()
    if not FAST_DEV_RUN and len(raw) != cfg.expected_characters:
        raise RuntimeError(f"Expected 100,000,000 characters, found {len(raw):,}")
    return raw


def encode_text8(raw: bytes) -> Tuple[np.ndarray, List[str]]:
    byte_values = np.frombuffer(raw, dtype=np.uint8)
    alphabet_bytes = np.unique(byte_values)
    expected = np.asarray([32] + list(range(97, 123)), dtype=np.uint8)
    if not np.array_equal(alphabet_bytes, expected):
        raise RuntimeError(f"Unexpected text8 alphabet: {alphabet_bytes.tolist()}")
    lookup = np.full(256, -1, dtype=np.int16)
    lookup[expected] = np.arange(len(expected), dtype=np.int16)
    ids = lookup[byte_values].astype(np.int64)
    alphabet = [chr(int(value)) for value in expected]
    return ids, alphabet


@dataclass
class SymbolicLevel:
    codes: np.ndarray
    probabilities: np.ndarray
    totals: np.ndarray
    gains: np.ndarray
    raw_unique: int
    eligible: int


class MDLContextMachine:
    """Explicit variable-order probabilistic automaton with MDL state admission."""

    def __init__(self, vocab_size: int, max_order: Optional[int] = None) -> None:
        self.vocab_size = int(vocab_size)
        self.max_order = int(max_order or cfg.max_order)
        if self.vocab_size ** (self.max_order + 1) >= 2**63:
            raise ValueError("Context encoding would overflow signed int64")
        self.levels: Dict[int, SymbolicLevel] = {}
        self.global_probability = np.empty(0, dtype=np.float32)
        self.fit_seconds = 0.0

    def _codes(self, data: np.ndarray, order: int) -> Tuple[np.ndarray, np.ndarray]:
        n = len(data) - order
        codes = np.zeros(n, dtype=np.uint64)
        base = np.uint64(self.vocab_size)
        for offset in range(order):
            codes = codes * base + data[offset : offset + n].astype(np.uint64)
        return codes, data[order:].astype(np.int64, copy=False)

    def _unique_counts(self, values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if device.type == "cuda" and len(values) >= cfg.gpu_unique_threshold:
            try:
                tensor = torch.from_numpy(values.view(np.int64)).to(device)
                unique, counts = torch.unique(tensor, sorted=True, return_counts=True)
                result_unique = unique.cpu().numpy().view(np.uint64)
                result_counts = counts.cpu().numpy().astype(np.int64, copy=False)
                del tensor, unique, counts
                return result_unique, result_counts
            except RuntimeError as error:
                print(f" GPU unique fallback to CPU: {str(error).splitlines()[0]}")
                torch.cuda.empty_cache()
        return np.unique(values, return_counts=True)

    def _candidate_contexts(
        self, values: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, int, int]:
        """Count contexts and cap them on GPU before copying to host memory."""
        pre_cap_limit = cfg.max_states_per_order * 4
        if device.type == "cuda" and len(values) >= cfg.gpu_unique_threshold:
            try:
                tensor = torch.from_numpy(values.view(np.int64)).to(device)
                unique, counts = torch.unique(tensor, sorted=True, return_counts=True)
                raw_unique = int(unique.numel())
                eligible_mask = counts >= cfg.min_count
                eligible_count = int(eligible_mask.sum().item())
                unique = unique[eligible_mask]
                counts = counts[eligible_mask]
                if unique.numel() > pre_cap_limit:
                    top = torch.topk(counts, k=pre_cap_limit, sorted=False).indices
                    unique = unique[top]
                    counts = counts[top]
                    ordering = torch.argsort(unique)
                    unique = unique[ordering]
                    counts = counts[ordering]
                result_unique = unique.cpu().numpy().view(np.uint64)
                result_counts = counts.cpu().numpy().astype(np.int64, copy=False)
                del tensor, unique, counts
                return result_unique, result_counts, raw_unique, eligible_count
            except RuntimeError as error:
                print(f" GPU context-count fallback to CPU: {str(error).splitlines()[0]}")
                torch.cuda.empty_cache()

        unique, counts = np.unique(values, return_counts=True)
        raw_unique = len(unique)
        eligible_mask = counts >= cfg.min_count
        unique, counts = unique[eligible_mask], counts[eligible_mask]
        eligible_count = len(unique)
        if len(unique) > pre_cap_limit:
            top = np.argpartition(counts, -pre_cap_limit)[-pre_cap_limit:]
            unique, counts = unique[top], counts[top]
            ordering = np.argsort(unique)
            unique, counts = unique[ordering], counts[ordering]
        return unique, counts, raw_unique, eligible_count

    def _pair_counts(
        self,
        codes: np.ndarray,
        targets: np.ndarray,
        candidate_codes: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Count transitions; on GPU, discard pairs outside the capped context set before host copy."""
        if device.type == "cuda" and len(codes) >= cfg.gpu_unique_threshold:
            try:
                code_tensor = torch.from_numpy(codes.view(np.int64)).to(device)
                target_tensor = torch.from_numpy(targets).to(device)
                pair_tensor = code_tensor * self.vocab_size + target_tensor
                unique_pairs, pair_totals = torch.unique(
                    pair_tensor, sorted=True, return_counts=True
                )
                candidate_tensor = torch.from_numpy(candidate_codes.view(np.int64)).to(device)
                pair_context = torch.div(
                    unique_pairs, self.vocab_size, rounding_mode="floor"
                )
                positions = torch.searchsorted(candidate_tensor, pair_context)
                safe = positions.clamp(max=max(int(candidate_tensor.numel()) - 1, 0))
                hit = (positions < candidate_tensor.numel()) & (
                    candidate_tensor[safe] == pair_context
                )
                result_pairs = unique_pairs[hit].cpu().numpy().view(np.uint64)
                result_totals = pair_totals[hit].cpu().numpy().astype(np.int64, copy=False)
                del (
                    code_tensor,
                    target_tensor,
                    pair_tensor,
                    unique_pairs,
                    pair_totals,
                    candidate_tensor,
                    pair_context,
                    positions,
                    safe,
                    hit,
                )
                return result_pairs, result_totals
            except RuntimeError as error:
                print(f" GPU pair-count fallback to CPU: {str(error).splitlines()[0]}")
                torch.cuda.empty_cache()

        pair_keys = codes * np.uint64(self.vocab_size) + targets.astype(np.uint64)
        unique_pairs, pair_totals = np.unique(pair_keys, return_counts=True)
        pair_context = unique_pairs // np.uint64(self.vocab_size)
        positions = np.searchsorted(candidate_codes, pair_context)
        safe = np.minimum(positions, max(len(candidate_codes) - 1, 0))
        hit = (positions < len(candidate_codes)) & (candidate_codes[safe] == pair_context)
        return unique_pairs[hit], pair_totals[hit]

    def _lookup(
        self, order: int, codes: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        n = len(codes)
        probabilities = np.repeat(self.global_probability[None, :], n, axis=0)
        supports = np.zeros(n, dtype=np.float32)
        used_orders = np.zeros(n, dtype=np.int16)
        unresolved = np.ones(n, dtype=bool)
        current_codes = codes.astype(np.uint64, copy=True)
        for current_order in range(order, 0, -1):
            level = self.levels.get(current_order)
            if level is not None and np.any(unresolved):
                rows = np.flatnonzero(unresolved)
                query = current_codes[rows]
                positions = np.searchsorted(level.codes, query)
                hit = positions < len(level.codes)
                if len(level.codes):
                    safe = np.minimum(positions, len(level.codes) - 1)
                    hit &= level.codes[safe] == query
                hit_rows = rows[hit]
                hit_positions = positions[hit]
                probabilities[hit_rows] = level.probabilities[hit_positions]
                supports[hit_rows] = level.totals[hit_positions]
                used_orders[hit_rows] = current_order
                unresolved[hit_rows] = False
            if current_order > 1:
                current_codes %= np.uint64(self.vocab_size ** (current_order - 1))
        return probabilities, supports, used_orders

    def fit(self, data: np.ndarray, verbose: bool = True) -> "MDLContextMachine":
        started = time.perf_counter()
        data = np.asarray(data, dtype=np.int64)
        counts = np.bincount(data, minlength=self.vocab_size).astype(np.float64)
        self.global_probability = (
            (counts + 0.15) / (counts.sum() + 0.15 * self.vocab_size)
        ).astype(np.float32)
        self.levels = {}
        for order in range(1, self.max_order + 1):
            order_started = time.perf_counter()
            codes, targets = self._codes(data, order)
            unique_codes, totals, raw_unique, eligible_count = self._candidate_contexts(codes)
            if eligible_count == 0:
                continue

            unique_pairs, pair_totals = self._pair_counts(codes, targets, unique_codes)
            del codes
            pair_context = unique_pairs // np.uint64(self.vocab_size)
            pair_target = (unique_pairs % np.uint64(self.vocab_size)).astype(np.int64)
            positions = np.searchsorted(unique_codes, pair_context)
            hit = positions < len(unique_codes)
            safe = np.minimum(positions, len(unique_codes) - 1)
            hit &= unique_codes[safe] == pair_context
            dense = np.zeros((len(unique_codes), self.vocab_size), dtype=np.float32)
            np.add.at(
                dense,
                (positions[hit], pair_target[hit]),
                pair_totals[hit].astype(np.float32),
            )
            del unique_pairs, pair_totals, pair_context, pair_target, positions, hit, safe

            if order == 1:
                base = np.repeat(self.global_probability[None, :], len(unique_codes), axis=0)
            else:
                suffix_codes = unique_codes % np.uint64(self.vocab_size ** (order - 1))
                base, _, _ = self._lookup(order - 1, suffix_codes)
            probabilities = (dense + cfg.backoff_strength * base) / (
                dense.sum(axis=1, keepdims=True) + cfg.backoff_strength
            )
            gains = np.sum(
                dense
                * (
                    np.log(np.clip(probabilities, 1e-12, 1.0))
                    - np.log(np.clip(base, 1e-12, 1.0))
                ),
                axis=1,
            )
            nonzero = np.sum(dense > 0, axis=1)
            penalty = cfg.definition_nats + 0.5 * np.maximum(nonzero - 1, 1) * np.log(
                totals + 1.0
            )
            keep = gains > penalty
            if not np.any(keep):
                continue
            kept = np.flatnonzero(keep)
            if len(kept) > cfg.max_states_per_order:
                score = gains[kept] - penalty[kept]
                kept = kept[
                    np.argpartition(score, -cfg.max_states_per_order)[
                        -cfg.max_states_per_order :
                    ]
                ]
            kept = kept[np.argsort(unique_codes[kept])]
            self.levels[order] = SymbolicLevel(
                codes=unique_codes[kept],
                probabilities=probabilities[kept].astype(np.float32),
                totals=totals[kept].astype(np.float32),
                gains=gains[kept].astype(np.float32),
                raw_unique=raw_unique,
                eligible=eligible_count,
            )
            if verbose:
                print(
                    f" order={order:2d} | raw={raw_unique:9,d} | eligible={eligible_count:9,d} | "
                    f"retained={len(kept):7,d} | median support={np.median(totals[kept]):7.1f} | "
                    f"{time.perf_counter() - order_started:6.1f}s"
                )
            del dense, base, probabilities, gains, penalty, nonzero
        self.fit_seconds = time.perf_counter() - started
        return self

    def predict_block(
        self, data: np.ndarray, maximum_order: Optional[int] = None
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        n_targets = len(data) - 1
        probabilities = np.repeat(self.global_probability[None, :], n_targets, axis=0)
        orders = np.zeros(n_targets, dtype=np.int16)
        supports = np.zeros(n_targets, dtype=np.float32)
        if maximum_order is None:
            maximum_order = self.max_order
        maximum_order = min(maximum_order, self.max_order, len(data) - 1)
        for order in range(1, maximum_order + 1):
            level = self.levels.get(order)
            if level is None:
                continue
            codes, _ = self._codes(data, order)
            positions = np.searchsorted(level.codes, codes)
            hit = positions < len(level.codes)
            safe = np.minimum(positions, len(level.codes) - 1)
            hit &= level.codes[safe] == codes
            rows = np.arange(order - 1, n_targets)[hit]
            probabilities[rows] = level.probabilities[positions[hit]]
            supports[rows] = level.totals[positions[hit]]
            orders[rows] = order
        return probabilities, supports, orders

    def predict_one(self, context: Sequence[int]) -> Tuple[np.ndarray, int]:
        usable = list(context)[-self.max_order :]
        for order in range(min(len(usable), self.max_order), 0, -1):
            code = np.uint64(0)
            for token in usable[-order:]:
                code = code * np.uint64(self.vocab_size) + np.uint64(token)
            level = self.levels.get(order)
            if level is None:
                continue
            position = int(np.searchsorted(level.codes, code))
            if position < len(level.codes) and level.codes[position] == code:
                return level.probabilities[position], order
        return self.global_probability, 0

    @property
    def state_count(self) -> int:
        return int(sum(len(level.codes) for level in self.levels.values()))

    @property
    def eligible_count(self) -> int:
        return int(sum(level.eligible for level in self.levels.values()))

    @property
    def model_bytes(self) -> int:
        total = self.global_probability.nbytes
        for level in self.levels.values():
            total += sum(
                array.nbytes
                for array in (level.codes, level.probabilities, level.totals, level.gains)
            )
        return int(total)


def evaluate(
    machine: MDLContextMachine,
    data: np.ndarray,
    maximum_order: Optional[int] = None,
) -> Dict[str, object]:
    total_nll = 0.0
    correct = 0
    count = 0
    order_hist = np.zeros(machine.max_order + 1, dtype=np.int64)
    for start in range(0, len(data) - 1, cfg.evaluation_block):
        stop = min(len(data), start + cfg.evaluation_block + 1)
        block = data[start:stop]
        if len(block) < 2:
            continue
        probabilities, _, orders = machine.predict_block(block, maximum_order=maximum_order)
        targets = block[1:]
        selected = probabilities[np.arange(len(targets)), targets]
        total_nll -= float(np.log(np.clip(selected, 1e-12, 1.0)).sum())
        correct += int((probabilities.argmax(axis=1) == targets).sum())
        count += len(targets)
        order_hist += np.bincount(orders, minlength=len(order_hist))
    nll = total_nll / count
    return {
        "nll": nll,
        "bpc": nll / math.log(2.0),
        "ppl": math.exp(nll),
        "top1": correct / count,
        "characters": count,
        "order_histogram": order_hist.tolist(),
    }


def transition_audit(machine: MDLContextMachine, trials: int = 2_000) -> float:
    """Audit that state evolution equals append-symbol then longest retained suffix."""
    rng = np.random.default_rng(cfg.seed + 11)
    levels = [order for order, level in machine.levels.items() if len(level.codes)]
    passed = 0
    for _ in range(trials):
        order = int(rng.choice(levels))
        level = machine.levels[order]
        code = int(level.codes[int(rng.integers(len(level.codes)))])
        symbols = [0] * order
        remaining = code
        for index in range(order - 1, -1, -1):
            symbols[index] = remaining % machine.vocab_size
            remaining //= machine.vocab_size
        emitted = int(rng.integers(machine.vocab_size))
        _, actual_order = machine.predict_one(symbols + [emitted])
        expected_order = 0
        for candidate_order in range(min(machine.max_order, order + 1), 0, -1):
            suffix = symbols[-(candidate_order - 1) :] + [emitted] if candidate_order > 1 else [emitted]
            suffix_code = 0
            for token in suffix:
                suffix_code = suffix_code * machine.vocab_size + token
            candidate_level = machine.levels.get(candidate_order)
            if candidate_level is None:
                continue
            position = int(np.searchsorted(candidate_level.codes, np.uint64(suffix_code)))
            if position < len(candidate_level.codes) and int(candidate_level.codes[position]) == suffix_code:
                expected_order = candidate_order
                break
        passed += int(actual_order == expected_order)
    return passed / trials


def generate(machine: MDLContextMachine, alphabet: List[str]) -> str:
    rng = np.random.default_rng(cfg.seed + 23)
    seed_text = "the meaning of"
    lookup = {character: index for index, character in enumerate(alphabet)}
    tokens = [lookup[character] for character in seed_text]
    for _ in range(cfg.generation_characters):
        probability, _ = machine.predict_one(tokens)
        logits = np.log(np.clip(probability, 1e-12, 1.0)) / cfg.generation_temperature
        probability = np.exp(logits - logits.max())
        probability /= probability.sum()
        tokens.append(int(rng.choice(len(alphabet), p=probability)))
    return "".join(alphabet[token] for token in tokens)


print("=" * 112)
print("MDL TEXT8 V12 — finite-state language model at the next scale")
print("=" * 112)
print(
    f"device={device} | torch={torch.__version__} | fast_dev={FAST_DEV_RUN} | "
    f"max_order={cfg.max_order} | cap/order={cfg.max_states_per_order:,}"
)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU={props.name} | VRAM={props.total_memory / 2**30:.1f} GiB | GPU counting=True")

raw = download_text8()
raw_digest = hashlib.sha256(raw).hexdigest()
data, alphabet = encode_text8(raw)
validation = data[
    cfg.validation_start : cfg.validation_start + cfg.validation_characters
]
test = data[cfg.test_start : cfg.test_start + cfg.test_characters]
print(
    f"corpus={len(data):,} chars | vocab={len(alphabet)} | conventional split=90M/5M/5M | "
    f"evaluated validation/test={len(validation):,}/{len(test):,}"
)

results: List[Dict[str, object]] = []
final_machine: Optional[MDLContextMachine] = None
for observations in cfg.fit_prefixes:
    print("\n" + "-" * 112)
    print(f"FIT PREFIX: {observations:,} one-pass character observations")
    machine = MDLContextMachine(len(alphabet)).fit(data[:observations], verbose=True)
    validation_metrics = evaluate(machine, validation)
    test_metrics = evaluate(machine, test)
    row = {
        "observations": observations,
        "states": machine.state_count,
        "eligible_contexts": machine.eligible_count,
        "compression_ratio": machine.eligible_count / max(machine.state_count, 1),
        "model_mib": machine.model_bytes / 2**20,
        "fit_seconds": machine.fit_seconds,
        "validation_nll": validation_metrics["nll"],
        "validation_bpc": validation_metrics["bpc"],
        "validation_top1": validation_metrics["top1"],
        "test_nll": test_metrics["nll"],
        "test_bpc": test_metrics["bpc"],
        "test_top1": test_metrics["top1"],
        "test_order_histogram": test_metrics["order_histogram"],
    }
    results.append(row)
    print(
        f"RESULT | states={machine.state_count:,} | eligible/retained={row['compression_ratio']:.2f}x | "
        f"model={row['model_mib']:.1f} MiB | fit={machine.fit_seconds:.1f}s | "
        f"val bpc={validation_metrics['bpc']:.4f} | test bpc={test_metrics['bpc']:.4f} | "
        f"test top1={100 * test_metrics['top1']:.2f}%"
    )
    final_machine = machine

assert final_machine is not None
unigram_test = evaluate(final_machine, test, maximum_order=0)
order4_test = evaluate(final_machine, test, maximum_order=min(4, cfg.max_order))
full_test = evaluate(final_machine, test)
audit_accuracy = transition_audit(final_machine)
sample = generate(final_machine, alphabet)

test_hist = np.asarray(full_test["order_histogram"], dtype=np.float64)
test_usage = test_hist / max(test_hist.sum(), 1)
monotone_scaling = all(
    results[index + 1]["test_bpc"] <= results[index]["test_bpc"] + 0.01
    for index in range(len(results) - 1)
)
predictions = {
    "P1_nontrivial_finite_machine": final_machine.state_count >= 10_000,
    "P2_one_pass_scaling_improves_test_bpc": monotone_scaling
    and results[-1]["test_bpc"] < results[0]["test_bpc"],
    "P3_deep_symbolic_states_beat_order4": full_test["bpc"] < order4_test["bpc"],
    "P4_machine_beats_unigram_by_20pct": full_test["bpc"] <= 0.80 * unigram_test["bpc"],
    "P5_mdl_compresses_candidate_contexts": results[-1]["compression_ratio"] >= 2.0,
    "P6_exact_symbolic_transition_audit": audit_accuracy == 1.0,
}

print("\n" + "=" * 112)
print("FINAL HELD-OUT TEST — MDL MACHINE ONLY")
print("=" * 112)
print(f"unigram       | bpc={unigram_test['bpc']:.4f} | top1={100 * unigram_test['top1']:.2f}%")
print(f"MDL order<=4  | bpc={order4_test['bpc']:.4f} | top1={100 * order4_test['top1']:.2f}%")
print(f"MDL order<={cfg.max_order:<2d} | bpc={full_test['bpc']:.4f} | top1={100 * full_test['top1']:.2f}%")
print(
    f"states={final_machine.state_count:,} | model={final_machine.model_bytes / 2**20:.1f} MiB | "
    f"eligible/retained={results[-1]['compression_ratio']:.2f}x | transition audit={100*audit_accuracy:.2f}%"
)
print("\nPREDECLARED PREDICTIONS")
for name, passed in predictions.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")
print("\nGENERATED SAMPLE")
print(sample)

with (out_dir / "scaling.csv").open("w", newline="") as handle:
    fieldnames = [key for key in results[0] if key != "test_order_histogram"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for row in results:
        writer.writerow({key: row[key] for key in fieldnames})

state_rows = []
for order, level in final_machine.levels.items():
    state_rows.append(
        {
            "order": order,
            "raw_unique": level.raw_unique,
            "eligible": level.eligible,
            "retained": len(level.codes),
            "median_support": float(np.median(level.totals)),
            "median_gain_nats": float(np.median(level.gains)),
            "test_usage": float(test_usage[order]),
        }
    )
with (out_dir / "states_by_order.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(state_rows[0]))
    writer.writeheader()
    writer.writerows(state_rows)

summary = {
    "config": asdict(cfg),
    "corpus_sha256": raw_digest,
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "scaling": results,
    "final": {
        "unigram": unigram_test,
        "order4": order4_test,
        "full": full_test,
        "states": final_machine.state_count,
        "eligible_contexts": final_machine.eligible_count,
        "model_bytes": final_machine.model_bytes,
        "transition_audit": audit_accuracy,
    },
    "predictions": predictions,
    "generated_sample": sample,
}
(out_dir / "results.json").write_text(json.dumps(summary, indent=2))
(out_dir / "generated_sample.txt").write_text(sample)
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

figure, axes = plt.subplots(2, 2, figsize=(15, 10))
observations = np.asarray([row["observations"] for row in results])
test_bpc = np.asarray([row["test_bpc"] for row in results])
validation_bpc = np.asarray([row["validation_bpc"] for row in results])
states = np.asarray([row["states"] for row in results])

axes[0, 0].plot(observations, validation_bpc, "o-", label="validation")
axes[0, 0].plot(observations, test_bpc, "o-", label="test")
axes[0, 0].set_xscale("log")
axes[0, 0].set_xlabel("one-pass character observations")
axes[0, 0].set_ylabel("bits per character")
axes[0, 0].set_title("Held-out scaling")
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend()

axes[0, 1].plot(observations, states, "o-", color="#59a14f")
axes[0, 1].set_xscale("log")
axes[0, 1].set_xlabel("one-pass character observations")
axes[0, 1].set_ylabel("retained finite states")
axes[0, 1].set_title("MDL state growth")
axes[0, 1].grid(alpha=0.25)

orders = np.arange(len(test_usage))
axes[1, 0].bar(orders, test_usage, color="#b07aa1")
axes[1, 0].set_xlabel("longest retained suffix used")
axes[1, 0].set_ylabel("fraction of held-out characters")
axes[1, 0].set_title("Which symbolic depths actually fire")

labels = ["unigram", "order<=4", f"order<={cfg.max_order}"]
values = [unigram_test["bpc"], order4_test["bpc"], full_test["bpc"]]
axes[1, 1].bar(labels, values, color=["#9c9c9c", "#f2c14e", "#59a14f"])
axes[1, 1].set_ylabel("test bits per character")
axes[1, 1].set_title("Depth ablation (lower is better)")
axes[1, 1].grid(axis="y", alpha=0.25)

figure.suptitle(
    f"MDL finite-state language model on text8 — {cfg.fit_prefixes[-1]:,} observations, "
    f"{final_machine.state_count:,} retained states",
    fontsize=15,
)
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(out_dir / "summary.png", dpi=170)
plt.show()

for source_name in ("results.json", "config.json", "scaling.csv", "states_by_order.csv", "generated_sample.txt", "summary.png"):
    if not (out_dir / source_name).exists():
        raise RuntimeError(f"Missing expected artifact: {source_name}")
shutil.make_archive(str(out_dir / "run_bundle"), "zip", root_dir=out_dir)
print("\n" + "=" * 112)
print("V12 COMPLETE")
print("=" * 112)
print(f"artifacts: {out_dir}")
print(f"bundle: {out_dir / 'run_bundle.zip'}")


## Experiment 5: Neural-quotient active learner V13

Original cell `4`.


In [ ]:
"""
NEURAL-QUOTIENT ACTIVE LEARNER V13 — SINGLE-CELL A100 EXPERIMENT

Breakthrough test
-----------------
Treat an imperfect neural transition table as a noisy codeword on a structured
program manifold.  Decode that codeword against a broad finite-machine grammar,
ask the environment only the maximally informative missing transitions, quotient
the verified rule by arbitrary state renaming, and retain recurring rules by MDL.

The decisive transfer test uses coordinate permutations never seen by the neural
learner.  The system must recover a complete 24-transition world from substantially
fewer than 24 observations, reject a genuinely novel law, and reorganize only when
the same residual law recurs.

What is learned and what is supplied
------------------------------------
Learned: a real in-context Transformer transition predictor; recurring canonical
templates; a held-out calibrated neural/symbolic responsibility gate.

Supplied: eight states, three actions, a finite grammar of candidate deterministic
machines, and the permission to query any state-action cell.  This experiment tests
error-correcting crystallization and invariant transfer, not autonomous ontology or
grammar discovery.
"""

from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


FAST_DEV_RUN = os.environ.get("NEURAL_QUOTIENT_V13_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("NEURAL_QUOTIENT_V13_SEED", "20260813"))
    n_states: int = 8
    n_actions: int = 3
    candidate_templates: int = 32
    recurring_templates: int = 4
    train_permutations: int = 30_000
    neural_calibration_permutations: int = 4_000
    prior_permutations: int = 2_000
    gate_permutations: int = 2_000
    prior_tasks_per_family: int = 6
    gate_tasks_per_family: int = 3
    test_tasks_per_family: int = 8
    initial_context: int = 8
    maximum_context: int = 8
    extraction_extra_queries: int = 8
    proposal_top_k: int = 4_096
    recurrence_threshold: int = 3
    neural_target_accuracy: float = 0.53
    neural_target_tolerance: float = 0.025
    neural_steps: int = 5_000
    neural_batch: int = 1_024
    neural_eval_examples: int = 8_192
    neural_eval_every: int = 25
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    width: int = 128
    heads: int = 4
    layers: int = 5
    dropout: float = 0.0
    gate_betas: Tuple[float, ...] = (0.0, 0.10, 0.25, 0.50, 1.0)
    report_budgets: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6, 8, 10, 12)
    novel_tasks: int = 24
    novel_changes: int = 4
    audit_period: int = 3
    output_dir: str = os.environ.get(
        "NEURAL_QUOTIENT_V13_OUTPUT_DIR",
        "outputs/05-neural-quotient-active-learner-v13/neural_quotient_active_v13_results"
        if Path("/content").exists()
        else "outputs/05-neural-quotient-active-learner-v13/neural_quotient_active_v13_results",
    )


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        candidate_templates=8,
        train_permutations=24_000,
        neural_calibration_permutations=3_000,
        prior_permutations=1_000,
        gate_permutations=1_000,
        prior_tasks_per_family=2,
        gate_tasks_per_family=1,
        test_tasks_per_family=2,
        proposal_top_k=1_024,
        neural_steps=1_000,
        neural_batch=256,
        neural_eval_examples=1_024,
        neural_eval_every=10,
        width=64,
        layers=3,
        novel_tasks=8,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
rng = np.random.default_rng(cfg.seed)
N, A = cfg.n_states, cfg.n_actions
CELLS = N * A
PERMUTATIONS = np.asarray(list(itertools.permutations(range(N))), dtype=np.int16)
INVERSES = np.argsort(PERMUTATIONS, axis=1).astype(np.int16)
N_PERMUTATIONS = len(PERMUTATIONS)


def strongly_connected(table: np.ndarray) -> bool:
    for source in range(N):
        seen = {source}
        frontier = [source]
        while frontier:
            state = frontier.pop()
            for successor in table[state]:
                successor = int(successor)
                if successor not in seen:
                    seen.add(successor)
                    frontier.append(successor)
        if len(seen) != N:
            return False
    return True


def all_relabelings(table: np.ndarray) -> np.ndarray:
    old_successors = table[INVERSES]
    rows = np.arange(N_PERMUTATIONS)[:, None, None]
    return PERMUTATIONS[rows, old_successors]


def canonicalize(table: np.ndarray) -> Tuple[np.ndarray, int]:
    relabeled = all_relabelings(np.asarray(table, dtype=np.int16))
    flat = relabeled.reshape(N_PERMUTATIONS, CELLS)
    first = int(np.lexsort(flat[:, ::-1].T)[0])
    canonical = relabeled[first].copy()
    automorphisms = int(np.sum(np.all(flat == canonical.reshape(1, -1), axis=1)))
    return canonical, automorphisms


def relabel_table(table: np.ndarray, permutation_index: int) -> np.ndarray:
    old_to_new = PERMUTATIONS[int(permutation_index)]
    new_to_old = INVERSES[int(permutation_index)]
    return old_to_new[table[new_to_old]]


def generate_machine(excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = rng.integers(0, N, size=(N, A), dtype=np.int16)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a distinct asymmetric machine")


def mutate_machine(base: np.ndarray, changes: int, excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = base.copy()
        for cell in rng.choice(CELLS, size=changes, replace=False):
            state, action = divmod(int(cell), A)
            old = int(candidate[state, action])
            draw = int(rng.integers(0, N - 1))
            candidate[state, action] = draw + (draw >= old)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a novel machine")


def hypothesis_predictions(library: Sequence[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    predictions: List[np.ndarray] = []
    family_ids: List[np.ndarray] = []
    row_index = np.arange(N_PERMUTATIONS)[:, None, None]
    action_index = np.arange(A)[None, None, :]
    for family, table in enumerate(library):
        canonical_next = table[PERMUTATIONS[:, :, None], action_index]
        observed_next = INVERSES[row_index, canonical_next]
        predictions.append(observed_next.reshape(N_PERMUTATIONS, CELLS).astype(np.int8))
        family_ids.append(np.full(N_PERMUTATIONS, family, dtype=np.int16))
    return np.concatenate(predictions), np.concatenate(family_ids)


class EquivariantTransitionGNN(nn.Module):
    """Permutation-equivariant neural predictor over a partially observed transition graph."""

    def __init__(self) -> None:
        super().__init__()
        width = cfg.width
        self.action = nn.Embedding(A, width)
        self.base_node = nn.Parameter(torch.zeros(width))
        self.query_role = nn.Embedding(2, width)
        self.to_source = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.to_target = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.updates = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(3 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(cfg.layers)])
        self.scorer = nn.Sequential(
            nn.Linear(4 * width + 1, 2 * width),
            nn.GELU(),
            nn.Linear(2 * width, width),
            nn.GELU(),
            nn.Linear(width, 1),
        )

    def forward(
        self,
        context_cells: torch.Tensor,
        context_outputs: torch.Tensor,
        context_mask: torch.Tensor,
        query_cells: torch.Tensor,
    ) -> torch.Tensor:
        batch, _ = context_cells.shape
        context_states = torch.div(context_cells, A, rounding_mode="floor")
        context_actions = context_cells % A
        query_states = torch.div(query_cells, A, rounding_mode="floor")
        query_actions = query_cells % A
        node_ids = torch.arange(N, device=context_cells.device)[None, :]
        roles = (node_ids == query_states[:, None]).long()
        nodes = self.base_node[None, None, :].expand(batch, N, -1) + self.query_role(roles)

        valid_batch, valid_slot = torch.nonzero(context_mask, as_tuple=True)
        source = context_states[valid_batch, valid_slot]
        target = context_outputs[valid_batch, valid_slot]
        action = context_actions[valid_batch, valid_slot]
        batch_offsets = valid_batch * N
        source_flat = batch_offsets + source
        target_flat = batch_offsets + target

        for layer in range(cfg.layers):
            flat = nodes.reshape(batch * N, cfg.width)
            action_features = self.action(action)
            source_messages = self.to_source[layer](
                torch.cat([flat[target_flat], action_features], dim=-1)
            ).float()
            target_messages = self.to_target[layer](
                torch.cat([flat[source_flat], action_features], dim=-1)
            ).float()
            aggregate_source = torch.zeros_like(flat)
            aggregate_target = torch.zeros_like(flat)
            aggregate_source.index_add_(0, source_flat, source_messages)
            aggregate_target.index_add_(0, target_flat, target_messages)
            degree_source = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            degree_target = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            ones = torch.ones((len(source_flat), 1), device=flat.device, dtype=flat.dtype)
            degree_source.index_add_(0, source_flat, ones)
            degree_target.index_add_(0, target_flat, ones)
            aggregate_source /= degree_source.clamp_min(1.0)
            aggregate_target /= degree_target.clamp_min(1.0)
            update = self.updates[layer](
                torch.cat([flat, aggregate_source, aggregate_target], dim=-1)
            )
            nodes = self.norms[layer](flat + update).reshape(batch, N, cfg.width)

        source_features = nodes[torch.arange(batch, device=nodes.device), query_states]
        candidate_features = nodes
        action_features = self.action(query_actions)
        expanded_source = source_features[:, None, :].expand(-1, N, -1)
        expanded_action = action_features[:, None, :].expand(-1, N, -1)
        equality = (node_ids == query_states[:, None]).to(nodes.dtype)[..., None]
        features = torch.cat(
            [
                expanded_source,
                candidate_features,
                expanded_source * candidate_features,
                expanded_action,
                equality,
            ],
            dim=-1,
        )
        return self.scorer(features).squeeze(-1)


def permutation_splits() -> Dict[str, np.ndarray]:
    order = rng.permutation(N_PERMUTATIONS)
    cursor = 0
    result: Dict[str, np.ndarray] = {}
    for name, length in (
        ("train", cfg.train_permutations),
        ("neural_calibration", cfg.neural_calibration_permutations),
        ("prior", cfg.prior_permutations),
        ("gate", cfg.gate_permutations),
    ):
        result[name] = order[cursor : cursor + length]
        cursor += length
    result["test"] = order[cursor:]
    if len(result["test"]) < cfg.test_tasks_per_family + cfg.novel_tasks:
        raise RuntimeError("Permutation split leaves too few untouched test mappings")
    return result


def sample_neural_batch(
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    batch_size: int,
    fixed_context: Optional[int] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    family = torch.randint(0, cfg.recurring_templates, (batch_size,), device=device)
    selected_pool = torch.randint(0, len(pool_gpu), (batch_size,), device=device)
    permutation = pool_gpu[selected_pool]
    tables = relabelings_gpu[family, permutation].reshape(batch_size, CELLS)
    cell_order = torch.rand((batch_size, CELLS), device=device).argsort(dim=1)
    if fixed_context is None:
        lengths = torch.randint(0, cfg.maximum_context + 1, (batch_size,), device=device)
    else:
        lengths = torch.full((batch_size,), fixed_context, device=device, dtype=torch.long)
    context_cells = cell_order[:, : cfg.maximum_context]
    context_outputs = torch.gather(tables, 1, context_cells)
    mask = torch.arange(cfg.maximum_context, device=device)[None, :] < lengths[:, None]
    query_cells = torch.gather(cell_order, 1, lengths[:, None]).squeeze(1)
    targets = torch.gather(tables, 1, query_cells[:, None]).squeeze(1)
    return context_cells, context_outputs, mask, query_cells, targets


@torch.no_grad()
def neural_accuracy(
    model: nn.Module,
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    examples: int,
) -> float:
    model.eval()
    correct = 0
    seen = 0
    while seen < examples:
        batch = min(1_024, examples - seen)
        values = sample_neural_batch(
            relabelings_gpu, pool_gpu, batch, fixed_context=cfg.initial_context
        )
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(*values[:-1])
        correct += int((logits.argmax(dim=-1) == values[-1]).sum().item())
        seen += batch
    return correct / examples


def train_neural(
    recurring_relabelings: np.ndarray, splits: Dict[str, np.ndarray]
) -> Tuple[EquivariantTransitionGNN, List[Dict[str, float]]]:
    model = EquivariantTransitionGNN().to(device)
    tables_gpu = torch.from_numpy(recurring_relabelings.astype(np.int64)).to(device)
    train_pool = torch.from_numpy(splits["train"].astype(np.int64)).to(device)
    calibration_pool = torch.from_numpy(splits["neural_calibration"].astype(np.int64)).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, fused=device.type == "cuda"
    )
    scaler_enabled = device.type == "cuda" and amp_dtype == torch.float16
    if hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler(device.type, enabled=scaler_enabled)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
    history: List[Dict[str, float]] = []
    best_distance = float("inf")
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_step = 0
    best_accuracy = 0.0
    for step in range(1, cfg.neural_steps + 1):
        model.train()
        batch = sample_neural_batch(
            tables_gpu, train_pool, cfg.neural_batch, fixed_context=cfg.initial_context
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            loss = F.cross_entropy(model(*batch[:-1]), batch[-1])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if step % cfg.neural_eval_every == 0 or step == 1:
            accuracy = neural_accuracy(
                model, tables_gpu, calibration_pool, cfg.neural_eval_examples
            )
            history.append({"step": step, "loss": float(loss.item()), "calibration_accuracy": accuracy})
            distance = abs(accuracy - cfg.neural_target_accuracy)
            if distance < best_distance:
                best_distance = distance
                best_step = step
                best_accuracy = accuracy
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            print(
                f"neural step={step:4d} | loss={loss.item():.4f} | "
                f"heldout-k{cfg.initial_context} accuracy={100*accuracy:5.1f}%"
            )
            if (
                step >= 4 * cfg.neural_eval_every
                and distance <= cfg.neural_target_tolerance
                and accuracy >= cfg.neural_target_accuracy
            ):
                break
    if best_state is None:
        raise RuntimeError("No neural checkpoint was evaluated")
    model.load_state_dict(best_state)
    model.eval()
    print(f"restored imperfect-neural checkpoint step={best_step} | accuracy={100*best_accuracy:.2f}%")
    return model, history


@torch.no_grad()
def neural_full_probabilities(
    model: nn.Module, context_cells: Sequence[int], context_outputs: Sequence[int]
) -> np.ndarray:
    cells = list(context_cells)[-cfg.maximum_context :]
    outputs = list(context_outputs)[-cfg.maximum_context :]
    width = cfg.maximum_context
    padded_cells = np.zeros((CELLS, width), dtype=np.int64)
    padded_outputs = np.zeros((CELLS, width), dtype=np.int64)
    mask = np.zeros((CELLS, width), dtype=bool)
    if cells:
        padded_cells[:, : len(cells)] = np.asarray(cells)[None, :]
        padded_outputs[:, : len(outputs)] = np.asarray(outputs)[None, :]
        mask[:, : len(cells)] = True
    queries = np.arange(CELLS, dtype=np.int64)
    tensors = [
        torch.from_numpy(padded_cells).to(device),
        torch.from_numpy(padded_outputs).to(device),
        torch.from_numpy(mask).to(device),
        torch.from_numpy(queries).to(device),
    ]
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
        logits = model(*tensors)
    return logits.float().softmax(dim=-1).cpu().numpy()


def entropy_from_probabilities(probability: np.ndarray) -> float:
    positive = probability[probability > 0]
    return float(-np.sum(positive * np.log2(positive)))


def choose_weighted_query(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    unqueried: np.ndarray,
    generator: np.random.Generator,
    query_number: int,
    dual: bool = False,
) -> int:
    available = np.flatnonzero(unqueried)
    if dual and query_number > 0 and query_number % cfg.audit_period == 0:
        return int(generator.choice(available))
    entropies = []
    for cell in available:
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        entropies.append(entropy_from_probabilities(probability))
    entropies = np.asarray(entropies)
    best = available[np.isclose(entropies, entropies.max(), atol=1e-12)]
    return int(generator.choice(best))


def proposal_decode(
    truth_table: np.ndarray,
    neural_table: np.ndarray,
    grammar_predictions: np.ndarray,
    grammar_families: np.ndarray,
    initial_cells: np.ndarray,
    seed: int,
) -> Dict[str, object]:
    truth = truth_table.reshape(-1).astype(np.int16)
    distance = np.sum(grammar_predictions != neural_table[None, :], axis=1)
    top_k = min(cfg.proposal_top_k, len(distance))
    proposal = np.argpartition(distance, top_k - 1)[:top_k].astype(np.int32)
    truth_in_proposal = bool(np.any(np.all(grammar_predictions[proposal] == truth[None, :], axis=1)))
    observed_cells = [int(cell) for cell in initial_cells]
    observed_outputs = [int(truth[cell]) for cell in observed_cells]
    candidates = proposal
    for cell, output in zip(observed_cells, observed_outputs):
        candidates = candidates[grammar_predictions[candidates, cell] == output]
    unqueried = np.ones(CELLS, dtype=bool)
    unqueried[initial_cells] = False
    generator = np.random.default_rng(seed)
    extra = 0
    rejected = len(candidates) == 0
    while extra < cfg.extraction_extra_queries and len(candidates) > 0:
        unique_tables = np.unique(grammar_predictions[candidates], axis=0)
        if len(unique_tables) == 1:
            break
        weights = np.ones(len(candidates), dtype=np.float64) / len(candidates)
        cell = choose_weighted_query(
            candidates,
            grammar_predictions,
            weights,
            unqueried,
            generator,
            extra,
            dual=True,
        )
        unqueried[cell] = False
        observed_cells.append(cell)
        observed_outputs.append(int(truth[cell]))
        candidates = candidates[grammar_predictions[candidates, cell] == truth[cell]]
        extra += 1
        rejected = len(candidates) == 0
    if len(candidates):
        best = int(candidates[np.argmin(distance[candidates])])
        selected_table = grammar_predictions[best]
        selected_family = int(grammar_families[best])
        resolved = bool(np.all(grammar_predictions[candidates] == selected_table[None, :]))
        exact = bool(np.array_equal(selected_table, truth))
    else:
        selected_family = -1
        resolved = False
        exact = False
    return {
        "neural_accuracy": float(np.mean(neural_table == truth)),
        "truth_in_proposal": truth_in_proposal,
        "extra_queries": extra,
        "total_observations": len(initial_cells) + extra,
        "remaining_candidates": len(candidates),
        "resolved": resolved,
        "exact": exact,
        "selected_family": selected_family,
        "rejected": rejected,
    }


def posterior_metrics(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    truth: np.ndarray,
) -> Dict[str, float]:
    if len(candidates) == 0:
        return {
            "transition_accuracy": float("nan"),
            "posterior_nll": float("nan"),
            "resolved_fraction": float("nan"),
            "exact_table": 0.0,
            "candidates": 0.0,
        }
    top_correct = 0
    resolved = 0
    correct_probability = []
    for cell in range(CELLS):
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        top_correct += int(np.argmax(probability) == truth[cell])
        resolved += int(np.count_nonzero(probability > 1e-12) == 1)
        correct_probability.append(probability[int(truth[cell])])
    return {
        "transition_accuracy": top_correct / CELLS,
        "posterior_nll": float(-np.mean(np.log(np.clip(correct_probability, 1e-15, 1.0)))),
        "resolved_fraction": resolved / CELLS,
        "exact_table": float(resolved == CELLS),
        "candidates": float(len(candidates)),
    }


def neural_weights(
    candidates: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    beta: float,
) -> np.ndarray:
    if beta == 0.0:
        return np.ones(len(candidates), dtype=np.float64) / len(candidates)
    log_probability = np.log(np.clip(probabilities, 1e-9, 1.0))
    scores = log_probability[np.arange(CELLS)[None, :], predictions[candidates]].sum(axis=1)
    scores = beta * (scores - scores.max())
    weights = np.exp(np.clip(scores, -80.0, 0.0))
    return weights / weights.sum()


def transfer_trace(
    truth_table: np.ndarray,
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    beta: float,
    seed: int,
    maximum_queries: int = 12,
    dual: bool = False,
) -> Tuple[List[Dict[str, float]], Optional[int]]:
    truth = truth_table.reshape(-1).astype(np.int16)
    candidates = np.arange(len(predictions), dtype=np.int32)
    unqueried = np.ones(CELLS, dtype=bool)
    context_cells: List[int] = []
    context_outputs: List[int] = []
    generator = np.random.default_rng(seed)
    rows: List[Dict[str, float]] = []
    rejected_at: Optional[int] = None
    for query_count in range(maximum_queries + 1):
        neural_probability = neural_full_probabilities(model, context_cells, context_outputs)
        if len(candidates):
            weights = neural_weights(candidates, predictions, neural_probability, beta)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        else:
            weights = np.empty(0)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        neural_prediction = neural_probability.argmax(axis=1)
        rows.append(
            {
                "queries": float(query_count),
                **symbolic,
                "neural_transition_accuracy": float(np.mean(neural_prediction == truth)),
                "neural_exact_table": float(np.array_equal(neural_prediction, truth)),
            }
        )
        if query_count == maximum_queries or len(candidates) == 0:
            break
        cell = choose_weighted_query(
            candidates,
            predictions,
            weights,
            unqueried,
            generator,
            query_count,
            dual=dual,
        )
        unqueried[cell] = False
        context_cells.append(cell)
        context_outputs.append(int(truth[cell]))
        candidates = candidates[predictions[candidates, cell] == truth[cell]]
        if len(candidates) == 0:
            rejected_at = query_count + 1
    return rows, rejected_at


def first_exact(trace: Sequence[Dict[str, float]]) -> int:
    hits = [int(row["queries"]) for row in trace if row["exact_table"] == 1.0]
    return min(hits) if hits else CELLS + 1


def metric_at(trace: Sequence[Dict[str, float]], budget: int, key: str) -> float:
    eligible = [row for row in trace if int(row["queries"]) <= budget]
    return float(eligible[-1][key])


def evaluate_beta(
    beta: float,
    tasks: Sequence[np.ndarray],
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    seed_offset: int,
) -> Dict[str, float]:
    traces = [
        transfer_trace(task, predictions, model, beta, cfg.seed + seed_offset + index)[0]
        for index, task in enumerate(tasks)
    ]
    return {
        "beta": beta,
        "exact_at_8": float(np.mean([metric_at(trace, 8, "exact_table") for trace in traces])),
        "accuracy_at_4": float(np.mean([metric_at(trace, 4, "transition_accuracy") for trace in traces])),
        "median_exact_queries": float(np.median([first_exact(trace) for trace in traces])),
    }


def write_csv(path: Path, rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


print("=" * 118)
print("NEURAL-QUOTIENT ACTIVE LEARNER V13 — decode, quotient, transfer, reorganize")
print("=" * 118)
print(
    f"device={device} | GPU={torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'} | "
    f"states={N} | actions={A} | cells={CELLS} | permutations={N_PERMUTATIONS:,} | fast={FAST_DEV_RUN}"
)
started = time.perf_counter()

excluded: set[bytes] = set()
grammar = [generate_machine(excluded) for _ in range(cfg.candidate_templates)]
recurring = grammar[: cfg.recurring_templates]
grammar_predictions, grammar_families = hypothesis_predictions(grammar)
recurring_relabelings = np.stack([all_relabelings(table) for table in recurring]).astype(np.int8)
splits = permutation_splits()
print(
    f"candidate grammar={len(grammar)} canonical laws | hypotheses={len(grammar_predictions):,} | "
    f"recurring latent laws={len(recurring)}"
)

model, training_history = train_neural(recurring_relabelings, splits)

# Crystallize recurring rules from imperfect neural tables plus actively requested evidence.
prior_rows: List[Dict[str, object]] = []
shuffled_rows: List[Dict[str, object]] = []
naive_keys: set[bytes] = set()
decoded_family_counts: Dict[int, int] = {}
prior_pool = splits["prior"]
prior_cursor = 0
for family in range(cfg.recurring_templates):
    for task_in_family in range(cfg.prior_tasks_per_family):
        permutation_index = int(prior_pool[prior_cursor % len(prior_pool)])
        prior_cursor += 1
        truth = relabel_table(recurring[family], permutation_index)
        flat_truth = truth.reshape(-1)
        cell_order = np.random.default_rng(cfg.seed + 10_000 + prior_cursor).permutation(CELLS)
        initial_cells = cell_order[: cfg.initial_context]
        initial_outputs = flat_truth[initial_cells]
        probability = neural_full_probabilities(model, initial_cells, initial_outputs)
        neural_table = probability.argmax(axis=1).astype(np.int16)
        naive_canonical, _ = canonicalize(neural_table.reshape(N, A))
        naive_keys.add(naive_canonical.astype(np.int8).tobytes())
        decoded = proposal_decode(
            truth,
            neural_table,
            grammar_predictions,
            grammar_families,
            initial_cells,
            cfg.seed + 20_000 + prior_cursor,
        )
        prior_rows.append({"family": family, "task": task_in_family, **decoded})
        if decoded["resolved"] and not decoded["rejected"]:
            selected = int(decoded["selected_family"])
            decoded_family_counts[selected] = decoded_family_counts.get(selected, 0) + 1

        shuffled = neural_table[np.random.default_rng(cfg.seed + 30_000 + prior_cursor).permutation(CELLS)]
        shuffled_decoded = proposal_decode(
            truth,
            shuffled,
            grammar_predictions,
            grammar_families,
            initial_cells,
            cfg.seed + 40_000 + prior_cursor,
        )
        shuffled_rows.append({"family": family, "task": task_in_family, **shuffled_decoded})

decoded_ids = sorted(
    family for family, count in decoded_family_counts.items() if count >= cfg.recurrence_threshold
)
decoded_library = [grammar[family] for family in decoded_ids]
clean_library = list(recurring)
expected_keys = {table.astype(np.int8).tobytes() for table in recurring}
decoded_keys = {table.astype(np.int8).tobytes() for table in decoded_library}
library_recovery = len(expected_keys & decoded_keys) / len(expected_keys)
prior_neural_accuracy = float(np.mean([row["neural_accuracy"] for row in prior_rows]))
prior_exact = float(np.mean([row["exact"] for row in prior_rows]))
prior_resolved = float(np.mean([row["resolved"] for row in prior_rows]))
prior_median_observations = float(
    np.median([row["total_observations"] for row in prior_rows if row["exact"]])
)
shuffled_exact = float(np.mean([row["exact"] for row in shuffled_rows]))
print("\nCRYSTALLIZATION FROM THE IMPERFECT NEURAL CHANNEL")
print("-" * 118)
print(
    f"neural table accuracy={100*prior_neural_accuracy:.2f}% | active exact={100*prior_exact:.1f}% | "
    f"resolved={100*prior_resolved:.1f}% | median total observations={prior_median_observations:.1f}/{CELLS}"
)
print(
    f"naive noisy canonical templates={len(naive_keys)} | recurring decoded templates={len(decoded_library)} | "
    f"true-template recovery={100*library_recovery:.1f}% | shuffled-control exact={100*shuffled_exact:.1f}%"
)
if not decoded_library:
    raise RuntimeError("No recurring rules survived active crystallization")

decoded_predictions, _ = hypothesis_predictions(decoded_library)
clean_predictions, _ = hypothesis_predictions(clean_library)

# Calibrate neural responsibility on mappings disjoint from training, crystallization, and test.
gate_tasks: List[np.ndarray] = []
gate_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.gate_tasks_per_family):
        gate_tasks.append(relabel_table(recurring[family], int(splits["gate"][gate_cursor])))
        gate_cursor += 1
gate_rows = [
    evaluate_beta(beta, gate_tasks, decoded_predictions, model, 100_000 + int(10_000 * beta))
    for beta in cfg.gate_betas
]
symbolic_gate = next(row for row in gate_rows if row["beta"] == 0.0)
safe_gate_rows = [
    row
    for row in gate_rows
    if row["exact_at_8"] + 1e-12 >= symbolic_gate["exact_at_8"]
    and row["accuracy_at_4"] + 1e-12 >= symbolic_gate["accuracy_at_4"]
]
selected_gate = max(
    safe_gate_rows,
    key=lambda row: (row["exact_at_8"], -row["median_exact_queries"], row["accuracy_at_4"], -row["beta"]),
)
selected_beta = float(selected_gate["beta"])
print("\nHELD-OUT GATE CALIBRATION")
print("-" * 118)
for row in gate_rows:
    print(
        f"beta={row['beta']:.2f} | exact@8={100*row['exact_at_8']:5.1f}% | "
        f"accuracy@4={100*row['accuracy_at_4']:5.1f}% | median exact={row['median_exact_queries']:.1f}"
    )
print(f"selected no-harm beta={selected_beta:.2f}")

# Untouched permutation transfer.
test_tasks: List[Tuple[int, np.ndarray]] = []
test_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.test_tasks_per_family):
        test_tasks.append((family, relabel_table(recurring[family], int(splits["test"][test_cursor]))))
        test_cursor += 1

transfer_rows: List[Dict[str, object]] = []
exact_queries: Dict[str, List[int]] = {"symbolic": [], "calibrated-hybrid": [], "clean-oracle": []}
for task_id, (family, truth) in enumerate(test_tasks):
    for method, predictions, beta in (
        ("symbolic", decoded_predictions, 0.0),
        ("calibrated-hybrid", decoded_predictions, selected_beta),
        ("clean-oracle", clean_predictions, 0.0),
    ):
        trace, rejected = transfer_trace(
            truth,
            predictions,
            model,
            beta,
            cfg.seed + 200_000 + 101 * task_id + {"symbolic": 1, "calibrated-hybrid": 2, "clean-oracle": 3}[method],
        )
        if rejected is not None:
            raise RuntimeError(f"Known-family task rejected by {method}")
        exact_queries[method].append(first_exact(trace))
        for row in trace:
            if int(row["queries"]) in cfg.report_budgets:
                transfer_rows.append(
                    {"method": method, "task": task_id, "family": family, **row}
                )


def transfer_summary(method: str, budget: int, key: str) -> float:
    rows = [
        row
        for row in transfer_rows
        if row["method"] == method and int(row["queries"]) == budget
    ]
    return float(np.nanmean([float(row[key]) for row in rows]))


print("\nUNTOUCHED-REPRESENTATION TRANSFER")
print("-" * 118)
for method in exact_queries:
    print(
        f"{method:18s} | median exact queries={np.median(exact_queries[method]):4.1f} | "
        f"exact@8={100*transfer_summary(method, 8, 'exact_table'):5.1f}% | "
        f"table accuracy@4={100*transfer_summary(method, 4, 'transition_accuracy'):5.1f}%"
    )
neural_accuracy_at_8 = transfer_summary("symbolic", 8, "neural_transition_accuracy")
neural_exact_at_8 = transfer_summary("symbolic", 8, "neural_exact_table")
print(
    f"direct imperfect neural @8 observations | table accuracy={100*neural_accuracy_at_8:.1f}% | "
    f"exact tables={100*neural_exact_at_8:.1f}% | local tabular exact@8=0.0%"
)

# Novel-law rejection, MDL promotion, and future reuse.
novel = mutate_machine(recurring[0], cfg.novel_changes, excluded)
novel_tables = [
    relabel_table(novel, int(splits["test"][(test_cursor + index) % len(splits["test"])]))
    for index in range(cfg.novel_tasks)
]
detection_queries: List[int] = []
for task_id, truth in enumerate(novel_tables):
    _, rejected = transfer_trace(
        truth,
        decoded_predictions,
        model,
        0.0,
        cfg.seed + 400_000 + task_id,
        maximum_queries=12,
        dual=True,
    )
    detection_queries.append(CELLS + 1 if rejected is None else rejected)

nearest_mismatches = int(
    min(np.min(np.sum(predictions != novel_tables[0].reshape(1, -1), axis=1)) for predictions in [decoded_predictions])
)
permutation_bits = math.log2(math.factorial(N))
existing_task_bits = (
    math.log2(len(decoded_library))
    + permutation_bits
    + math.log2(math.comb(CELLS, nearest_mismatches))
    + nearest_mismatches * math.log2(N - 1)
)
definition_bits = CELLS * math.log2(N)
new_task_bits = math.log2(len(decoded_library) + 1) + permutation_bits
promotion_at: Optional[int] = None
promotion_rows: List[Dict[str, object]] = []
for repeats in range(1, cfg.novel_tasks + 1):
    exceptions_bits = repeats * existing_task_bits
    promotion_bits = definition_bits + repeats * new_task_bits
    if promotion_at is None and promotion_bits < exceptions_bits:
        promotion_at = repeats
    promotion_rows.append(
        {
            "repeated_worlds": repeats,
            "exceptions_bits": exceptions_bits,
            "promotion_bits": promotion_bits,
        }
    )

expanded_predictions, _ = hypothesis_predictions(decoded_library + [novel])
future_exact = []
for task_id, truth in enumerate(novel_tables):
    trace, rejected = transfer_trace(
        truth,
        expanded_predictions,
        model,
        0.0,
        cfg.seed + 500_000 + task_id,
    )
    if rejected is not None:
        raise RuntimeError("Promoted family was rejected")
    future_exact.append(first_exact(trace))
detected_by_12 = float(np.mean(np.asarray(detection_queries) <= 12))
future_median = float(np.median(future_exact))
print("\nNOVEL LAW AND REORGANIZATION")
print("-" * 118)
print(
    f"detected by 12={100*detected_by_12:.1f}% | median detection={np.median(detection_queries):.1f} | "
    f"nearest residual={nearest_mismatches} transitions | promote after={promotion_at} recurring worlds | "
    f"future median exact={future_median:.1f} queries"
)

predictions_declared = {
    "P1_actual_neural_channel_is_imperfect_45_to_60pct": 0.45 <= prior_neural_accuracy <= 0.60,
    "P2_active_structural_decoder_recovers_90pct_with_fewer_than_24_observations": (
        prior_exact >= 0.90 and prior_median_observations <= 16
    ),
    "P3_noisy_active_crystallization_recovers_all_recurring_templates": library_recovery == 1.0,
    "P4_structure_not_cell_order_drives_decoding": prior_exact - shuffled_exact >= 0.50,
    "P5_unseen_representation_exact_at_8_at_least_90pct": transfer_summary(
        "symbolic", 8, "exact_table"
    )
    >= 0.90,
    "P6_calibrated_gate_no_harm_on_untouched_test": (
        transfer_summary("calibrated-hybrid", 8, "exact_table") + 1e-12
        >= transfer_summary("symbolic", 8, "exact_table")
    ),
    "P7_symbolic_transfer_beats_direct_neural_and_local_at_8": (
        transfer_summary("symbolic", 8, "exact_table") > neural_exact_at_8
        and transfer_summary("symbolic", 8, "exact_table") > 0.0
    ),
    "P8_novel_law_is_rejected_then_reusable_after_mdl_promotion": (
        detected_by_12 >= 0.90 and promotion_at is not None and promotion_at <= 8 and future_median < CELLS / 2
    ),
}
print("\nPREDECLARED PREDICTIONS")
print("-" * 118)
for name, passed in predictions_declared.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(out_dir / "training_history.csv", training_history)
write_csv(out_dir / "prior_crystallization.csv", prior_rows)
write_csv(out_dir / "shuffled_control.csv", shuffled_rows)
write_csv(out_dir / "gate_calibration.csv", gate_rows)
write_csv(out_dir / "transfer_trials.csv", transfer_rows)
write_csv(out_dir / "promotion.csv", promotion_rows)

result = {
    "config": asdict(cfg),
    "predictions": predictions_declared,
    "neural": {
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "prior_table_accuracy": prior_neural_accuracy,
    },
    "crystallization": {
        "prior_exact": prior_exact,
        "prior_resolved": prior_resolved,
        "median_total_observations": prior_median_observations,
        "naive_noisy_templates": len(naive_keys),
        "decoded_templates": len(decoded_library),
        "template_recovery": library_recovery,
        "shuffled_exact": shuffled_exact,
    },
    "gate": {"selected_beta": selected_beta, "calibration": gate_rows},
    "transfer": {
        method: {
            "median_exact_queries": float(np.median(values)),
            "exact_at_8": transfer_summary(method, 8, "exact_table"),
            "accuracy_at_4": transfer_summary(method, 4, "transition_accuracy"),
        }
        for method, values in exact_queries.items()
    },
    "direct_neural": {
        "accuracy_at_8": neural_accuracy_at_8,
        "exact_at_8": neural_exact_at_8,
    },
    "novelty": {
        "detected_by_12": detected_by_12,
        "median_detection_queries": float(np.median(detection_queries)),
        "promotion_at": promotion_at,
        "future_median_exact_queries": future_median,
    },
    "elapsed_seconds": time.perf_counter() - started,
}
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
(out_dir / "results.json").write_text(json.dumps(result, indent=2))
torch.save(model.state_dict(), out_dir / "imperfect_neural.pt")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
ax.plot(
    [row["step"] for row in training_history],
    [100 * row["calibration_accuracy"] for row in training_history],
    marker="o",
)
ax.axhline(100 * cfg.neural_target_accuracy, color="black", linestyle="--", label="target imperfect channel")
ax.set_title("Actual neural learner checkpoint")
ax.set_xlabel("optimizer step")
ax.set_ylabel("held-out transition accuracy (%)")
ax.grid(alpha=0.25)
ax.legend()

ax = axes[0, 1]
labels = ["neural table", "shuffled control", "active decode"]
values = [prior_neural_accuracy, shuffled_exact, prior_exact]
ax.bar(labels, [100 * value for value in values], color=["#4C78A8", "#E45756", "#54A24B"])
ax.set_ylabel("accuracy / exact recovery (%)")
ax.set_title("Noisy codeword becomes an exact rule")
ax.set_ylim(0, 105)

ax = axes[1, 0]
for method, color in (("symbolic", "#54A24B"), ("calibrated-hybrid", "#B279A2"), ("clean-oracle", "#F2CF5B")):
    budgets = []
    exact = []
    for budget in cfg.report_budgets:
        rows = [row for row in transfer_rows if row["method"] == method and int(row["queries"]) == budget]
        if rows:
            budgets.append(budget)
            exact.append(100 * np.mean([row["exact_table"] for row in rows]))
    ax.plot(budgets, exact, marker="o", label=method, color=color)
ax.axvline(8, color="black", linestyle=":")
ax.set_title("Exact rule on unseen state names")
ax.set_xlabel("actively observed transitions")
ax.set_ylabel("exact full table (%)")
ax.set_ylim(-2, 102)
ax.grid(alpha=0.25)
ax.legend()

ax = axes[1, 1]
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["exceptions_bits"] for row in promotion_rows],
    marker="o",
    label="keep as residuals",
    color="#E45756",
)
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["promotion_bits"] for row in promotion_rows],
    marker="o",
    label="promote rule",
    color="#54A24B",
)
if promotion_at is not None:
    ax.axvline(promotion_at, color="black", linestyle="--", label=f"promotion at {promotion_at}")
ax.set_title("Residual recurrence triggers reorganization")
ax.set_xlabel("repeated novel worlds")
ax.set_ylabel("description length (bits)")
ax.grid(alpha=0.25)
ax.legend()

fig.suptitle(
    "Neural-to-symbolic active learning: predict noisily, decode exactly, transfer invariantly",
    fontsize=15,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

shutil.make_archive(str(out_dir / "run_bundle"), "zip", root_dir=out_dir)
print("\n" + "=" * 118)
print("FINAL V13 SUMMARY")
print("=" * 118)
print(
    f"neural={100*prior_neural_accuracy:.1f}% local transitions -> active crystallization={100*prior_exact:.1f}% exact | "
    f"library recovery={100*library_recovery:.1f}%"
)
print(
    f"unseen representation exact@8={100*transfer_summary('symbolic', 8, 'exact_table'):.1f}% | "
    f"median exact={np.median(exact_queries['symbolic']):.1f}/24 queries | novel detected@12={100*detected_by_12:.1f}%"
)
print(f"elapsed={result['elapsed_seconds']:.1f}s | artifacts={out_dir} | bundle={out_dir / 'run_bundle.zip'}")


## Experiment 6: Neural-quotient active learner V13.1

Original cell `5`, split part `1`.

**Cleanup note.** Two complete programs were concatenated in the source cell; they are separated here so each has valid future imports and independent artifacts.


In [ ]:
"""
NEURAL-QUOTIENT ACTIVE LEARNER V13.1 — SINGLE-CELL A100 EXPERIMENT

Breakthrough test
-----------------
Treat an imperfect neural transition table as a noisy codeword on a structured
program manifold.  Decode that codeword against a broad finite-machine grammar,
ask the environment only the maximally informative missing transitions, quotient
the verified rule by arbitrary state renaming, and retain recurring rules by MDL.

The decisive transfer test uses coordinate permutations never seen by the neural
learner.  The system must recover a complete 24-transition world from substantially
fewer than 24 observations, reject a genuinely novel law, and reorganize only when
the same residual law recurs.

What is learned and what is supplied
------------------------------------
Learned: a real in-context Transformer transition predictor; recurring canonical
templates; a held-out calibrated neural/symbolic responsibility gate.

Supplied: eight states, three actions, a finite grammar of candidate deterministic
machines, and the permission to query any state-action cell.  This experiment tests
error-correcting crystallization and invariant transfer, not autonomous ontology or
grammar discovery.
"""

from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


FAST_DEV_RUN = os.environ.get("NEURAL_QUOTIENT_V13_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("NEURAL_QUOTIENT_V13_SEED", "20260813"))
    n_states: int = 8
    n_actions: int = 3
    candidate_templates: int = 32
    recurring_templates: int = 4
    train_permutations: int = 30_000
    neural_calibration_permutations: int = 4_000
    prior_permutations: int = 2_000
    gate_permutations: int = 2_000
    prior_tasks_per_family: int = 6
    gate_tasks_per_family: int = 3
    test_tasks_per_family: int = 8
    initial_context: int = 8
    maximum_context: int = 8
    extraction_extra_queries: int = 8
    proposal_top_k: int = 16_384
    proposal_sweep: Tuple[int, ...] = (1_024, 4_096, 16_384, 65_536)
    recurrence_threshold: int = 3
    neural_target_accuracy: float = 0.53
    neural_target_tolerance: float = 0.025
    neural_steps: int = 5_000
    neural_batch: int = 1_024
    neural_eval_examples: int = 8_192
    neural_eval_every: int = 25
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    width: int = 128
    heads: int = 4
    layers: int = 5
    dropout: float = 0.0
    gate_betas: Tuple[float, ...] = (0.0, 0.10, 0.25, 0.50, 1.0)
    report_budgets: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6, 8, 10, 12)
    novel_tasks: int = 24
    novel_changes: int = 4
    audit_period: int = 3
    output_dir: str = os.environ.get(
        "NEURAL_QUOTIENT_V13_OUTPUT_DIR",
        "outputs/06-neural-quotient-active-learner-v13-1/neural_quotient_active_v13_1_results"
        if Path("/content").exists()
        else "outputs/06-neural-quotient-active-learner-v13-1/neural_quotient_active_v13_1_results",
    )


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        candidate_templates=8,
        train_permutations=24_000,
        neural_calibration_permutations=3_000,
        prior_permutations=1_000,
        gate_permutations=1_000,
        prior_tasks_per_family=2,
        gate_tasks_per_family=1,
        test_tasks_per_family=2,
        proposal_top_k=4_096,
        proposal_sweep=(1_024, 4_096),
        neural_steps=1_000,
        neural_batch=256,
        neural_eval_examples=1_024,
        neural_eval_every=10,
        width=64,
        layers=3,
        novel_tasks=8,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
rng = np.random.default_rng(cfg.seed)
N, A = cfg.n_states, cfg.n_actions
CELLS = N * A
PERMUTATIONS = np.asarray(list(itertools.permutations(range(N))), dtype=np.int16)
INVERSES = np.argsort(PERMUTATIONS, axis=1).astype(np.int16)
N_PERMUTATIONS = len(PERMUTATIONS)


def strongly_connected(table: np.ndarray) -> bool:
    for source in range(N):
        seen = {source}
        frontier = [source]
        while frontier:
            state = frontier.pop()
            for successor in table[state]:
                successor = int(successor)
                if successor not in seen:
                    seen.add(successor)
                    frontier.append(successor)
        if len(seen) != N:
            return False
    return True


def all_relabelings(table: np.ndarray) -> np.ndarray:
    old_successors = table[INVERSES]
    rows = np.arange(N_PERMUTATIONS)[:, None, None]
    return PERMUTATIONS[rows, old_successors]


def canonicalize(table: np.ndarray) -> Tuple[np.ndarray, int]:
    relabeled = all_relabelings(np.asarray(table, dtype=np.int16))
    flat = relabeled.reshape(N_PERMUTATIONS, CELLS)
    first = int(np.lexsort(flat[:, ::-1].T)[0])
    canonical = relabeled[first].copy()
    automorphisms = int(np.sum(np.all(flat == canonical.reshape(1, -1), axis=1)))
    return canonical, automorphisms


def relabel_table(table: np.ndarray, permutation_index: int) -> np.ndarray:
    old_to_new = PERMUTATIONS[int(permutation_index)]
    new_to_old = INVERSES[int(permutation_index)]
    return old_to_new[table[new_to_old]]


def generate_machine(excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = rng.integers(0, N, size=(N, A), dtype=np.int16)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a distinct asymmetric machine")


def mutate_machine(base: np.ndarray, changes: int, excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = base.copy()
        for cell in rng.choice(CELLS, size=changes, replace=False):
            state, action = divmod(int(cell), A)
            old = int(candidate[state, action])
            draw = int(rng.integers(0, N - 1))
            candidate[state, action] = draw + (draw >= old)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a novel machine")


def hypothesis_predictions(library: Sequence[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    predictions: List[np.ndarray] = []
    family_ids: List[np.ndarray] = []
    row_index = np.arange(N_PERMUTATIONS)[:, None, None]
    action_index = np.arange(A)[None, None, :]
    for family, table in enumerate(library):
        canonical_next = table[PERMUTATIONS[:, :, None], action_index]
        observed_next = INVERSES[row_index, canonical_next]
        predictions.append(observed_next.reshape(N_PERMUTATIONS, CELLS).astype(np.int8))
        family_ids.append(np.full(N_PERMUTATIONS, family, dtype=np.int16))
    return np.concatenate(predictions), np.concatenate(family_ids)


class EquivariantTransitionGNN(nn.Module):
    """Permutation-equivariant neural predictor over a partially observed transition graph."""

    def __init__(self) -> None:
        super().__init__()
        width = cfg.width
        self.action = nn.Embedding(A, width)
        self.base_node = nn.Parameter(torch.zeros(width))
        self.query_role = nn.Embedding(2, width)
        self.to_source = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.to_target = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.updates = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(3 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(cfg.layers)])
        self.scorer = nn.Sequential(
            nn.Linear(4 * width + 1, 2 * width),
            nn.GELU(),
            nn.Linear(2 * width, width),
            nn.GELU(),
            nn.Linear(width, 1),
        )

    def forward(
        self,
        context_cells: torch.Tensor,
        context_outputs: torch.Tensor,
        context_mask: torch.Tensor,
        query_cells: torch.Tensor,
    ) -> torch.Tensor:
        batch, _ = context_cells.shape
        context_states = torch.div(context_cells, A, rounding_mode="floor")
        context_actions = context_cells % A
        query_states = torch.div(query_cells, A, rounding_mode="floor")
        query_actions = query_cells % A
        node_ids = torch.arange(N, device=context_cells.device)[None, :]
        roles = (node_ids == query_states[:, None]).long()
        nodes = self.base_node[None, None, :].expand(batch, N, -1) + self.query_role(roles)

        valid_batch, valid_slot = torch.nonzero(context_mask, as_tuple=True)
        source = context_states[valid_batch, valid_slot]
        target = context_outputs[valid_batch, valid_slot]
        action = context_actions[valid_batch, valid_slot]
        batch_offsets = valid_batch * N
        source_flat = batch_offsets + source
        target_flat = batch_offsets + target

        for layer in range(cfg.layers):
            flat = nodes.reshape(batch * N, cfg.width)
            action_features = self.action(action)
            source_messages = self.to_source[layer](
                torch.cat([flat[target_flat], action_features], dim=-1)
            ).float()
            target_messages = self.to_target[layer](
                torch.cat([flat[source_flat], action_features], dim=-1)
            ).float()
            aggregate_source = torch.zeros_like(flat)
            aggregate_target = torch.zeros_like(flat)
            aggregate_source.index_add_(0, source_flat, source_messages)
            aggregate_target.index_add_(0, target_flat, target_messages)
            degree_source = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            degree_target = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            ones = torch.ones((len(source_flat), 1), device=flat.device, dtype=flat.dtype)
            degree_source.index_add_(0, source_flat, ones)
            degree_target.index_add_(0, target_flat, ones)
            aggregate_source /= degree_source.clamp_min(1.0)
            aggregate_target /= degree_target.clamp_min(1.0)
            update = self.updates[layer](
                torch.cat([flat, aggregate_source, aggregate_target], dim=-1)
            )
            nodes = self.norms[layer](flat + update).reshape(batch, N, cfg.width)

        source_features = nodes[torch.arange(batch, device=nodes.device), query_states]
        candidate_features = nodes
        action_features = self.action(query_actions)
        expanded_source = source_features[:, None, :].expand(-1, N, -1)
        expanded_action = action_features[:, None, :].expand(-1, N, -1)
        equality = (node_ids == query_states[:, None]).to(nodes.dtype)[..., None]
        features = torch.cat(
            [
                expanded_source,
                candidate_features,
                expanded_source * candidate_features,
                expanded_action,
                equality,
            ],
            dim=-1,
        )
        return self.scorer(features).squeeze(-1)


def permutation_splits() -> Dict[str, np.ndarray]:
    order = rng.permutation(N_PERMUTATIONS)
    cursor = 0
    result: Dict[str, np.ndarray] = {}
    for name, length in (
        ("train", cfg.train_permutations),
        ("neural_calibration", cfg.neural_calibration_permutations),
        ("prior", cfg.prior_permutations),
        ("gate", cfg.gate_permutations),
    ):
        result[name] = order[cursor : cursor + length]
        cursor += length
    result["test"] = order[cursor:]
    if len(result["test"]) < cfg.test_tasks_per_family + cfg.novel_tasks:
        raise RuntimeError("Permutation split leaves too few untouched test mappings")
    return result


def sample_neural_batch(
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    batch_size: int,
    fixed_context: Optional[int] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    family = torch.randint(0, cfg.recurring_templates, (batch_size,), device=device)
    selected_pool = torch.randint(0, len(pool_gpu), (batch_size,), device=device)
    permutation = pool_gpu[selected_pool]
    tables = relabelings_gpu[family, permutation].reshape(batch_size, CELLS)
    cell_order = torch.rand((batch_size, CELLS), device=device).argsort(dim=1)
    if fixed_context is None:
        lengths = torch.randint(0, cfg.maximum_context + 1, (batch_size,), device=device)
    else:
        lengths = torch.full((batch_size,), fixed_context, device=device, dtype=torch.long)
    context_cells = cell_order[:, : cfg.maximum_context]
    context_outputs = torch.gather(tables, 1, context_cells)
    mask = torch.arange(cfg.maximum_context, device=device)[None, :] < lengths[:, None]
    query_cells = torch.gather(cell_order, 1, lengths[:, None]).squeeze(1)
    targets = torch.gather(tables, 1, query_cells[:, None]).squeeze(1)
    return context_cells, context_outputs, mask, query_cells, targets


@torch.no_grad()
def neural_accuracy(
    model: nn.Module,
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    examples: int,
) -> float:
    model.eval()
    correct = 0
    seen = 0
    while seen < examples:
        batch = min(1_024, examples - seen)
        values = sample_neural_batch(
            relabelings_gpu, pool_gpu, batch, fixed_context=cfg.initial_context
        )
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(*values[:-1])
        correct += int((logits.argmax(dim=-1) == values[-1]).sum().item())
        seen += batch
    return correct / examples


def train_neural(
    recurring_relabelings: np.ndarray, splits: Dict[str, np.ndarray]
) -> Tuple[EquivariantTransitionGNN, List[Dict[str, float]]]:
    model = EquivariantTransitionGNN().to(device)
    tables_gpu = torch.from_numpy(recurring_relabelings.astype(np.int64)).to(device)
    train_pool = torch.from_numpy(splits["train"].astype(np.int64)).to(device)
    calibration_pool = torch.from_numpy(splits["neural_calibration"].astype(np.int64)).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, fused=device.type == "cuda"
    )
    scaler_enabled = device.type == "cuda" and amp_dtype == torch.float16
    if hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler(device.type, enabled=scaler_enabled)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
    history: List[Dict[str, float]] = []
    best_distance = float("inf")
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_step = 0
    best_accuracy = 0.0
    for step in range(1, cfg.neural_steps + 1):
        model.train()
        batch = sample_neural_batch(
            tables_gpu, train_pool, cfg.neural_batch, fixed_context=cfg.initial_context
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            loss = F.cross_entropy(model(*batch[:-1]), batch[-1])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if step % cfg.neural_eval_every == 0 or step == 1:
            accuracy = neural_accuracy(
                model, tables_gpu, calibration_pool, cfg.neural_eval_examples
            )
            history.append({"step": step, "loss": float(loss.item()), "calibration_accuracy": accuracy})
            distance = abs(accuracy - cfg.neural_target_accuracy)
            if distance < best_distance:
                best_distance = distance
                best_step = step
                best_accuracy = accuracy
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            print(
                f"neural step={step:4d} | loss={loss.item():.4f} | "
                f"heldout-k{cfg.initial_context} accuracy={100*accuracy:5.1f}%"
            )
            if (
                step >= 4 * cfg.neural_eval_every
                and distance <= cfg.neural_target_tolerance
                and accuracy >= cfg.neural_target_accuracy
            ):
                break
    if best_state is None:
        raise RuntimeError("No neural checkpoint was evaluated")
    model.load_state_dict(best_state)
    model.eval()
    print(f"restored imperfect-neural checkpoint step={best_step} | accuracy={100*best_accuracy:.2f}%")
    return model, history


@torch.no_grad()
def neural_full_probabilities(
    model: nn.Module, context_cells: Sequence[int], context_outputs: Sequence[int]
) -> np.ndarray:
    cells = list(context_cells)[-cfg.maximum_context :]
    outputs = list(context_outputs)[-cfg.maximum_context :]
    width = cfg.maximum_context
    padded_cells = np.zeros((CELLS, width), dtype=np.int64)
    padded_outputs = np.zeros((CELLS, width), dtype=np.int64)
    mask = np.zeros((CELLS, width), dtype=bool)
    if cells:
        padded_cells[:, : len(cells)] = np.asarray(cells)[None, :]
        padded_outputs[:, : len(outputs)] = np.asarray(outputs)[None, :]
        mask[:, : len(cells)] = True
    queries = np.arange(CELLS, dtype=np.int64)
    tensors = [
        torch.from_numpy(padded_cells).to(device),
        torch.from_numpy(padded_outputs).to(device),
        torch.from_numpy(mask).to(device),
        torch.from_numpy(queries).to(device),
    ]
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
        logits = model(*tensors)
    return logits.float().softmax(dim=-1).cpu().numpy()


def entropy_from_probabilities(probability: np.ndarray) -> float:
    positive = probability[probability > 0]
    return float(-np.sum(positive * np.log2(positive)))


def choose_weighted_query(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    unqueried: np.ndarray,
    generator: np.random.Generator,
    query_number: int,
    dual: bool = False,
) -> int:
    available = np.flatnonzero(unqueried)
    if dual and query_number > 0 and query_number % cfg.audit_period == 0:
        return int(generator.choice(available))
    entropies = []
    for cell in available:
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        entropies.append(entropy_from_probabilities(probability))
    entropies = np.asarray(entropies)
    best = available[np.isclose(entropies, entropies.max(), atol=1e-12)]
    return int(generator.choice(best))


def proposal_decode(
    truth_table: np.ndarray,
    neural_table: np.ndarray,
    grammar_predictions: np.ndarray,
    grammar_families: np.ndarray,
    initial_cells: np.ndarray,
    seed: int,
    top_k_override: Optional[int] = None,
) -> Dict[str, object]:
    truth = truth_table.reshape(-1).astype(np.int16)
    distance = np.sum(grammar_predictions != neural_table[None, :], axis=1)
    top_k = min(top_k_override or cfg.proposal_top_k, len(distance))
    proposal = np.argpartition(distance, top_k - 1)[:top_k].astype(np.int32)
    truth_in_proposal = bool(np.any(np.all(grammar_predictions[proposal] == truth[None, :], axis=1)))
    observed_cells = [int(cell) for cell in initial_cells]
    observed_outputs = [int(truth[cell]) for cell in observed_cells]
    candidates = proposal
    for cell, output in zip(observed_cells, observed_outputs):
        candidates = candidates[grammar_predictions[candidates, cell] == output]
    unqueried = np.ones(CELLS, dtype=bool)
    unqueried[initial_cells] = False
    generator = np.random.default_rng(seed)
    extra = 0
    rejected = len(candidates) == 0
    while extra < cfg.extraction_extra_queries and len(candidates) > 0:
        unique_tables = np.unique(grammar_predictions[candidates], axis=0)
        if len(unique_tables) == 1:
            break
        weights = np.ones(len(candidates), dtype=np.float64) / len(candidates)
        cell = choose_weighted_query(
            candidates,
            grammar_predictions,
            weights,
            unqueried,
            generator,
            extra,
            dual=True,
        )
        unqueried[cell] = False
        observed_cells.append(cell)
        observed_outputs.append(int(truth[cell]))
        candidates = candidates[grammar_predictions[candidates, cell] == truth[cell]]
        extra += 1
        rejected = len(candidates) == 0
    if len(candidates):
        best = int(candidates[np.argmin(distance[candidates])])
        selected_table = grammar_predictions[best]
        selected_family = int(grammar_families[best])
        resolved = bool(np.all(grammar_predictions[candidates] == selected_table[None, :]))
        exact = bool(np.array_equal(selected_table, truth))
    else:
        selected_family = -1
        resolved = False
        exact = False
    return {
        "neural_accuracy": float(np.mean(neural_table == truth)),
        "proposal_top_k": top_k,
        "truth_in_proposal": truth_in_proposal,
        "extra_queries": extra,
        "total_observations": len(initial_cells) + extra,
        "remaining_candidates": len(candidates),
        "resolved": resolved,
        "exact": exact,
        "selected_family": selected_family,
        "rejected": rejected,
    }


def posterior_metrics(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    truth: np.ndarray,
) -> Dict[str, float]:
    if len(candidates) == 0:
        return {
            "transition_accuracy": float("nan"),
            "posterior_nll": float("nan"),
            "resolved_fraction": float("nan"),
            "exact_table": 0.0,
            "candidates": 0.0,
        }
    top_correct = 0
    resolved = 0
    correct_probability = []
    for cell in range(CELLS):
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        top_correct += int(np.argmax(probability) == truth[cell])
        resolved += int(np.count_nonzero(probability > 1e-12) == 1)
        correct_probability.append(probability[int(truth[cell])])
    return {
        "transition_accuracy": top_correct / CELLS,
        "posterior_nll": float(-np.mean(np.log(np.clip(correct_probability, 1e-15, 1.0)))),
        "resolved_fraction": resolved / CELLS,
        "exact_table": float(resolved == CELLS),
        "candidates": float(len(candidates)),
    }


def neural_weights(
    candidates: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    beta: float,
) -> np.ndarray:
    if beta == 0.0:
        return np.ones(len(candidates), dtype=np.float64) / len(candidates)
    log_probability = np.log(np.clip(probabilities, 1e-9, 1.0))
    scores = log_probability[np.arange(CELLS)[None, :], predictions[candidates]].sum(axis=1)
    scores = beta * (scores - scores.max())
    weights = np.exp(np.clip(scores, -80.0, 0.0))
    return weights / weights.sum()


def transfer_trace(
    truth_table: np.ndarray,
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    beta: float,
    seed: int,
    maximum_queries: int = 12,
    dual: bool = False,
) -> Tuple[List[Dict[str, float]], Optional[int]]:
    truth = truth_table.reshape(-1).astype(np.int16)
    candidates = np.arange(len(predictions), dtype=np.int32)
    unqueried = np.ones(CELLS, dtype=bool)
    context_cells: List[int] = []
    context_outputs: List[int] = []
    generator = np.random.default_rng(seed)
    rows: List[Dict[str, float]] = []
    rejected_at: Optional[int] = None
    for query_count in range(maximum_queries + 1):
        neural_probability = neural_full_probabilities(model, context_cells, context_outputs)
        if len(candidates):
            weights = neural_weights(candidates, predictions, neural_probability, beta)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        else:
            weights = np.empty(0)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        neural_prediction = neural_probability.argmax(axis=1)
        rows.append(
            {
                "queries": float(query_count),
                **symbolic,
                "neural_transition_accuracy": float(np.mean(neural_prediction == truth)),
                "neural_exact_table": float(np.array_equal(neural_prediction, truth)),
            }
        )
        if query_count == maximum_queries or len(candidates) == 0:
            break
        cell = choose_weighted_query(
            candidates,
            predictions,
            weights,
            unqueried,
            generator,
            query_count,
            dual=dual,
        )
        unqueried[cell] = False
        context_cells.append(cell)
        context_outputs.append(int(truth[cell]))
        candidates = candidates[predictions[candidates, cell] == truth[cell]]
        if len(candidates) == 0:
            rejected_at = query_count + 1
    return rows, rejected_at


def first_exact(trace: Sequence[Dict[str, float]]) -> int:
    hits = [int(row["queries"]) for row in trace if row["exact_table"] == 1.0]
    return min(hits) if hits else CELLS + 1


def metric_at(trace: Sequence[Dict[str, float]], budget: int, key: str) -> float:
    eligible = [row for row in trace if int(row["queries"]) <= budget]
    return float(eligible[-1][key])


def evaluate_beta(
    beta: float,
    tasks: Sequence[np.ndarray],
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    seed_offset: int,
) -> Dict[str, float]:
    traces = [
        transfer_trace(task, predictions, model, beta, cfg.seed + seed_offset + index)[0]
        for index, task in enumerate(tasks)
    ]
    return {
        "beta": beta,
        "exact_at_8": float(np.mean([metric_at(trace, 8, "exact_table") for trace in traces])),
        "accuracy_at_4": float(np.mean([metric_at(trace, 4, "transition_accuracy") for trace in traces])),
        "median_exact_queries": float(np.median([first_exact(trace) for trace in traces])),
    }


def write_csv(path: Path, rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


print("=" * 118)
print("NEURAL-QUOTIENT ACTIVE LEARNER V13.1 — decode, quotient, transfer, reorganize")
print("=" * 118)
print(
    f"device={device} | GPU={torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'} | "
    f"states={N} | actions={A} | cells={CELLS} | permutations={N_PERMUTATIONS:,} | fast={FAST_DEV_RUN}"
)
started = time.perf_counter()

excluded: set[bytes] = set()
grammar = [generate_machine(excluded) for _ in range(cfg.candidate_templates)]
recurring = grammar[: cfg.recurring_templates]
grammar_predictions, grammar_families = hypothesis_predictions(grammar)
recurring_relabelings = np.stack([all_relabelings(table) for table in recurring]).astype(np.int8)
splits = permutation_splits()
print(
    f"candidate grammar={len(grammar)} canonical laws | hypotheses={len(grammar_predictions):,} | "
    f"recurring latent laws={len(recurring)}"
)

model, training_history = train_neural(recurring_relabelings, splits)

# Crystallize recurring rules from imperfect neural tables plus actively requested evidence.
prior_rows: List[Dict[str, object]] = []
shuffled_rows: List[Dict[str, object]] = []
proposal_sweep_rows: List[Dict[str, object]] = []
naive_keys: set[bytes] = set()
decoded_family_counts: Dict[int, int] = {}
prior_pool = splits["prior"]
prior_cursor = 0
for family in range(cfg.recurring_templates):
    for task_in_family in range(cfg.prior_tasks_per_family):
        permutation_index = int(prior_pool[prior_cursor % len(prior_pool)])
        prior_cursor += 1
        truth = relabel_table(recurring[family], permutation_index)
        flat_truth = truth.reshape(-1)
        cell_order = np.random.default_rng(cfg.seed + 10_000 + prior_cursor).permutation(CELLS)
        initial_cells = cell_order[: cfg.initial_context]
        initial_outputs = flat_truth[initial_cells]
        probability = neural_full_probabilities(model, initial_cells, initial_outputs)
        neural_table = probability.argmax(axis=1).astype(np.int16)
        naive_canonical, _ = canonicalize(neural_table.reshape(N, A))
        naive_keys.add(naive_canonical.astype(np.int8).tobytes())
        sweep_decodes: Dict[int, Dict[str, object]] = {}
        for proposal_k in cfg.proposal_sweep:
            sweep_decodes[proposal_k] = proposal_decode(
                truth,
                neural_table,
                grammar_predictions,
                grammar_families,
                initial_cells,
                cfg.seed + 20_000 + prior_cursor,
                top_k_override=proposal_k,
            )
            proposal_sweep_rows.append(
                {
                    "family": family,
                    "task": task_in_family,
                    **sweep_decodes[proposal_k],
                }
            )
        if cfg.proposal_top_k in sweep_decodes:
            decoded = sweep_decodes[cfg.proposal_top_k]
        else:
            decoded = proposal_decode(
                truth,
                neural_table,
                grammar_predictions,
                grammar_families,
                initial_cells,
                cfg.seed + 20_000 + prior_cursor,
            )
        prior_rows.append({"family": family, "task": task_in_family, **decoded})
        if decoded["resolved"] and not decoded["rejected"]:
            selected = int(decoded["selected_family"])
            decoded_family_counts[selected] = decoded_family_counts.get(selected, 0) + 1

        shuffled = neural_table[np.random.default_rng(cfg.seed + 30_000 + prior_cursor).permutation(CELLS)]
        shuffled_decoded = proposal_decode(
            truth,
            shuffled,
            grammar_predictions,
            grammar_families,
            initial_cells,
            cfg.seed + 40_000 + prior_cursor,
        )
        shuffled_rows.append({"family": family, "task": task_in_family, **shuffled_decoded})

decoded_ids = sorted(
    family for family, count in decoded_family_counts.items() if count >= cfg.recurrence_threshold
)
decoded_library = [grammar[family] for family in decoded_ids]
clean_library = list(recurring)
expected_keys = {table.astype(np.int8).tobytes() for table in recurring}
decoded_keys = {table.astype(np.int8).tobytes() for table in decoded_library}
library_recovery = len(expected_keys & decoded_keys) / len(expected_keys)
prior_neural_accuracy = float(np.mean([row["neural_accuracy"] for row in prior_rows]))
prior_exact = float(np.mean([row["exact"] for row in prior_rows]))
prior_resolved = float(np.mean([row["resolved"] for row in prior_rows]))
prior_median_observations = float(
    np.median([row["total_observations"] for row in prior_rows if row["exact"]])
)
shuffled_exact = float(np.mean([row["exact"] for row in shuffled_rows]))
print("\nCRYSTALLIZATION FROM THE IMPERFECT NEURAL CHANNEL")
print("-" * 118)
print(
    f"neural table accuracy={100*prior_neural_accuracy:.2f}% | active exact={100*prior_exact:.1f}% | "
    f"resolved={100*prior_resolved:.1f}% | median total observations={prior_median_observations:.1f}/{CELLS}"
)
for proposal_k in cfg.proposal_sweep:
    rows = [row for row in proposal_sweep_rows if int(row["proposal_top_k"]) == proposal_k]
    exact_rows = [row for row in rows if row["exact"]]
    print(
        f" proposal top-k={proposal_k:6,d} | truth recall={100*np.mean([row['truth_in_proposal'] for row in rows]):5.1f}% | "
        f"active exact={100*np.mean([row['exact'] for row in rows]):5.1f}% | "
        f"median observations={np.median([row['total_observations'] for row in exact_rows]) if exact_rows else float('nan'):.1f}"
    )
print(
    f"naive noisy canonical templates={len(naive_keys)} | recurring decoded templates={len(decoded_library)} | "
    f"true-template recovery={100*library_recovery:.1f}% | shuffled-control exact={100*shuffled_exact:.1f}%"
)
if not decoded_library:
    raise RuntimeError("No recurring rules survived active crystallization")

decoded_predictions, _ = hypothesis_predictions(decoded_library)
clean_predictions, _ = hypothesis_predictions(clean_library)

# Calibrate neural responsibility on mappings disjoint from training, crystallization, and test.
gate_tasks: List[np.ndarray] = []
gate_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.gate_tasks_per_family):
        gate_tasks.append(relabel_table(recurring[family], int(splits["gate"][gate_cursor])))
        gate_cursor += 1
gate_rows = [
    evaluate_beta(beta, gate_tasks, decoded_predictions, model, 100_000 + int(10_000 * beta))
    for beta in cfg.gate_betas
]
symbolic_gate = next(row for row in gate_rows if row["beta"] == 0.0)
safe_gate_rows = [
    row
    for row in gate_rows
    if row["exact_at_8"] + 1e-12 >= symbolic_gate["exact_at_8"]
    and row["accuracy_at_4"] + 1e-12 >= symbolic_gate["accuracy_at_4"]
]
selected_gate = max(
    safe_gate_rows,
    key=lambda row: (row["exact_at_8"], -row["median_exact_queries"], row["accuracy_at_4"], -row["beta"]),
)
selected_beta = float(selected_gate["beta"])
print("\nHELD-OUT GATE CALIBRATION")
print("-" * 118)
for row in gate_rows:
    print(
        f"beta={row['beta']:.2f} | exact@8={100*row['exact_at_8']:5.1f}% | "
        f"accuracy@4={100*row['accuracy_at_4']:5.1f}% | median exact={row['median_exact_queries']:.1f}"
    )
print(f"selected no-harm beta={selected_beta:.2f}")

# Untouched permutation transfer.
test_tasks: List[Tuple[int, np.ndarray]] = []
test_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.test_tasks_per_family):
        test_tasks.append((family, relabel_table(recurring[family], int(splits["test"][test_cursor]))))
        test_cursor += 1

transfer_rows: List[Dict[str, object]] = []
exact_queries: Dict[str, List[int]] = {"symbolic": [], "calibrated-hybrid": [], "clean-oracle": []}
for task_id, (family, truth) in enumerate(test_tasks):
    for method, predictions, beta in (
        ("symbolic", decoded_predictions, 0.0),
        ("calibrated-hybrid", decoded_predictions, selected_beta),
        ("clean-oracle", clean_predictions, 0.0),
    ):
        trace, rejected = transfer_trace(
            truth,
            predictions,
            model,
            beta,
            cfg.seed + 200_000 + 101 * task_id + {"symbolic": 1, "calibrated-hybrid": 2, "clean-oracle": 3}[method],
        )
        if rejected is not None:
            raise RuntimeError(f"Known-family task rejected by {method}")
        exact_queries[method].append(first_exact(trace))
        for row in trace:
            if int(row["queries"]) in cfg.report_budgets:
                transfer_rows.append(
                    {"method": method, "task": task_id, "family": family, **row}
                )


def transfer_summary(method: str, budget: int, key: str) -> float:
    rows = [
        row
        for row in transfer_rows
        if row["method"] == method and int(row["queries"]) == budget
    ]
    return float(np.nanmean([float(row[key]) for row in rows]))


print("\nUNTOUCHED-REPRESENTATION TRANSFER")
print("-" * 118)
for method in exact_queries:
    print(
        f"{method:18s} | median exact queries={np.median(exact_queries[method]):4.1f} | "
        f"exact@8={100*transfer_summary(method, 8, 'exact_table'):5.1f}% | "
        f"table accuracy@4={100*transfer_summary(method, 4, 'transition_accuracy'):5.1f}%"
    )
neural_accuracy_at_8 = transfer_summary("symbolic", 8, "neural_transition_accuracy")
neural_exact_at_8 = transfer_summary("symbolic", 8, "neural_exact_table")
print(
    f"direct imperfect neural @8 observations | table accuracy={100*neural_accuracy_at_8:.1f}% | "
    f"exact tables={100*neural_exact_at_8:.1f}% | local tabular exact@8=0.0%"
)

# Novel-law rejection, MDL promotion, and future reuse.
novel = mutate_machine(recurring[0], cfg.novel_changes, excluded)
novel_tables = [
    relabel_table(novel, int(splits["test"][(test_cursor + index) % len(splits["test"])]))
    for index in range(cfg.novel_tasks)
]
detection_queries: List[int] = []
for task_id, truth in enumerate(novel_tables):
    _, rejected = transfer_trace(
        truth,
        decoded_predictions,
        model,
        0.0,
        cfg.seed + 400_000 + task_id,
        maximum_queries=12,
        dual=True,
    )
    detection_queries.append(CELLS + 1 if rejected is None else rejected)

nearest_mismatches = int(
    min(np.min(np.sum(predictions != novel_tables[0].reshape(1, -1), axis=1)) for predictions in [decoded_predictions])
)
permutation_bits = math.log2(math.factorial(N))
existing_task_bits = (
    math.log2(len(decoded_library))
    + permutation_bits
    + math.log2(math.comb(CELLS, nearest_mismatches))
    + nearest_mismatches * math.log2(N - 1)
)
definition_bits = CELLS * math.log2(N)
new_task_bits = math.log2(len(decoded_library) + 1) + permutation_bits
promotion_at: Optional[int] = None
promotion_rows: List[Dict[str, object]] = []
for repeats in range(1, cfg.novel_tasks + 1):
    exceptions_bits = repeats * existing_task_bits
    promotion_bits = definition_bits + repeats * new_task_bits
    if promotion_at is None and promotion_bits < exceptions_bits:
        promotion_at = repeats
    promotion_rows.append(
        {
            "repeated_worlds": repeats,
            "exceptions_bits": exceptions_bits,
            "promotion_bits": promotion_bits,
        }
    )

expanded_predictions, _ = hypothesis_predictions(decoded_library + [novel])
future_exact = []
for task_id, truth in enumerate(novel_tables):
    trace, rejected = transfer_trace(
        truth,
        expanded_predictions,
        model,
        0.0,
        cfg.seed + 500_000 + task_id,
    )
    if rejected is not None:
        raise RuntimeError("Promoted family was rejected")
    future_exact.append(first_exact(trace))
detected_by_12 = float(np.mean(np.asarray(detection_queries) <= 12))
future_median = float(np.median(future_exact))
print("\nNOVEL LAW AND REORGANIZATION")
print("-" * 118)
print(
    f"detected by 12={100*detected_by_12:.1f}% | median detection={np.median(detection_queries):.1f} | "
    f"nearest residual={nearest_mismatches} transitions | promote after={promotion_at} recurring worlds | "
    f"future median exact={future_median:.1f} queries"
)

predictions_declared = {
    "P1_actual_neural_channel_is_imperfect_45_to_60pct": 0.45 <= prior_neural_accuracy <= 0.60,
    "P2_active_structural_decoder_recovers_90pct_with_fewer_than_24_observations": (
        prior_exact >= 0.90 and prior_median_observations <= 16
    ),
    "P3_noisy_active_crystallization_recovers_all_recurring_templates": library_recovery == 1.0,
    "P4_structure_not_cell_order_drives_decoding": prior_exact - shuffled_exact >= 0.50,
    "P5_unseen_representation_exact_at_8_at_least_90pct": transfer_summary(
        "symbolic", 8, "exact_table"
    )
    >= 0.90,
    "P6_calibrated_gate_no_harm_on_untouched_test": (
        transfer_summary("calibrated-hybrid", 8, "exact_table") + 1e-12
        >= transfer_summary("symbolic", 8, "exact_table")
    ),
    "P7_symbolic_transfer_beats_direct_neural_and_local_at_8": (
        transfer_summary("symbolic", 8, "exact_table") > neural_exact_at_8
        and transfer_summary("symbolic", 8, "exact_table") > 0.0
    ),
    "P8_novel_law_is_rejected_then_reusable_after_mdl_promotion": (
        detected_by_12 >= 0.90 and promotion_at is not None and promotion_at <= 8 and future_median < CELLS / 2
    ),
}
print("\nPREDECLARED PREDICTIONS")
print("-" * 118)
for name, passed in predictions_declared.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(out_dir / "training_history.csv", training_history)
write_csv(out_dir / "prior_crystallization.csv", prior_rows)
write_csv(out_dir / "proposal_sweep.csv", proposal_sweep_rows)
write_csv(out_dir / "shuffled_control.csv", shuffled_rows)
write_csv(out_dir / "gate_calibration.csv", gate_rows)
write_csv(out_dir / "transfer_trials.csv", transfer_rows)
write_csv(out_dir / "promotion.csv", promotion_rows)

result = {
    "config": asdict(cfg),
    "predictions": predictions_declared,
    "neural": {
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "prior_table_accuracy": prior_neural_accuracy,
    },
    "crystallization": {
        "prior_exact": prior_exact,
        "prior_resolved": prior_resolved,
        "median_total_observations": prior_median_observations,
        "naive_noisy_templates": len(naive_keys),
        "decoded_templates": len(decoded_library),
        "template_recovery": library_recovery,
        "shuffled_exact": shuffled_exact,
        "proposal_sweep": [
            {
                "top_k": proposal_k,
                "truth_recall": float(
                    np.mean(
                        [
                            row["truth_in_proposal"]
                            for row in proposal_sweep_rows
                            if int(row["proposal_top_k"]) == proposal_k
                        ]
                    )
                ),
                "exact": float(
                    np.mean(
                        [
                            row["exact"]
                            for row in proposal_sweep_rows
                            if int(row["proposal_top_k"]) == proposal_k
                        ]
                    )
                ),
            }
            for proposal_k in cfg.proposal_sweep
        ],
    },
    "gate": {"selected_beta": selected_beta, "calibration": gate_rows},
    "transfer": {
        method: {
            "median_exact_queries": float(np.median(values)),
            "exact_at_8": transfer_summary(method, 8, "exact_table"),
            "accuracy_at_4": transfer_summary(method, 4, "transition_accuracy"),
        }
        for method, values in exact_queries.items()
    },
    "direct_neural": {
        "accuracy_at_8": neural_accuracy_at_8,
        "exact_at_8": neural_exact_at_8,
    },
    "novelty": {
        "detected_by_12": detected_by_12,
        "median_detection_queries": float(np.median(detection_queries)),
        "promotion_at": promotion_at,
        "future_median_exact_queries": future_median,
    },
    "elapsed_seconds": time.perf_counter() - started,
}
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
(out_dir / "results.json").write_text(json.dumps(result, indent=2))
torch.save(model.state_dict(), out_dir / "imperfect_neural.pt")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
ax.plot(
    [row["step"] for row in training_history],
    [100 * row["calibration_accuracy"] for row in training_history],
    marker="o",
)
ax.axhline(100 * cfg.neural_target_accuracy, color="black", linestyle="--", label="target imperfect channel")
ax.set_title("Actual neural learner checkpoint")
ax.set_xlabel("optimizer step")
ax.set_ylabel("held-out transition accuracy (%)")
ax.grid(alpha=0.25)
ax.legend()

ax = axes[0, 1]
labels = ["neural table", "shuffled control", "active decode"]
values = [prior_neural_accuracy, shuffled_exact, prior_exact]
ax.bar(labels, [100 * value for value in values], color=["#4C78A8", "#E45756", "#54A24B"])
ax.set_ylabel("accuracy / exact recovery (%)")
ax.set_title("Noisy codeword becomes an exact rule")
ax.set_ylim(0, 105)

ax = axes[1, 0]
for method, color in (("symbolic", "#54A24B"), ("calibrated-hybrid", "#B279A2"), ("clean-oracle", "#F2CF5B")):
    budgets = []
    exact = []
    for budget in cfg.report_budgets:
        rows = [row for row in transfer_rows if row["method"] == method and int(row["queries"]) == budget]
        if rows:
            budgets.append(budget)
            exact.append(100 * np.mean([row["exact_table"] for row in rows]))
    ax.plot(budgets, exact, marker="o", label=method, color=color)
ax.axvline(8, color="black", linestyle=":")
ax.set_title("Exact rule on unseen state names")
ax.set_xlabel("actively observed transitions")
ax.set_ylabel("exact full table (%)")
ax.set_ylim(-2, 102)
ax.grid(alpha=0.25)
ax.legend()

ax = axes[1, 1]
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["exceptions_bits"] for row in promotion_rows],
    marker="o",
    label="keep as residuals",
    color="#E45756",
)
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["promotion_bits"] for row in promotion_rows],
    marker="o",
    label="promote rule",
    color="#54A24B",
)
if promotion_at is not None:
    ax.axvline(promotion_at, color="black", linestyle="--", label=f"promotion at {promotion_at}")
ax.set_title("Residual recurrence triggers reorganization")
ax.set_xlabel("repeated novel worlds")
ax.set_ylabel("description length (bits)")
ax.grid(alpha=0.25)
ax.legend()

fig.suptitle(
    "Neural-to-symbolic active learning: predict noisily, decode exactly, transfer invariantly",
    fontsize=15,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

bundle = shutil.make_archive(
    str(out_dir.parent / f"{out_dir.name}_bundle"), "zip", root_dir=out_dir
)
print("\n" + "=" * 118)
print("FINAL V13.1 SUMMARY")
print("=" * 118)
print(
    f"neural={100*prior_neural_accuracy:.1f}% local transitions -> active crystallization={100*prior_exact:.1f}% exact | "
    f"library recovery={100*library_recovery:.1f}%"
)
print(
    f"unseen representation exact@8={100*transfer_summary('symbolic', 8, 'exact_table'):.1f}% | "
    f"median exact={np.median(exact_queries['symbolic']):.1f}/24 queries | novel detected@12={100*detected_by_12:.1f}%"
)
print(f"elapsed={result['elapsed_seconds']:.1f}s | artifacts={out_dir} | bundle={bundle}")
"""
NEURAL-QUOTIENT ACTIVE LEARNER V13 — SINGLE-CELL A100 EXPERIMENT

Breakthrough test
-----------------
Treat an imperfect neural transition table as a noisy codeword on a structured
program manifold.  Decode that codeword against a broad finite-machine grammar,
ask the environment only the maximally informative missing transitions, quotient
the verified rule by arbitrary state renaming, and retain recurring rules by MDL.

The decisive transfer test uses coordinate permutations never seen by the neural
learner.  The system must recover a complete 24-transition world from substantially
fewer than 24 observations, reject a genuinely novel law, and reorganize only when
the same residual law recurs.

What is learned and what is supplied
------------------------------------
Learned: a real in-context Transformer transition predictor; recurring canonical
templates; a held-out calibrated neural/symbolic responsibility gate.

Supplied: eight states, three actions, a finite grammar of candidate deterministic
machines, and the permission to query any state-action cell.  This experiment tests
error-correcting crystallization and invariant transfer, not autonomous ontology or
grammar discovery.
"""


## Experiment 7: Appended older V13 rerun (interrupted)

Original cell `5`, split part `2`.

**Cleanup note.** Two complete programs were concatenated in the source cell; they are separated here so each has valid future imports and independent artifacts.


In [ ]:
from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


FAST_DEV_RUN = os.environ.get("NEURAL_QUOTIENT_V13_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("NEURAL_QUOTIENT_V13_SEED", "20260813"))
    n_states: int = 8
    n_actions: int = 3
    candidate_templates: int = 32
    recurring_templates: int = 4
    train_permutations: int = 30_000
    neural_calibration_permutations: int = 4_000
    prior_permutations: int = 2_000
    gate_permutations: int = 2_000
    prior_tasks_per_family: int = 6
    gate_tasks_per_family: int = 3
    test_tasks_per_family: int = 8
    initial_context: int = 8
    maximum_context: int = 8
    extraction_extra_queries: int = 8
    proposal_top_k: int = 4_096
    recurrence_threshold: int = 3
    neural_target_accuracy: float = 0.53
    neural_target_tolerance: float = 0.025
    neural_steps: int = 5_000
    neural_batch: int = 1_024
    neural_eval_examples: int = 8_192
    neural_eval_every: int = 25
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    width: int = 128
    heads: int = 4
    layers: int = 5
    dropout: float = 0.0
    gate_betas: Tuple[float, ...] = (0.0, 0.10, 0.25, 0.50, 1.0)
    report_budgets: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6, 8, 10, 12)
    novel_tasks: int = 24
    novel_changes: int = 4
    audit_period: int = 3
    output_dir: str = os.environ.get(
        "NEURAL_QUOTIENT_V13_OUTPUT_DIR",
        "outputs/07-appended-older-v13-rerun-interrupted/neural_quotient_active_v13_results"
        if Path("/content").exists()
        else "outputs/07-appended-older-v13-rerun-interrupted/neural_quotient_active_v13_results",
    )


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        candidate_templates=8,
        train_permutations=24_000,
        neural_calibration_permutations=3_000,
        prior_permutations=1_000,
        gate_permutations=1_000,
        prior_tasks_per_family=2,
        gate_tasks_per_family=1,
        test_tasks_per_family=2,
        proposal_top_k=1_024,
        neural_steps=1_000,
        neural_batch=256,
        neural_eval_examples=1_024,
        neural_eval_every=10,
        width=64,
        layers=3,
        novel_tasks=8,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
rng = np.random.default_rng(cfg.seed)
N, A = cfg.n_states, cfg.n_actions
CELLS = N * A
PERMUTATIONS = np.asarray(list(itertools.permutations(range(N))), dtype=np.int16)
INVERSES = np.argsort(PERMUTATIONS, axis=1).astype(np.int16)
N_PERMUTATIONS = len(PERMUTATIONS)


def strongly_connected(table: np.ndarray) -> bool:
    for source in range(N):
        seen = {source}
        frontier = [source]
        while frontier:
            state = frontier.pop()
            for successor in table[state]:
                successor = int(successor)
                if successor not in seen:
                    seen.add(successor)
                    frontier.append(successor)
        if len(seen) != N:
            return False
    return True


def all_relabelings(table: np.ndarray) -> np.ndarray:
    old_successors = table[INVERSES]
    rows = np.arange(N_PERMUTATIONS)[:, None, None]
    return PERMUTATIONS[rows, old_successors]


def canonicalize(table: np.ndarray) -> Tuple[np.ndarray, int]:
    relabeled = all_relabelings(np.asarray(table, dtype=np.int16))
    flat = relabeled.reshape(N_PERMUTATIONS, CELLS)
    first = int(np.lexsort(flat[:, ::-1].T)[0])
    canonical = relabeled[first].copy()
    automorphisms = int(np.sum(np.all(flat == canonical.reshape(1, -1), axis=1)))
    return canonical, automorphisms


def relabel_table(table: np.ndarray, permutation_index: int) -> np.ndarray:
    old_to_new = PERMUTATIONS[int(permutation_index)]
    new_to_old = INVERSES[int(permutation_index)]
    return old_to_new[table[new_to_old]]


def generate_machine(excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = rng.integers(0, N, size=(N, A), dtype=np.int16)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a distinct asymmetric machine")


def mutate_machine(base: np.ndarray, changes: int, excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = base.copy()
        for cell in rng.choice(CELLS, size=changes, replace=False):
            state, action = divmod(int(cell), A)
            old = int(candidate[state, action])
            draw = int(rng.integers(0, N - 1))
            candidate[state, action] = draw + (draw >= old)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a novel machine")


def hypothesis_predictions(library: Sequence[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    predictions: List[np.ndarray] = []
    family_ids: List[np.ndarray] = []
    row_index = np.arange(N_PERMUTATIONS)[:, None, None]
    action_index = np.arange(A)[None, None, :]
    for family, table in enumerate(library):
        canonical_next = table[PERMUTATIONS[:, :, None], action_index]
        observed_next = INVERSES[row_index, canonical_next]
        predictions.append(observed_next.reshape(N_PERMUTATIONS, CELLS).astype(np.int8))
        family_ids.append(np.full(N_PERMUTATIONS, family, dtype=np.int16))
    return np.concatenate(predictions), np.concatenate(family_ids)


class EquivariantTransitionGNN(nn.Module):
    """Permutation-equivariant neural predictor over a partially observed transition graph."""

    def __init__(self) -> None:
        super().__init__()
        width = cfg.width
        self.action = nn.Embedding(A, width)
        self.base_node = nn.Parameter(torch.zeros(width))
        self.query_role = nn.Embedding(2, width)
        self.to_source = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.to_target = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.updates = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(3 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(cfg.layers)])
        self.scorer = nn.Sequential(
            nn.Linear(4 * width + 1, 2 * width),
            nn.GELU(),
            nn.Linear(2 * width, width),
            nn.GELU(),
            nn.Linear(width, 1),
        )

    def forward(
        self,
        context_cells: torch.Tensor,
        context_outputs: torch.Tensor,
        context_mask: torch.Tensor,
        query_cells: torch.Tensor,
    ) -> torch.Tensor:
        batch, _ = context_cells.shape
        context_states = torch.div(context_cells, A, rounding_mode="floor")
        context_actions = context_cells % A
        query_states = torch.div(query_cells, A, rounding_mode="floor")
        query_actions = query_cells % A
        node_ids = torch.arange(N, device=context_cells.device)[None, :]
        roles = (node_ids == query_states[:, None]).long()
        nodes = self.base_node[None, None, :].expand(batch, N, -1) + self.query_role(roles)

        valid_batch, valid_slot = torch.nonzero(context_mask, as_tuple=True)
        source = context_states[valid_batch, valid_slot]
        target = context_outputs[valid_batch, valid_slot]
        action = context_actions[valid_batch, valid_slot]
        batch_offsets = valid_batch * N
        source_flat = batch_offsets + source
        target_flat = batch_offsets + target

        for layer in range(cfg.layers):
            flat = nodes.reshape(batch * N, cfg.width)
            action_features = self.action(action)
            source_messages = self.to_source[layer](
                torch.cat([flat[target_flat], action_features], dim=-1)
            ).float()
            target_messages = self.to_target[layer](
                torch.cat([flat[source_flat], action_features], dim=-1)
            ).float()
            aggregate_source = torch.zeros_like(flat)
            aggregate_target = torch.zeros_like(flat)
            aggregate_source.index_add_(0, source_flat, source_messages)
            aggregate_target.index_add_(0, target_flat, target_messages)
            degree_source = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            degree_target = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            ones = torch.ones((len(source_flat), 1), device=flat.device, dtype=flat.dtype)
            degree_source.index_add_(0, source_flat, ones)
            degree_target.index_add_(0, target_flat, ones)
            aggregate_source /= degree_source.clamp_min(1.0)
            aggregate_target /= degree_target.clamp_min(1.0)
            update = self.updates[layer](
                torch.cat([flat, aggregate_source, aggregate_target], dim=-1)
            )
            nodes = self.norms[layer](flat + update).reshape(batch, N, cfg.width)

        source_features = nodes[torch.arange(batch, device=nodes.device), query_states]
        candidate_features = nodes
        action_features = self.action(query_actions)
        expanded_source = source_features[:, None, :].expand(-1, N, -1)
        expanded_action = action_features[:, None, :].expand(-1, N, -1)
        equality = (node_ids == query_states[:, None]).to(nodes.dtype)[..., None]
        features = torch.cat(
            [
                expanded_source,
                candidate_features,
                expanded_source * candidate_features,
                expanded_action,
                equality,
            ],
            dim=-1,
        )
        return self.scorer(features).squeeze(-1)


def permutation_splits() -> Dict[str, np.ndarray]:
    order = rng.permutation(N_PERMUTATIONS)
    cursor = 0
    result: Dict[str, np.ndarray] = {}
    for name, length in (
        ("train", cfg.train_permutations),
        ("neural_calibration", cfg.neural_calibration_permutations),
        ("prior", cfg.prior_permutations),
        ("gate", cfg.gate_permutations),
    ):
        result[name] = order[cursor : cursor + length]
        cursor += length
    result["test"] = order[cursor:]
    if len(result["test"]) < cfg.test_tasks_per_family + cfg.novel_tasks:
        raise RuntimeError("Permutation split leaves too few untouched test mappings")
    return result


def sample_neural_batch(
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    batch_size: int,
    fixed_context: Optional[int] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    family = torch.randint(0, cfg.recurring_templates, (batch_size,), device=device)
    selected_pool = torch.randint(0, len(pool_gpu), (batch_size,), device=device)
    permutation = pool_gpu[selected_pool]
    tables = relabelings_gpu[family, permutation].reshape(batch_size, CELLS)
    cell_order = torch.rand((batch_size, CELLS), device=device).argsort(dim=1)
    if fixed_context is None:
        lengths = torch.randint(0, cfg.maximum_context + 1, (batch_size,), device=device)
    else:
        lengths = torch.full((batch_size,), fixed_context, device=device, dtype=torch.long)
    context_cells = cell_order[:, : cfg.maximum_context]
    context_outputs = torch.gather(tables, 1, context_cells)
    mask = torch.arange(cfg.maximum_context, device=device)[None, :] < lengths[:, None]
    query_cells = torch.gather(cell_order, 1, lengths[:, None]).squeeze(1)
    targets = torch.gather(tables, 1, query_cells[:, None]).squeeze(1)
    return context_cells, context_outputs, mask, query_cells, targets


@torch.no_grad()
def neural_accuracy(
    model: nn.Module,
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    examples: int,
) -> float:
    model.eval()
    correct = 0
    seen = 0
    while seen < examples:
        batch = min(1_024, examples - seen)
        values = sample_neural_batch(
            relabelings_gpu, pool_gpu, batch, fixed_context=cfg.initial_context
        )
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(*values[:-1])
        correct += int((logits.argmax(dim=-1) == values[-1]).sum().item())
        seen += batch
    return correct / examples


def train_neural(
    recurring_relabelings: np.ndarray, splits: Dict[str, np.ndarray]
) -> Tuple[EquivariantTransitionGNN, List[Dict[str, float]]]:
    model = EquivariantTransitionGNN().to(device)
    tables_gpu = torch.from_numpy(recurring_relabelings.astype(np.int64)).to(device)
    train_pool = torch.from_numpy(splits["train"].astype(np.int64)).to(device)
    calibration_pool = torch.from_numpy(splits["neural_calibration"].astype(np.int64)).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, fused=device.type == "cuda"
    )
    scaler_enabled = device.type == "cuda" and amp_dtype == torch.float16
    if hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler(device.type, enabled=scaler_enabled)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
    history: List[Dict[str, float]] = []
    best_distance = float("inf")
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_step = 0
    best_accuracy = 0.0
    for step in range(1, cfg.neural_steps + 1):
        model.train()
        batch = sample_neural_batch(
            tables_gpu, train_pool, cfg.neural_batch, fixed_context=cfg.initial_context
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            loss = F.cross_entropy(model(*batch[:-1]), batch[-1])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if step % cfg.neural_eval_every == 0 or step == 1:
            accuracy = neural_accuracy(
                model, tables_gpu, calibration_pool, cfg.neural_eval_examples
            )
            history.append({"step": step, "loss": float(loss.item()), "calibration_accuracy": accuracy})
            distance = abs(accuracy - cfg.neural_target_accuracy)
            if distance < best_distance:
                best_distance = distance
                best_step = step
                best_accuracy = accuracy
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            print(
                f"neural step={step:4d} | loss={loss.item():.4f} | "
                f"heldout-k{cfg.initial_context} accuracy={100*accuracy:5.1f}%"
            )
            if (
                step >= 4 * cfg.neural_eval_every
                and distance <= cfg.neural_target_tolerance
                and accuracy >= cfg.neural_target_accuracy
            ):
                break
    if best_state is None:
        raise RuntimeError("No neural checkpoint was evaluated")
    model.load_state_dict(best_state)
    model.eval()
    print(f"restored imperfect-neural checkpoint step={best_step} | accuracy={100*best_accuracy:.2f}%")
    return model, history


@torch.no_grad()
def neural_full_probabilities(
    model: nn.Module, context_cells: Sequence[int], context_outputs: Sequence[int]
) -> np.ndarray:
    cells = list(context_cells)[-cfg.maximum_context :]
    outputs = list(context_outputs)[-cfg.maximum_context :]
    width = cfg.maximum_context
    padded_cells = np.zeros((CELLS, width), dtype=np.int64)
    padded_outputs = np.zeros((CELLS, width), dtype=np.int64)
    mask = np.zeros((CELLS, width), dtype=bool)
    if cells:
        padded_cells[:, : len(cells)] = np.asarray(cells)[None, :]
        padded_outputs[:, : len(outputs)] = np.asarray(outputs)[None, :]
        mask[:, : len(cells)] = True
    queries = np.arange(CELLS, dtype=np.int64)
    tensors = [
        torch.from_numpy(padded_cells).to(device),
        torch.from_numpy(padded_outputs).to(device),
        torch.from_numpy(mask).to(device),
        torch.from_numpy(queries).to(device),
    ]
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
        logits = model(*tensors)
    return logits.float().softmax(dim=-1).cpu().numpy()


def entropy_from_probabilities(probability: np.ndarray) -> float:
    positive = probability[probability > 0]
    return float(-np.sum(positive * np.log2(positive)))


def choose_weighted_query(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    unqueried: np.ndarray,
    generator: np.random.Generator,
    query_number: int,
    dual: bool = False,
) -> int:
    available = np.flatnonzero(unqueried)
    if dual and query_number > 0 and query_number % cfg.audit_period == 0:
        return int(generator.choice(available))
    entropies = []
    for cell in available:
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        entropies.append(entropy_from_probabilities(probability))
    entropies = np.asarray(entropies)
    best = available[np.isclose(entropies, entropies.max(), atol=1e-12)]
    return int(generator.choice(best))


def proposal_decode(
    truth_table: np.ndarray,
    neural_table: np.ndarray,
    grammar_predictions: np.ndarray,
    grammar_families: np.ndarray,
    initial_cells: np.ndarray,
    seed: int,
) -> Dict[str, object]:
    truth = truth_table.reshape(-1).astype(np.int16)
    distance = np.sum(grammar_predictions != neural_table[None, :], axis=1)
    top_k = min(cfg.proposal_top_k, len(distance))
    proposal = np.argpartition(distance, top_k - 1)[:top_k].astype(np.int32)
    truth_in_proposal = bool(np.any(np.all(grammar_predictions[proposal] == truth[None, :], axis=1)))
    observed_cells = [int(cell) for cell in initial_cells]
    observed_outputs = [int(truth[cell]) for cell in observed_cells]
    candidates = proposal
    for cell, output in zip(observed_cells, observed_outputs):
        candidates = candidates[grammar_predictions[candidates, cell] == output]
    unqueried = np.ones(CELLS, dtype=bool)
    unqueried[initial_cells] = False
    generator = np.random.default_rng(seed)
    extra = 0
    rejected = len(candidates) == 0
    while extra < cfg.extraction_extra_queries and len(candidates) > 0:
        unique_tables = np.unique(grammar_predictions[candidates], axis=0)
        if len(unique_tables) == 1:
            break
        weights = np.ones(len(candidates), dtype=np.float64) / len(candidates)
        cell = choose_weighted_query(
            candidates,
            grammar_predictions,
            weights,
            unqueried,
            generator,
            extra,
            dual=True,
        )
        unqueried[cell] = False
        observed_cells.append(cell)
        observed_outputs.append(int(truth[cell]))
        candidates = candidates[grammar_predictions[candidates, cell] == truth[cell]]
        extra += 1
        rejected = len(candidates) == 0
    if len(candidates):
        best = int(candidates[np.argmin(distance[candidates])])
        selected_table = grammar_predictions[best]
        selected_family = int(grammar_families[best])
        resolved = bool(np.all(grammar_predictions[candidates] == selected_table[None, :]))
        exact = bool(np.array_equal(selected_table, truth))
    else:
        selected_family = -1
        resolved = False
        exact = False
    return {
        "neural_accuracy": float(np.mean(neural_table == truth)),
        "truth_in_proposal": truth_in_proposal,
        "extra_queries": extra,
        "total_observations": len(initial_cells) + extra,
        "remaining_candidates": len(candidates),
        "resolved": resolved,
        "exact": exact,
        "selected_family": selected_family,
        "rejected": rejected,
    }


def posterior_metrics(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    truth: np.ndarray,
) -> Dict[str, float]:
    if len(candidates) == 0:
        return {
            "transition_accuracy": float("nan"),
            "posterior_nll": float("nan"),
            "resolved_fraction": float("nan"),
            "exact_table": 0.0,
            "candidates": 0.0,
        }
    top_correct = 0
    resolved = 0
    correct_probability = []
    for cell in range(CELLS):
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        top_correct += int(np.argmax(probability) == truth[cell])
        resolved += int(np.count_nonzero(probability > 1e-12) == 1)
        correct_probability.append(probability[int(truth[cell])])
    return {
        "transition_accuracy": top_correct / CELLS,
        "posterior_nll": float(-np.mean(np.log(np.clip(correct_probability, 1e-15, 1.0)))),
        "resolved_fraction": resolved / CELLS,
        "exact_table": float(resolved == CELLS),
        "candidates": float(len(candidates)),
    }


def neural_weights(
    candidates: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    beta: float,
) -> np.ndarray:
    if beta == 0.0:
        return np.ones(len(candidates), dtype=np.float64) / len(candidates)
    log_probability = np.log(np.clip(probabilities, 1e-9, 1.0))
    scores = log_probability[np.arange(CELLS)[None, :], predictions[candidates]].sum(axis=1)
    scores = beta * (scores - scores.max())
    weights = np.exp(np.clip(scores, -80.0, 0.0))
    return weights / weights.sum()


def transfer_trace(
    truth_table: np.ndarray,
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    beta: float,
    seed: int,
    maximum_queries: int = 12,
    dual: bool = False,
) -> Tuple[List[Dict[str, float]], Optional[int]]:
    truth = truth_table.reshape(-1).astype(np.int16)
    candidates = np.arange(len(predictions), dtype=np.int32)
    unqueried = np.ones(CELLS, dtype=bool)
    context_cells: List[int] = []
    context_outputs: List[int] = []
    generator = np.random.default_rng(seed)
    rows: List[Dict[str, float]] = []
    rejected_at: Optional[int] = None
    for query_count in range(maximum_queries + 1):
        neural_probability = neural_full_probabilities(model, context_cells, context_outputs)
        if len(candidates):
            weights = neural_weights(candidates, predictions, neural_probability, beta)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        else:
            weights = np.empty(0)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        neural_prediction = neural_probability.argmax(axis=1)
        rows.append(
            {
                "queries": float(query_count),
                **symbolic,
                "neural_transition_accuracy": float(np.mean(neural_prediction == truth)),
                "neural_exact_table": float(np.array_equal(neural_prediction, truth)),
            }
        )
        if query_count == maximum_queries or len(candidates) == 0:
            break
        cell = choose_weighted_query(
            candidates,
            predictions,
            weights,
            unqueried,
            generator,
            query_count,
            dual=dual,
        )
        unqueried[cell] = False
        context_cells.append(cell)
        context_outputs.append(int(truth[cell]))
        candidates = candidates[predictions[candidates, cell] == truth[cell]]
        if len(candidates) == 0:
            rejected_at = query_count + 1
    return rows, rejected_at


def first_exact(trace: Sequence[Dict[str, float]]) -> int:
    hits = [int(row["queries"]) for row in trace if row["exact_table"] == 1.0]
    return min(hits) if hits else CELLS + 1


def metric_at(trace: Sequence[Dict[str, float]], budget: int, key: str) -> float:
    eligible = [row for row in trace if int(row["queries"]) <= budget]
    return float(eligible[-1][key])


def evaluate_beta(
    beta: float,
    tasks: Sequence[np.ndarray],
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    seed_offset: int,
) -> Dict[str, float]:
    traces = [
        transfer_trace(task, predictions, model, beta, cfg.seed + seed_offset + index)[0]
        for index, task in enumerate(tasks)
    ]
    return {
        "beta": beta,
        "exact_at_8": float(np.mean([metric_at(trace, 8, "exact_table") for trace in traces])),
        "accuracy_at_4": float(np.mean([metric_at(trace, 4, "transition_accuracy") for trace in traces])),
        "median_exact_queries": float(np.median([first_exact(trace) for trace in traces])),
    }


def write_csv(path: Path, rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


print("=" * 118)
print("NEURAL-QUOTIENT ACTIVE LEARNER V13 — decode, quotient, transfer, reorganize")
print("=" * 118)
print(
    f"device={device} | GPU={torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'} | "
    f"states={N} | actions={A} | cells={CELLS} | permutations={N_PERMUTATIONS:,} | fast={FAST_DEV_RUN}"
)
started = time.perf_counter()

excluded: set[bytes] = set()
grammar = [generate_machine(excluded) for _ in range(cfg.candidate_templates)]
recurring = grammar[: cfg.recurring_templates]
grammar_predictions, grammar_families = hypothesis_predictions(grammar)
recurring_relabelings = np.stack([all_relabelings(table) for table in recurring]).astype(np.int8)
splits = permutation_splits()
print(
    f"candidate grammar={len(grammar)} canonical laws | hypotheses={len(grammar_predictions):,} | "
    f"recurring latent laws={len(recurring)}"
)

model, training_history = train_neural(recurring_relabelings, splits)

# Crystallize recurring rules from imperfect neural tables plus actively requested evidence.
prior_rows: List[Dict[str, object]] = []
shuffled_rows: List[Dict[str, object]] = []
naive_keys: set[bytes] = set()
decoded_family_counts: Dict[int, int] = {}
prior_pool = splits["prior"]
prior_cursor = 0
for family in range(cfg.recurring_templates):
    for task_in_family in range(cfg.prior_tasks_per_family):
        permutation_index = int(prior_pool[prior_cursor % len(prior_pool)])
        prior_cursor += 1
        truth = relabel_table(recurring[family], permutation_index)
        flat_truth = truth.reshape(-1)
        cell_order = np.random.default_rng(cfg.seed + 10_000 + prior_cursor).permutation(CELLS)
        initial_cells = cell_order[: cfg.initial_context]
        initial_outputs = flat_truth[initial_cells]
        probability = neural_full_probabilities(model, initial_cells, initial_outputs)
        neural_table = probability.argmax(axis=1).astype(np.int16)
        naive_canonical, _ = canonicalize(neural_table.reshape(N, A))
        naive_keys.add(naive_canonical.astype(np.int8).tobytes())
        decoded = proposal_decode(
            truth,
            neural_table,
            grammar_predictions,
            grammar_families,
            initial_cells,
            cfg.seed + 20_000 + prior_cursor,
        )
        prior_rows.append({"family": family, "task": task_in_family, **decoded})
        if decoded["resolved"] and not decoded["rejected"]:
            selected = int(decoded["selected_family"])
            decoded_family_counts[selected] = decoded_family_counts.get(selected, 0) + 1

        shuffled = neural_table[np.random.default_rng(cfg.seed + 30_000 + prior_cursor).permutation(CELLS)]
        shuffled_decoded = proposal_decode(
            truth,
            shuffled,
            grammar_predictions,
            grammar_families,
            initial_cells,
            cfg.seed + 40_000 + prior_cursor,
        )
        shuffled_rows.append({"family": family, "task": task_in_family, **shuffled_decoded})

decoded_ids = sorted(
    family for family, count in decoded_family_counts.items() if count >= cfg.recurrence_threshold
)
decoded_library = [grammar[family] for family in decoded_ids]
clean_library = list(recurring)
expected_keys = {table.astype(np.int8).tobytes() for table in recurring}
decoded_keys = {table.astype(np.int8).tobytes() for table in decoded_library}
library_recovery = len(expected_keys & decoded_keys) / len(expected_keys)
prior_neural_accuracy = float(np.mean([row["neural_accuracy"] for row in prior_rows]))
prior_exact = float(np.mean([row["exact"] for row in prior_rows]))
prior_resolved = float(np.mean([row["resolved"] for row in prior_rows]))
prior_median_observations = float(
    np.median([row["total_observations"] for row in prior_rows if row["exact"]])
)
shuffled_exact = float(np.mean([row["exact"] for row in shuffled_rows]))
print("\nCRYSTALLIZATION FROM THE IMPERFECT NEURAL CHANNEL")
print("-" * 118)
print(
    f"neural table accuracy={100*prior_neural_accuracy:.2f}% | active exact={100*prior_exact:.1f}% | "
    f"resolved={100*prior_resolved:.1f}% | median total observations={prior_median_observations:.1f}/{CELLS}"
)
print(
    f"naive noisy canonical templates={len(naive_keys)} | recurring decoded templates={len(decoded_library)} | "
    f"true-template recovery={100*library_recovery:.1f}% | shuffled-control exact={100*shuffled_exact:.1f}%"
)
if not decoded_library:
    raise RuntimeError("No recurring rules survived active crystallization")

decoded_predictions, _ = hypothesis_predictions(decoded_library)
clean_predictions, _ = hypothesis_predictions(clean_library)

# Calibrate neural responsibility on mappings disjoint from training, crystallization, and test.
gate_tasks: List[np.ndarray] = []
gate_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.gate_tasks_per_family):
        gate_tasks.append(relabel_table(recurring[family], int(splits["gate"][gate_cursor])))
        gate_cursor += 1
gate_rows = [
    evaluate_beta(beta, gate_tasks, decoded_predictions, model, 100_000 + int(10_000 * beta))
    for beta in cfg.gate_betas
]
symbolic_gate = next(row for row in gate_rows if row["beta"] == 0.0)
safe_gate_rows = [
    row
    for row in gate_rows
    if row["exact_at_8"] + 1e-12 >= symbolic_gate["exact_at_8"]
    and row["accuracy_at_4"] + 1e-12 >= symbolic_gate["accuracy_at_4"]
]
selected_gate = max(
    safe_gate_rows,
    key=lambda row: (row["exact_at_8"], -row["median_exact_queries"], row["accuracy_at_4"], -row["beta"]),
)
selected_beta = float(selected_gate["beta"])
print("\nHELD-OUT GATE CALIBRATION")
print("-" * 118)
for row in gate_rows:
    print(
        f"beta={row['beta']:.2f} | exact@8={100*row['exact_at_8']:5.1f}% | "
        f"accuracy@4={100*row['accuracy_at_4']:5.1f}% | median exact={row['median_exact_queries']:.1f}"
    )
print(f"selected no-harm beta={selected_beta:.2f}")

# Untouched permutation transfer.
test_tasks: List[Tuple[int, np.ndarray]] = []
test_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.test_tasks_per_family):
        test_tasks.append((family, relabel_table(recurring[family], int(splits["test"][test_cursor]))))
        test_cursor += 1

transfer_rows: List[Dict[str, object]] = []
exact_queries: Dict[str, List[int]] = {"symbolic": [], "calibrated-hybrid": [], "clean-oracle": []}
for task_id, (family, truth) in enumerate(test_tasks):
    for method, predictions, beta in (
        ("symbolic", decoded_predictions, 0.0),
        ("calibrated-hybrid", decoded_predictions, selected_beta),
        ("clean-oracle", clean_predictions, 0.0),
    ):
        trace, rejected = transfer_trace(
            truth,
            predictions,
            model,
            beta,
            cfg.seed + 200_000 + 101 * task_id + {"symbolic": 1, "calibrated-hybrid": 2, "clean-oracle": 3}[method],
        )
        if rejected is not None:
            raise RuntimeError(f"Known-family task rejected by {method}")
        exact_queries[method].append(first_exact(trace))
        for row in trace:
            if int(row["queries"]) in cfg.report_budgets:
                transfer_rows.append(
                    {"method": method, "task": task_id, "family": family, **row}
                )


def transfer_summary(method: str, budget: int, key: str) -> float:
    rows = [
        row
        for row in transfer_rows
        if row["method"] == method and int(row["queries"]) == budget
    ]
    return float(np.nanmean([float(row[key]) for row in rows]))


print("\nUNTOUCHED-REPRESENTATION TRANSFER")
print("-" * 118)
for method in exact_queries:
    print(
        f"{method:18s} | median exact queries={np.median(exact_queries[method]):4.1f} | "
        f"exact@8={100*transfer_summary(method, 8, 'exact_table'):5.1f}% | "
        f"table accuracy@4={100*transfer_summary(method, 4, 'transition_accuracy'):5.1f}%"
    )
neural_accuracy_at_8 = transfer_summary("symbolic", 8, "neural_transition_accuracy")
neural_exact_at_8 = transfer_summary("symbolic", 8, "neural_exact_table")
print(
    f"direct imperfect neural @8 observations | table accuracy={100*neural_accuracy_at_8:.1f}% | "
    f"exact tables={100*neural_exact_at_8:.1f}% | local tabular exact@8=0.0%"
)

# Novel-law rejection, MDL promotion, and future reuse.
novel = mutate_machine(recurring[0], cfg.novel_changes, excluded)
novel_tables = [
    relabel_table(novel, int(splits["test"][(test_cursor + index) % len(splits["test"])]))
    for index in range(cfg.novel_tasks)
]
detection_queries: List[int] = []
for task_id, truth in enumerate(novel_tables):
    _, rejected = transfer_trace(
        truth,
        decoded_predictions,
        model,
        0.0,
        cfg.seed + 400_000 + task_id,
        maximum_queries=12,
        dual=True,
    )
    detection_queries.append(CELLS + 1 if rejected is None else rejected)

nearest_mismatches = int(
    min(np.min(np.sum(predictions != novel_tables[0].reshape(1, -1), axis=1)) for predictions in [decoded_predictions])
)
permutation_bits = math.log2(math.factorial(N))
existing_task_bits = (
    math.log2(len(decoded_library))
    + permutation_bits
    + math.log2(math.comb(CELLS, nearest_mismatches))
    + nearest_mismatches * math.log2(N - 1)
)
definition_bits = CELLS * math.log2(N)
new_task_bits = math.log2(len(decoded_library) + 1) + permutation_bits
promotion_at: Optional[int] = None
promotion_rows: List[Dict[str, object]] = []
for repeats in range(1, cfg.novel_tasks + 1):
    exceptions_bits = repeats * existing_task_bits
    promotion_bits = definition_bits + repeats * new_task_bits
    if promotion_at is None and promotion_bits < exceptions_bits:
        promotion_at = repeats
    promotion_rows.append(
        {
            "repeated_worlds": repeats,
            "exceptions_bits": exceptions_bits,
            "promotion_bits": promotion_bits,
        }
    )

expanded_predictions, _ = hypothesis_predictions(decoded_library + [novel])
future_exact = []
for task_id, truth in enumerate(novel_tables):
    trace, rejected = transfer_trace(
        truth,
        expanded_predictions,
        model,
        0.0,
        cfg.seed + 500_000 + task_id,
    )
    if rejected is not None:
        raise RuntimeError("Promoted family was rejected")
    future_exact.append(first_exact(trace))
detected_by_12 = float(np.mean(np.asarray(detection_queries) <= 12))
future_median = float(np.median(future_exact))
print("\nNOVEL LAW AND REORGANIZATION")
print("-" * 118)
print(
    f"detected by 12={100*detected_by_12:.1f}% | median detection={np.median(detection_queries):.1f} | "
    f"nearest residual={nearest_mismatches} transitions | promote after={promotion_at} recurring worlds | "
    f"future median exact={future_median:.1f} queries"
)

predictions_declared = {
    "P1_actual_neural_channel_is_imperfect_45_to_60pct": 0.45 <= prior_neural_accuracy <= 0.60,
    "P2_active_structural_decoder_recovers_90pct_with_fewer_than_24_observations": (
        prior_exact >= 0.90 and prior_median_observations <= 16
    ),
    "P3_noisy_active_crystallization_recovers_all_recurring_templates": library_recovery == 1.0,
    "P4_structure_not_cell_order_drives_decoding": prior_exact - shuffled_exact >= 0.50,
    "P5_unseen_representation_exact_at_8_at_least_90pct": transfer_summary(
        "symbolic", 8, "exact_table"
    )
    >= 0.90,
    "P6_calibrated_gate_no_harm_on_untouched_test": (
        transfer_summary("calibrated-hybrid", 8, "exact_table") + 1e-12
        >= transfer_summary("symbolic", 8, "exact_table")
    ),
    "P7_symbolic_transfer_beats_direct_neural_and_local_at_8": (
        transfer_summary("symbolic", 8, "exact_table") > neural_exact_at_8
        and transfer_summary("symbolic", 8, "exact_table") > 0.0
    ),
    "P8_novel_law_is_rejected_then_reusable_after_mdl_promotion": (
        detected_by_12 >= 0.90 and promotion_at is not None and promotion_at <= 8 and future_median < CELLS / 2
    ),
}
print("\nPREDECLARED PREDICTIONS")
print("-" * 118)
for name, passed in predictions_declared.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(out_dir / "training_history.csv", training_history)
write_csv(out_dir / "prior_crystallization.csv", prior_rows)
write_csv(out_dir / "shuffled_control.csv", shuffled_rows)
write_csv(out_dir / "gate_calibration.csv", gate_rows)
write_csv(out_dir / "transfer_trials.csv", transfer_rows)
write_csv(out_dir / "promotion.csv", promotion_rows)

result = {
    "config": asdict(cfg),
    "predictions": predictions_declared,
    "neural": {
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "prior_table_accuracy": prior_neural_accuracy,
    },
    "crystallization": {
        "prior_exact": prior_exact,
        "prior_resolved": prior_resolved,
        "median_total_observations": prior_median_observations,
        "naive_noisy_templates": len(naive_keys),
        "decoded_templates": len(decoded_library),
        "template_recovery": library_recovery,
        "shuffled_exact": shuffled_exact,
    },
    "gate": {"selected_beta": selected_beta, "calibration": gate_rows},
    "transfer": {
        method: {
            "median_exact_queries": float(np.median(values)),
            "exact_at_8": transfer_summary(method, 8, "exact_table"),
            "accuracy_at_4": transfer_summary(method, 4, "transition_accuracy"),
        }
        for method, values in exact_queries.items()
    },
    "direct_neural": {
        "accuracy_at_8": neural_accuracy_at_8,
        "exact_at_8": neural_exact_at_8,
    },
    "novelty": {
        "detected_by_12": detected_by_12,
        "median_detection_queries": float(np.median(detection_queries)),
        "promotion_at": promotion_at,
        "future_median_exact_queries": future_median,
    },
    "elapsed_seconds": time.perf_counter() - started,
}
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
(out_dir / "results.json").write_text(json.dumps(result, indent=2))
torch.save(model.state_dict(), out_dir / "imperfect_neural.pt")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
ax.plot(
    [row["step"] for row in training_history],
    [100 * row["calibration_accuracy"] for row in training_history],
    marker="o",
)
ax.axhline(100 * cfg.neural_target_accuracy, color="black", linestyle="--", label="target imperfect channel")
ax.set_title("Actual neural learner checkpoint")
ax.set_xlabel("optimizer step")
ax.set_ylabel("held-out transition accuracy (%)")
ax.grid(alpha=0.25)
ax.legend()

ax = axes[0, 1]
labels = ["neural table", "shuffled control", "active decode"]
values = [prior_neural_accuracy, shuffled_exact, prior_exact]
ax.bar(labels, [100 * value for value in values], color=["#4C78A8", "#E45756", "#54A24B"])
ax.set_ylabel("accuracy / exact recovery (%)")
ax.set_title("Noisy codeword becomes an exact rule")
ax.set_ylim(0, 105)

ax = axes[1, 0]
for method, color in (("symbolic", "#54A24B"), ("calibrated-hybrid", "#B279A2"), ("clean-oracle", "#F2CF5B")):
    budgets = []
    exact = []
    for budget in cfg.report_budgets:
        rows = [row for row in transfer_rows if row["method"] == method and int(row["queries"]) == budget]
        if rows:
            budgets.append(budget)
            exact.append(100 * np.mean([row["exact_table"] for row in rows]))
    ax.plot(budgets, exact, marker="o", label=method, color=color)
ax.axvline(8, color="black", linestyle=":")
ax.set_title("Exact rule on unseen state names")
ax.set_xlabel("actively observed transitions")
ax.set_ylabel("exact full table (%)")
ax.set_ylim(-2, 102)
ax.grid(alpha=0.25)
ax.legend()

ax = axes[1, 1]
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["exceptions_bits"] for row in promotion_rows],
    marker="o",
    label="keep as residuals",
    color="#E45756",
)
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["promotion_bits"] for row in promotion_rows],
    marker="o",
    label="promote rule",
    color="#54A24B",
)
if promotion_at is not None:
    ax.axvline(promotion_at, color="black", linestyle="--", label=f"promotion at {promotion_at}")
ax.set_title("Residual recurrence triggers reorganization")
ax.set_xlabel("repeated novel worlds")
ax.set_ylabel("description length (bits)")
ax.grid(alpha=0.25)
ax.legend()

fig.suptitle(
    "Neural-to-symbolic active learning: predict noisily, decode exactly, transfer invariantly",
    fontsize=15,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

shutil.make_archive(str(out_dir / "run_bundle"), "zip", root_dir=out_dir)
print("\n" + "=" * 118)
print("FINAL V13 SUMMARY")
print("=" * 118)
print(
    f"neural={100*prior_neural_accuracy:.1f}% local transitions -> active crystallization={100*prior_exact:.1f}% exact | "
    f"library recovery={100*library_recovery:.1f}%"
)
print(
    f"unseen representation exact@8={100*transfer_summary('symbolic', 8, 'exact_table'):.1f}% | "
    f"median exact={np.median(exact_queries['symbolic']):.1f}/24 queries | novel detected@12={100*detected_by_12:.1f}%"
)
print(f"elapsed={result['elapsed_seconds']:.1f}s | artifacts={out_dir} | bundle={out_dir / 'run_bundle.zip'}")


## Experiment 8: Neural-quotient V13.1 continuation

Original cell `6`.


In [ ]:
"""
NEURAL-QUOTIENT ACTIVE LEARNER V13.1 — SINGLE-CELL A100 EXPERIMENT

Breakthrough test
-----------------
Treat an imperfect neural transition table as a noisy codeword on a structured
program manifold.  Decode that codeword against a broad finite-machine grammar,
ask the environment only the maximally informative missing transitions, quotient
the verified rule by arbitrary state renaming, and retain recurring rules by MDL.

The decisive transfer test uses coordinate permutations never seen by the neural
learner.  The system must recover a complete 24-transition world from substantially
fewer than 24 observations, reject a genuinely novel law, and reorganize only when
the same residual law recurs.

What is learned and what is supplied
------------------------------------
Learned: a real in-context Transformer transition predictor; recurring canonical
templates; a held-out calibrated neural/symbolic responsibility gate.

Supplied: eight states, three actions, a finite grammar of candidate deterministic
machines, and the permission to query any state-action cell.  This experiment tests
error-correcting crystallization and invariant transfer, not autonomous ontology or
grammar discovery.
"""

from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


FAST_DEV_RUN = os.environ.get("NEURAL_QUOTIENT_V13_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("NEURAL_QUOTIENT_V13_SEED", "20260813"))
    n_states: int = 8
    n_actions: int = 3
    candidate_templates: int = 32
    recurring_templates: int = 4
    train_permutations: int = 30_000
    neural_calibration_permutations: int = 4_000
    prior_permutations: int = 2_000
    gate_permutations: int = 2_000
    prior_tasks_per_family: int = 6
    gate_tasks_per_family: int = 3
    test_tasks_per_family: int = 8
    initial_context: int = 8
    maximum_context: int = 8
    extraction_extra_queries: int = 8
    proposal_top_k: int = 16_384
    proposal_sweep: Tuple[int, ...] = (1_024, 4_096, 16_384, 65_536)
    recurrence_threshold: int = 3
    neural_target_accuracy: float = 0.53
    neural_target_tolerance: float = 0.025
    neural_steps: int = 5_000
    neural_batch: int = 1_024
    neural_eval_examples: int = 8_192
    neural_eval_every: int = 25
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    width: int = 128
    heads: int = 4
    layers: int = 5
    dropout: float = 0.0
    gate_betas: Tuple[float, ...] = (0.0, 0.10, 0.25, 0.50, 1.0)
    report_budgets: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6, 8, 10, 12)
    novel_tasks: int = 24
    novel_changes: int = 4
    audit_period: int = 3
    output_dir: str = os.environ.get(
        "NEURAL_QUOTIENT_V13_OUTPUT_DIR",
        "outputs/08-neural-quotient-v13-1-continuation/neural_quotient_active_v13_1_results"
        if Path("/content").exists()
        else "outputs/08-neural-quotient-v13-1-continuation/neural_quotient_active_v13_1_results",
    )


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        candidate_templates=8,
        train_permutations=24_000,
        neural_calibration_permutations=3_000,
        prior_permutations=1_000,
        gate_permutations=1_000,
        prior_tasks_per_family=2,
        gate_tasks_per_family=1,
        test_tasks_per_family=2,
        proposal_top_k=4_096,
        proposal_sweep=(1_024, 4_096),
        neural_steps=1_000,
        neural_batch=256,
        neural_eval_examples=1_024,
        neural_eval_every=10,
        width=64,
        layers=3,
        novel_tasks=8,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
rng = np.random.default_rng(cfg.seed)
N, A = cfg.n_states, cfg.n_actions
CELLS = N * A
PERMUTATIONS = np.asarray(list(itertools.permutations(range(N))), dtype=np.int16)
INVERSES = np.argsort(PERMUTATIONS, axis=1).astype(np.int16)
N_PERMUTATIONS = len(PERMUTATIONS)


def strongly_connected(table: np.ndarray) -> bool:
    for source in range(N):
        seen = {source}
        frontier = [source]
        while frontier:
            state = frontier.pop()
            for successor in table[state]:
                successor = int(successor)
                if successor not in seen:
                    seen.add(successor)
                    frontier.append(successor)
        if len(seen) != N:
            return False
    return True


def all_relabelings(table: np.ndarray) -> np.ndarray:
    old_successors = table[INVERSES]
    rows = np.arange(N_PERMUTATIONS)[:, None, None]
    return PERMUTATIONS[rows, old_successors]


def canonicalize(table: np.ndarray) -> Tuple[np.ndarray, int]:
    relabeled = all_relabelings(np.asarray(table, dtype=np.int16))
    flat = relabeled.reshape(N_PERMUTATIONS, CELLS)
    first = int(np.lexsort(flat[:, ::-1].T)[0])
    canonical = relabeled[first].copy()
    automorphisms = int(np.sum(np.all(flat == canonical.reshape(1, -1), axis=1)))
    return canonical, automorphisms


def relabel_table(table: np.ndarray, permutation_index: int) -> np.ndarray:
    old_to_new = PERMUTATIONS[int(permutation_index)]
    new_to_old = INVERSES[int(permutation_index)]
    return old_to_new[table[new_to_old]]


def generate_machine(excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = rng.integers(0, N, size=(N, A), dtype=np.int16)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a distinct asymmetric machine")


def mutate_machine(base: np.ndarray, changes: int, excluded: set[bytes]) -> np.ndarray:
    for _ in range(50_000):
        candidate = base.copy()
        for cell in rng.choice(CELLS, size=changes, replace=False):
            state, action = divmod(int(cell), A)
            old = int(candidate[state, action])
            draw = int(rng.integers(0, N - 1))
            candidate[state, action] = draw + (draw >= old)
        if not strongly_connected(candidate):
            continue
        canonical, automorphisms = canonicalize(candidate)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a novel machine")


def hypothesis_predictions(library: Sequence[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    predictions: List[np.ndarray] = []
    family_ids: List[np.ndarray] = []
    row_index = np.arange(N_PERMUTATIONS)[:, None, None]
    action_index = np.arange(A)[None, None, :]
    for family, table in enumerate(library):
        canonical_next = table[PERMUTATIONS[:, :, None], action_index]
        observed_next = INVERSES[row_index, canonical_next]
        predictions.append(observed_next.reshape(N_PERMUTATIONS, CELLS).astype(np.int8))
        family_ids.append(np.full(N_PERMUTATIONS, family, dtype=np.int16))
    return np.concatenate(predictions), np.concatenate(family_ids)


class EquivariantTransitionGNN(nn.Module):
    """Permutation-equivariant neural predictor over a partially observed transition graph."""

    def __init__(self) -> None:
        super().__init__()
        width = cfg.width
        self.action = nn.Embedding(A, width)
        self.base_node = nn.Parameter(torch.zeros(width))
        self.query_role = nn.Embedding(2, width)
        self.to_source = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.to_target = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.updates = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(3 * width, 2 * width),
                    nn.GELU(),
                    nn.Linear(2 * width, width),
                )
                for _ in range(cfg.layers)
            ]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(cfg.layers)])
        self.scorer = nn.Sequential(
            nn.Linear(4 * width + 1, 2 * width),
            nn.GELU(),
            nn.Linear(2 * width, width),
            nn.GELU(),
            nn.Linear(width, 1),
        )

    def forward(
        self,
        context_cells: torch.Tensor,
        context_outputs: torch.Tensor,
        context_mask: torch.Tensor,
        query_cells: torch.Tensor,
    ) -> torch.Tensor:
        batch, _ = context_cells.shape
        context_states = torch.div(context_cells, A, rounding_mode="floor")
        context_actions = context_cells % A
        query_states = torch.div(query_cells, A, rounding_mode="floor")
        query_actions = query_cells % A
        node_ids = torch.arange(N, device=context_cells.device)[None, :]
        roles = (node_ids == query_states[:, None]).long()
        nodes = self.base_node[None, None, :].expand(batch, N, -1) + self.query_role(roles)

        valid_batch, valid_slot = torch.nonzero(context_mask, as_tuple=True)
        source = context_states[valid_batch, valid_slot]
        target = context_outputs[valid_batch, valid_slot]
        action = context_actions[valid_batch, valid_slot]
        batch_offsets = valid_batch * N
        source_flat = batch_offsets + source
        target_flat = batch_offsets + target

        for layer in range(cfg.layers):
            flat = nodes.reshape(batch * N, cfg.width)
            action_features = self.action(action)
            source_messages = self.to_source[layer](
                torch.cat([flat[target_flat], action_features], dim=-1)
            ).float()
            target_messages = self.to_target[layer](
                torch.cat([flat[source_flat], action_features], dim=-1)
            ).float()
            aggregate_source = torch.zeros_like(flat)
            aggregate_target = torch.zeros_like(flat)
            aggregate_source.index_add_(0, source_flat, source_messages)
            aggregate_target.index_add_(0, target_flat, target_messages)
            degree_source = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            degree_target = torch.zeros((batch * N, 1), device=flat.device, dtype=flat.dtype)
            ones = torch.ones((len(source_flat), 1), device=flat.device, dtype=flat.dtype)
            degree_source.index_add_(0, source_flat, ones)
            degree_target.index_add_(0, target_flat, ones)
            aggregate_source /= degree_source.clamp_min(1.0)
            aggregate_target /= degree_target.clamp_min(1.0)
            update = self.updates[layer](
                torch.cat([flat, aggregate_source, aggregate_target], dim=-1)
            )
            nodes = self.norms[layer](flat + update).reshape(batch, N, cfg.width)

        source_features = nodes[torch.arange(batch, device=nodes.device), query_states]
        candidate_features = nodes
        action_features = self.action(query_actions)
        expanded_source = source_features[:, None, :].expand(-1, N, -1)
        expanded_action = action_features[:, None, :].expand(-1, N, -1)
        equality = (node_ids == query_states[:, None]).to(nodes.dtype)[..., None]
        features = torch.cat(
            [
                expanded_source,
                candidate_features,
                expanded_source * candidate_features,
                expanded_action,
                equality,
            ],
            dim=-1,
        )
        return self.scorer(features).squeeze(-1)


def permutation_splits() -> Dict[str, np.ndarray]:
    order = rng.permutation(N_PERMUTATIONS)
    cursor = 0
    result: Dict[str, np.ndarray] = {}
    for name, length in (
        ("train", cfg.train_permutations),
        ("neural_calibration", cfg.neural_calibration_permutations),
        ("prior", cfg.prior_permutations),
        ("gate", cfg.gate_permutations),
    ):
        result[name] = order[cursor : cursor + length]
        cursor += length
    result["test"] = order[cursor:]
    if len(result["test"]) < cfg.test_tasks_per_family + cfg.novel_tasks:
        raise RuntimeError("Permutation split leaves too few untouched test mappings")
    return result


def sample_neural_batch(
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    batch_size: int,
    fixed_context: Optional[int] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    family = torch.randint(0, cfg.recurring_templates, (batch_size,), device=device)
    selected_pool = torch.randint(0, len(pool_gpu), (batch_size,), device=device)
    permutation = pool_gpu[selected_pool]
    tables = relabelings_gpu[family, permutation].reshape(batch_size, CELLS)
    cell_order = torch.rand((batch_size, CELLS), device=device).argsort(dim=1)
    if fixed_context is None:
        lengths = torch.randint(0, cfg.maximum_context + 1, (batch_size,), device=device)
    else:
        lengths = torch.full((batch_size,), fixed_context, device=device, dtype=torch.long)
    context_cells = cell_order[:, : cfg.maximum_context]
    context_outputs = torch.gather(tables, 1, context_cells)
    mask = torch.arange(cfg.maximum_context, device=device)[None, :] < lengths[:, None]
    query_cells = torch.gather(cell_order, 1, lengths[:, None]).squeeze(1)
    targets = torch.gather(tables, 1, query_cells[:, None]).squeeze(1)
    return context_cells, context_outputs, mask, query_cells, targets


@torch.no_grad()
def neural_accuracy(
    model: nn.Module,
    relabelings_gpu: torch.Tensor,
    pool_gpu: torch.Tensor,
    examples: int,
) -> float:
    model.eval()
    correct = 0
    seen = 0
    while seen < examples:
        batch = min(1_024, examples - seen)
        values = sample_neural_batch(
            relabelings_gpu, pool_gpu, batch, fixed_context=cfg.initial_context
        )
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(*values[:-1])
        correct += int((logits.argmax(dim=-1) == values[-1]).sum().item())
        seen += batch
    return correct / examples


def train_neural(
    recurring_relabelings: np.ndarray, splits: Dict[str, np.ndarray]
) -> Tuple[EquivariantTransitionGNN, List[Dict[str, float]]]:
    model = EquivariantTransitionGNN().to(device)
    tables_gpu = torch.from_numpy(recurring_relabelings.astype(np.int64)).to(device)
    train_pool = torch.from_numpy(splits["train"].astype(np.int64)).to(device)
    calibration_pool = torch.from_numpy(splits["neural_calibration"].astype(np.int64)).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, fused=device.type == "cuda"
    )
    scaler_enabled = device.type == "cuda" and amp_dtype == torch.float16
    if hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler(device.type, enabled=scaler_enabled)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
    history: List[Dict[str, float]] = []
    best_distance = float("inf")
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_step = 0
    best_accuracy = 0.0
    for step in range(1, cfg.neural_steps + 1):
        model.train()
        batch = sample_neural_batch(
            tables_gpu, train_pool, cfg.neural_batch, fixed_context=cfg.initial_context
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            loss = F.cross_entropy(model(*batch[:-1]), batch[-1])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if step % cfg.neural_eval_every == 0 or step == 1:
            accuracy = neural_accuracy(
                model, tables_gpu, calibration_pool, cfg.neural_eval_examples
            )
            history.append({"step": step, "loss": float(loss.item()), "calibration_accuracy": accuracy})
            distance = abs(accuracy - cfg.neural_target_accuracy)
            if distance < best_distance:
                best_distance = distance
                best_step = step
                best_accuracy = accuracy
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            print(
                f"neural step={step:4d} | loss={loss.item():.4f} | "
                f"heldout-k{cfg.initial_context} accuracy={100*accuracy:5.1f}%"
            )
            if (
                step >= 4 * cfg.neural_eval_every
                and distance <= cfg.neural_target_tolerance
                and accuracy >= cfg.neural_target_accuracy
            ):
                break
    if best_state is None:
        raise RuntimeError("No neural checkpoint was evaluated")
    model.load_state_dict(best_state)
    model.eval()
    print(f"restored imperfect-neural checkpoint step={best_step} | accuracy={100*best_accuracy:.2f}%")
    return model, history


@torch.no_grad()
def neural_full_probabilities(
    model: nn.Module, context_cells: Sequence[int], context_outputs: Sequence[int]
) -> np.ndarray:
    cells = list(context_cells)[-cfg.maximum_context :]
    outputs = list(context_outputs)[-cfg.maximum_context :]
    width = cfg.maximum_context
    padded_cells = np.zeros((CELLS, width), dtype=np.int64)
    padded_outputs = np.zeros((CELLS, width), dtype=np.int64)
    mask = np.zeros((CELLS, width), dtype=bool)
    if cells:
        padded_cells[:, : len(cells)] = np.asarray(cells)[None, :]
        padded_outputs[:, : len(outputs)] = np.asarray(outputs)[None, :]
        mask[:, : len(cells)] = True
    queries = np.arange(CELLS, dtype=np.int64)
    tensors = [
        torch.from_numpy(padded_cells).to(device),
        torch.from_numpy(padded_outputs).to(device),
        torch.from_numpy(mask).to(device),
        torch.from_numpy(queries).to(device),
    ]
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
        logits = model(*tensors)
    return logits.float().softmax(dim=-1).cpu().numpy()


def entropy_from_probabilities(probability: np.ndarray) -> float:
    positive = probability[probability > 0]
    return float(-np.sum(positive * np.log2(positive)))


def choose_weighted_query(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    unqueried: np.ndarray,
    generator: np.random.Generator,
    query_number: int,
    dual: bool = False,
) -> int:
    available = np.flatnonzero(unqueried)
    if dual and query_number > 0 and query_number % cfg.audit_period == 0:
        return int(generator.choice(available))
    entropies = []
    for cell in available:
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        entropies.append(entropy_from_probabilities(probability))
    entropies = np.asarray(entropies)
    best = available[np.isclose(entropies, entropies.max(), atol=1e-12)]
    return int(generator.choice(best))


def proposal_decode(
    truth_table: np.ndarray,
    neural_table: np.ndarray,
    grammar_predictions: np.ndarray,
    grammar_families: np.ndarray,
    initial_cells: np.ndarray,
    seed: int,
    top_k_override: Optional[int] = None,
) -> Dict[str, object]:
    truth = truth_table.reshape(-1).astype(np.int16)
    distance = np.sum(grammar_predictions != neural_table[None, :], axis=1)
    top_k = min(top_k_override or cfg.proposal_top_k, len(distance))
    proposal = np.argpartition(distance, top_k - 1)[:top_k].astype(np.int32)
    truth_in_proposal = bool(np.any(np.all(grammar_predictions[proposal] == truth[None, :], axis=1)))
    observed_cells = [int(cell) for cell in initial_cells]
    observed_outputs = [int(truth[cell]) for cell in observed_cells]
    candidates = proposal
    for cell, output in zip(observed_cells, observed_outputs):
        candidates = candidates[grammar_predictions[candidates, cell] == output]
    unqueried = np.ones(CELLS, dtype=bool)
    unqueried[initial_cells] = False
    generator = np.random.default_rng(seed)
    extra = 0
    rejected = len(candidates) == 0
    while extra < cfg.extraction_extra_queries and len(candidates) > 0:
        unique_tables = np.unique(grammar_predictions[candidates], axis=0)
        if len(unique_tables) == 1:
            break
        weights = np.ones(len(candidates), dtype=np.float64) / len(candidates)
        cell = choose_weighted_query(
            candidates,
            grammar_predictions,
            weights,
            unqueried,
            generator,
            extra,
            dual=True,
        )
        unqueried[cell] = False
        observed_cells.append(cell)
        observed_outputs.append(int(truth[cell]))
        candidates = candidates[grammar_predictions[candidates, cell] == truth[cell]]
        extra += 1
        rejected = len(candidates) == 0
    if len(candidates):
        best = int(candidates[np.argmin(distance[candidates])])
        selected_table = grammar_predictions[best]
        selected_family = int(grammar_families[best])
        resolved = bool(np.all(grammar_predictions[candidates] == selected_table[None, :]))
        exact = bool(np.array_equal(selected_table, truth))
    else:
        selected_family = -1
        resolved = False
        exact = False
    return {
        "neural_accuracy": float(np.mean(neural_table == truth)),
        "proposal_top_k": top_k,
        "truth_in_proposal": truth_in_proposal,
        "extra_queries": extra,
        "total_observations": len(initial_cells) + extra,
        "remaining_candidates": len(candidates),
        "resolved": resolved,
        "exact": exact,
        "selected_family": selected_family,
        "rejected": rejected,
    }


def posterior_metrics(
    candidates: np.ndarray,
    predictions: np.ndarray,
    weights: np.ndarray,
    truth: np.ndarray,
) -> Dict[str, float]:
    if len(candidates) == 0:
        return {
            "transition_accuracy": float("nan"),
            "posterior_nll": float("nan"),
            "resolved_fraction": float("nan"),
            "exact_table": 0.0,
            "candidates": 0.0,
        }
    top_correct = 0
    resolved = 0
    correct_probability = []
    for cell in range(CELLS):
        probability = np.bincount(
            predictions[candidates, cell], weights=weights, minlength=N
        ).astype(np.float64)
        probability /= probability.sum()
        top_correct += int(np.argmax(probability) == truth[cell])
        resolved += int(np.count_nonzero(probability > 1e-12) == 1)
        correct_probability.append(probability[int(truth[cell])])
    return {
        "transition_accuracy": top_correct / CELLS,
        "posterior_nll": float(-np.mean(np.log(np.clip(correct_probability, 1e-15, 1.0)))),
        "resolved_fraction": resolved / CELLS,
        "exact_table": float(resolved == CELLS),
        "candidates": float(len(candidates)),
    }


def neural_weights(
    candidates: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    beta: float,
) -> np.ndarray:
    if beta == 0.0:
        return np.ones(len(candidates), dtype=np.float64) / len(candidates)
    log_probability = np.log(np.clip(probabilities, 1e-9, 1.0))
    scores = log_probability[np.arange(CELLS)[None, :], predictions[candidates]].sum(axis=1)
    scores = beta * (scores - scores.max())
    weights = np.exp(np.clip(scores, -80.0, 0.0))
    return weights / weights.sum()


def transfer_trace(
    truth_table: np.ndarray,
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    beta: float,
    seed: int,
    maximum_queries: int = 12,
    dual: bool = False,
) -> Tuple[List[Dict[str, float]], Optional[int]]:
    truth = truth_table.reshape(-1).astype(np.int16)
    candidates = np.arange(len(predictions), dtype=np.int32)
    unqueried = np.ones(CELLS, dtype=bool)
    context_cells: List[int] = []
    context_outputs: List[int] = []
    generator = np.random.default_rng(seed)
    rows: List[Dict[str, float]] = []
    rejected_at: Optional[int] = None
    for query_count in range(maximum_queries + 1):
        neural_probability = neural_full_probabilities(model, context_cells, context_outputs)
        if len(candidates):
            weights = neural_weights(candidates, predictions, neural_probability, beta)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        else:
            weights = np.empty(0)
            symbolic = posterior_metrics(candidates, predictions, weights, truth)
        neural_prediction = neural_probability.argmax(axis=1)
        rows.append(
            {
                "queries": float(query_count),
                **symbolic,
                "neural_transition_accuracy": float(np.mean(neural_prediction == truth)),
                "neural_exact_table": float(np.array_equal(neural_prediction, truth)),
            }
        )
        if query_count == maximum_queries or len(candidates) == 0:
            break
        cell = choose_weighted_query(
            candidates,
            predictions,
            weights,
            unqueried,
            generator,
            query_count,
            dual=dual,
        )
        unqueried[cell] = False
        context_cells.append(cell)
        context_outputs.append(int(truth[cell]))
        candidates = candidates[predictions[candidates, cell] == truth[cell]]
        if len(candidates) == 0:
            rejected_at = query_count + 1
    return rows, rejected_at


def first_exact(trace: Sequence[Dict[str, float]]) -> int:
    hits = [int(row["queries"]) for row in trace if row["exact_table"] == 1.0]
    return min(hits) if hits else CELLS + 1


def metric_at(trace: Sequence[Dict[str, float]], budget: int, key: str) -> float:
    eligible = [row for row in trace if int(row["queries"]) <= budget]
    return float(eligible[-1][key])


def evaluate_beta(
    beta: float,
    tasks: Sequence[np.ndarray],
    predictions: np.ndarray,
    model: EquivariantTransitionGNN,
    seed_offset: int,
) -> Dict[str, float]:
    traces = [
        transfer_trace(task, predictions, model, beta, cfg.seed + seed_offset + index)[0]
        for index, task in enumerate(tasks)
    ]
    return {
        "beta": beta,
        "exact_at_8": float(np.mean([metric_at(trace, 8, "exact_table") for trace in traces])),
        "accuracy_at_4": float(np.mean([metric_at(trace, 4, "transition_accuracy") for trace in traces])),
        "median_exact_queries": float(np.median([first_exact(trace) for trace in traces])),
    }


def write_csv(path: Path, rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


print("=" * 118)
print("NEURAL-QUOTIENT ACTIVE LEARNER V13.1 — decode, quotient, transfer, reorganize")
print("=" * 118)
print(
    f"device={device} | GPU={torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'} | "
    f"states={N} | actions={A} | cells={CELLS} | permutations={N_PERMUTATIONS:,} | fast={FAST_DEV_RUN}"
)
started = time.perf_counter()

excluded: set[bytes] = set()
grammar = [generate_machine(excluded) for _ in range(cfg.candidate_templates)]
recurring = grammar[: cfg.recurring_templates]
grammar_predictions, grammar_families = hypothesis_predictions(grammar)
recurring_relabelings = np.stack([all_relabelings(table) for table in recurring]).astype(np.int8)
splits = permutation_splits()
print(
    f"candidate grammar={len(grammar)} canonical laws | hypotheses={len(grammar_predictions):,} | "
    f"recurring latent laws={len(recurring)}"
)

model, training_history = train_neural(recurring_relabelings, splits)

# Crystallize recurring rules from imperfect neural tables plus actively requested evidence.
prior_rows: List[Dict[str, object]] = []
shuffled_rows: List[Dict[str, object]] = []
proposal_sweep_rows: List[Dict[str, object]] = []
naive_keys: set[bytes] = set()
decoded_family_counts: Dict[int, int] = {}
prior_pool = splits["prior"]
prior_cursor = 0
for family in range(cfg.recurring_templates):
    for task_in_family in range(cfg.prior_tasks_per_family):
        permutation_index = int(prior_pool[prior_cursor % len(prior_pool)])
        prior_cursor += 1
        truth = relabel_table(recurring[family], permutation_index)
        flat_truth = truth.reshape(-1)
        cell_order = np.random.default_rng(cfg.seed + 10_000 + prior_cursor).permutation(CELLS)
        initial_cells = cell_order[: cfg.initial_context]
        initial_outputs = flat_truth[initial_cells]
        probability = neural_full_probabilities(model, initial_cells, initial_outputs)
        neural_table = probability.argmax(axis=1).astype(np.int16)
        naive_canonical, _ = canonicalize(neural_table.reshape(N, A))
        naive_keys.add(naive_canonical.astype(np.int8).tobytes())
        sweep_decodes: Dict[int, Dict[str, object]] = {}
        for proposal_k in cfg.proposal_sweep:
            sweep_decodes[proposal_k] = proposal_decode(
                truth,
                neural_table,
                grammar_predictions,
                grammar_families,
                initial_cells,
                cfg.seed + 20_000 + prior_cursor,
                top_k_override=proposal_k,
            )
            proposal_sweep_rows.append(
                {
                    "family": family,
                    "task": task_in_family,
                    **sweep_decodes[proposal_k],
                }
            )
        if cfg.proposal_top_k in sweep_decodes:
            decoded = sweep_decodes[cfg.proposal_top_k]
        else:
            decoded = proposal_decode(
                truth,
                neural_table,
                grammar_predictions,
                grammar_families,
                initial_cells,
                cfg.seed + 20_000 + prior_cursor,
            )
        prior_rows.append({"family": family, "task": task_in_family, **decoded})
        if decoded["resolved"] and not decoded["rejected"]:
            selected = int(decoded["selected_family"])
            decoded_family_counts[selected] = decoded_family_counts.get(selected, 0) + 1

        shuffled = neural_table[np.random.default_rng(cfg.seed + 30_000 + prior_cursor).permutation(CELLS)]
        shuffled_decoded = proposal_decode(
            truth,
            shuffled,
            grammar_predictions,
            grammar_families,
            initial_cells,
            cfg.seed + 40_000 + prior_cursor,
        )
        shuffled_rows.append({"family": family, "task": task_in_family, **shuffled_decoded})

decoded_ids = sorted(
    family for family, count in decoded_family_counts.items() if count >= cfg.recurrence_threshold
)
decoded_library = [grammar[family] for family in decoded_ids]
clean_library = list(recurring)
expected_keys = {table.astype(np.int8).tobytes() for table in recurring}
decoded_keys = {table.astype(np.int8).tobytes() for table in decoded_library}
library_recovery = len(expected_keys & decoded_keys) / len(expected_keys)
prior_neural_accuracy = float(np.mean([row["neural_accuracy"] for row in prior_rows]))
prior_exact = float(np.mean([row["exact"] for row in prior_rows]))
prior_resolved = float(np.mean([row["resolved"] for row in prior_rows]))
prior_median_observations = float(
    np.median([row["total_observations"] for row in prior_rows if row["exact"]])
)
shuffled_exact = float(np.mean([row["exact"] for row in shuffled_rows]))
print("\nCRYSTALLIZATION FROM THE IMPERFECT NEURAL CHANNEL")
print("-" * 118)
print(
    f"neural table accuracy={100*prior_neural_accuracy:.2f}% | active exact={100*prior_exact:.1f}% | "
    f"resolved={100*prior_resolved:.1f}% | median total observations={prior_median_observations:.1f}/{CELLS}"
)
for proposal_k in cfg.proposal_sweep:
    rows = [row for row in proposal_sweep_rows if int(row["proposal_top_k"]) == proposal_k]
    exact_rows = [row for row in rows if row["exact"]]
    print(
        f" proposal top-k={proposal_k:6,d} | truth recall={100*np.mean([row['truth_in_proposal'] for row in rows]):5.1f}% | "
        f"active exact={100*np.mean([row['exact'] for row in rows]):5.1f}% | "
        f"median observations={np.median([row['total_observations'] for row in exact_rows]) if exact_rows else float('nan'):.1f}"
    )
print(
    f"naive noisy canonical templates={len(naive_keys)} | recurring decoded templates={len(decoded_library)} | "
    f"true-template recovery={100*library_recovery:.1f}% | shuffled-control exact={100*shuffled_exact:.1f}%"
)
if not decoded_library:
    raise RuntimeError("No recurring rules survived active crystallization")

decoded_predictions, _ = hypothesis_predictions(decoded_library)
clean_predictions, _ = hypothesis_predictions(clean_library)

# Calibrate neural responsibility on mappings disjoint from training, crystallization, and test.
gate_tasks: List[np.ndarray] = []
gate_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.gate_tasks_per_family):
        gate_tasks.append(relabel_table(recurring[family], int(splits["gate"][gate_cursor])))
        gate_cursor += 1
gate_rows = [
    evaluate_beta(beta, gate_tasks, decoded_predictions, model, 100_000 + int(10_000 * beta))
    for beta in cfg.gate_betas
]
symbolic_gate = next(row for row in gate_rows if row["beta"] == 0.0)
safe_gate_rows = [
    row
    for row in gate_rows
    if row["exact_at_8"] + 1e-12 >= symbolic_gate["exact_at_8"]
    and row["accuracy_at_4"] + 1e-12 >= symbolic_gate["accuracy_at_4"]
]
selected_gate = max(
    safe_gate_rows,
    key=lambda row: (row["exact_at_8"], -row["median_exact_queries"], row["accuracy_at_4"], -row["beta"]),
)
selected_beta = float(selected_gate["beta"])
print("\nHELD-OUT GATE CALIBRATION")
print("-" * 118)
for row in gate_rows:
    print(
        f"beta={row['beta']:.2f} | exact@8={100*row['exact_at_8']:5.1f}% | "
        f"accuracy@4={100*row['accuracy_at_4']:5.1f}% | median exact={row['median_exact_queries']:.1f}"
    )
print(f"selected no-harm beta={selected_beta:.2f}")

# Untouched permutation transfer.
test_tasks: List[Tuple[int, np.ndarray]] = []
test_cursor = 0
for family in range(cfg.recurring_templates):
    for _ in range(cfg.test_tasks_per_family):
        test_tasks.append((family, relabel_table(recurring[family], int(splits["test"][test_cursor]))))
        test_cursor += 1

transfer_rows: List[Dict[str, object]] = []
exact_queries: Dict[str, List[int]] = {"symbolic": [], "calibrated-hybrid": [], "clean-oracle": []}
for task_id, (family, truth) in enumerate(test_tasks):
    for method, predictions, beta in (
        ("symbolic", decoded_predictions, 0.0),
        ("calibrated-hybrid", decoded_predictions, selected_beta),
        ("clean-oracle", clean_predictions, 0.0),
    ):
        trace, rejected = transfer_trace(
            truth,
            predictions,
            model,
            beta,
            cfg.seed + 200_000 + 101 * task_id + {"symbolic": 1, "calibrated-hybrid": 2, "clean-oracle": 3}[method],
        )
        if rejected is not None:
            raise RuntimeError(f"Known-family task rejected by {method}")
        exact_queries[method].append(first_exact(trace))
        for row in trace:
            if int(row["queries"]) in cfg.report_budgets:
                transfer_rows.append(
                    {"method": method, "task": task_id, "family": family, **row}
                )


def transfer_summary(method: str, budget: int, key: str) -> float:
    rows = [
        row
        for row in transfer_rows
        if row["method"] == method and int(row["queries"]) == budget
    ]
    return float(np.nanmean([float(row[key]) for row in rows]))


print("\nUNTOUCHED-REPRESENTATION TRANSFER")
print("-" * 118)
for method in exact_queries:
    print(
        f"{method:18s} | median exact queries={np.median(exact_queries[method]):4.1f} | "
        f"exact@8={100*transfer_summary(method, 8, 'exact_table'):5.1f}% | "
        f"table accuracy@4={100*transfer_summary(method, 4, 'transition_accuracy'):5.1f}%"
    )
neural_accuracy_at_8 = transfer_summary("symbolic", 8, "neural_transition_accuracy")
neural_exact_at_8 = transfer_summary("symbolic", 8, "neural_exact_table")
print(
    f"direct imperfect neural @8 observations | table accuracy={100*neural_accuracy_at_8:.1f}% | "
    f"exact tables={100*neural_exact_at_8:.1f}% | local tabular exact@8=0.0%"
)

# Novel-law rejection, MDL promotion, and future reuse.
novel = mutate_machine(recurring[0], cfg.novel_changes, excluded)
novel_tables = [
    relabel_table(novel, int(splits["test"][(test_cursor + index) % len(splits["test"])]))
    for index in range(cfg.novel_tasks)
]
detection_queries: List[int] = []
for task_id, truth in enumerate(novel_tables):
    _, rejected = transfer_trace(
        truth,
        decoded_predictions,
        model,
        0.0,
        cfg.seed + 400_000 + task_id,
        maximum_queries=12,
        dual=True,
    )
    detection_queries.append(CELLS + 1 if rejected is None else rejected)

nearest_mismatches = int(
    min(np.min(np.sum(predictions != novel_tables[0].reshape(1, -1), axis=1)) for predictions in [decoded_predictions])
)
permutation_bits = math.log2(math.factorial(N))
existing_task_bits = (
    math.log2(len(decoded_library))
    + permutation_bits
    + math.log2(math.comb(CELLS, nearest_mismatches))
    + nearest_mismatches * math.log2(N - 1)
)
definition_bits = CELLS * math.log2(N)
new_task_bits = math.log2(len(decoded_library) + 1) + permutation_bits
promotion_at: Optional[int] = None
promotion_rows: List[Dict[str, object]] = []
for repeats in range(1, cfg.novel_tasks + 1):
    exceptions_bits = repeats * existing_task_bits
    promotion_bits = definition_bits + repeats * new_task_bits
    if promotion_at is None and promotion_bits < exceptions_bits:
        promotion_at = repeats
    promotion_rows.append(
        {
            "repeated_worlds": repeats,
            "exceptions_bits": exceptions_bits,
            "promotion_bits": promotion_bits,
        }
    )

expanded_predictions, _ = hypothesis_predictions(decoded_library + [novel])
future_exact = []
for task_id, truth in enumerate(novel_tables):
    trace, rejected = transfer_trace(
        truth,
        expanded_predictions,
        model,
        0.0,
        cfg.seed + 500_000 + task_id,
    )
    if rejected is not None:
        raise RuntimeError("Promoted family was rejected")
    future_exact.append(first_exact(trace))
detected_by_12 = float(np.mean(np.asarray(detection_queries) <= 12))
future_median = float(np.median(future_exact))
print("\nNOVEL LAW AND REORGANIZATION")
print("-" * 118)
print(
    f"detected by 12={100*detected_by_12:.1f}% | median detection={np.median(detection_queries):.1f} | "
    f"nearest residual={nearest_mismatches} transitions | promote after={promotion_at} recurring worlds | "
    f"future median exact={future_median:.1f} queries"
)

predictions_declared = {
    "P1_actual_neural_channel_is_imperfect_45_to_60pct": 0.45 <= prior_neural_accuracy <= 0.60,
    "P2_active_structural_decoder_recovers_90pct_with_fewer_than_24_observations": (
        prior_exact >= 0.90 and prior_median_observations <= 16
    ),
    "P3_noisy_active_crystallization_recovers_all_recurring_templates": library_recovery == 1.0,
    "P4_structure_not_cell_order_drives_decoding": prior_exact - shuffled_exact >= 0.50,
    "P5_unseen_representation_exact_at_8_at_least_90pct": transfer_summary(
        "symbolic", 8, "exact_table"
    )
    >= 0.90,
    "P6_calibrated_gate_no_harm_on_untouched_test": (
        transfer_summary("calibrated-hybrid", 8, "exact_table") + 1e-12
        >= transfer_summary("symbolic", 8, "exact_table")
    ),
    "P7_symbolic_transfer_beats_direct_neural_and_local_at_8": (
        transfer_summary("symbolic", 8, "exact_table") > neural_exact_at_8
        and transfer_summary("symbolic", 8, "exact_table") > 0.0
    ),
    "P8_novel_law_is_rejected_then_reusable_after_mdl_promotion": (
        detected_by_12 >= 0.90 and promotion_at is not None and promotion_at <= 8 and future_median < CELLS / 2
    ),
}
print("\nPREDECLARED PREDICTIONS")
print("-" * 118)
for name, passed in predictions_declared.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(out_dir / "training_history.csv", training_history)
write_csv(out_dir / "prior_crystallization.csv", prior_rows)
write_csv(out_dir / "proposal_sweep.csv", proposal_sweep_rows)
write_csv(out_dir / "shuffled_control.csv", shuffled_rows)
write_csv(out_dir / "gate_calibration.csv", gate_rows)
write_csv(out_dir / "transfer_trials.csv", transfer_rows)
write_csv(out_dir / "promotion.csv", promotion_rows)

result = {
    "config": asdict(cfg),
    "predictions": predictions_declared,
    "neural": {
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "prior_table_accuracy": prior_neural_accuracy,
    },
    "crystallization": {
        "prior_exact": prior_exact,
        "prior_resolved": prior_resolved,
        "median_total_observations": prior_median_observations,
        "naive_noisy_templates": len(naive_keys),
        "decoded_templates": len(decoded_library),
        "template_recovery": library_recovery,
        "shuffled_exact": shuffled_exact,
        "proposal_sweep": [
            {
                "top_k": proposal_k,
                "truth_recall": float(
                    np.mean(
                        [
                            row["truth_in_proposal"]
                            for row in proposal_sweep_rows
                            if int(row["proposal_top_k"]) == proposal_k
                        ]
                    )
                ),
                "exact": float(
                    np.mean(
                        [
                            row["exact"]
                            for row in proposal_sweep_rows
                            if int(row["proposal_top_k"]) == proposal_k
                        ]
                    )
                ),
            }
            for proposal_k in cfg.proposal_sweep
        ],
    },
    "gate": {"selected_beta": selected_beta, "calibration": gate_rows},
    "transfer": {
        method: {
            "median_exact_queries": float(np.median(values)),
            "exact_at_8": transfer_summary(method, 8, "exact_table"),
            "accuracy_at_4": transfer_summary(method, 4, "transition_accuracy"),
        }
        for method, values in exact_queries.items()
    },
    "direct_neural": {
        "accuracy_at_8": neural_accuracy_at_8,
        "exact_at_8": neural_exact_at_8,
    },
    "novelty": {
        "detected_by_12": detected_by_12,
        "median_detection_queries": float(np.median(detection_queries)),
        "promotion_at": promotion_at,
        "future_median_exact_queries": future_median,
    },
    "elapsed_seconds": time.perf_counter() - started,
}
(out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
(out_dir / "results.json").write_text(json.dumps(result, indent=2))
torch.save(model.state_dict(), out_dir / "imperfect_neural.pt")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
ax.plot(
    [row["step"] for row in training_history],
    [100 * row["calibration_accuracy"] for row in training_history],
    marker="o",
)
ax.axhline(100 * cfg.neural_target_accuracy, color="black", linestyle="--", label="target imperfect channel")
ax.set_title("Actual neural learner checkpoint")
ax.set_xlabel("optimizer step")
ax.set_ylabel("held-out transition accuracy (%)")
ax.grid(alpha=0.25)
ax.legend()

ax = axes[0, 1]
labels = ["neural table", "shuffled control", "active decode"]
values = [prior_neural_accuracy, shuffled_exact, prior_exact]
ax.bar(labels, [100 * value for value in values], color=["#4C78A8", "#E45756", "#54A24B"])
ax.set_ylabel("accuracy / exact recovery (%)")
ax.set_title("Noisy codeword becomes an exact rule")
ax.set_ylim(0, 105)

ax = axes[1, 0]
for method, color in (("symbolic", "#54A24B"), ("calibrated-hybrid", "#B279A2"), ("clean-oracle", "#F2CF5B")):
    budgets = []
    exact = []
    for budget in cfg.report_budgets:
        rows = [row for row in transfer_rows if row["method"] == method and int(row["queries"]) == budget]
        if rows:
            budgets.append(budget)
            exact.append(100 * np.mean([row["exact_table"] for row in rows]))
    ax.plot(budgets, exact, marker="o", label=method, color=color)
ax.axvline(8, color="black", linestyle=":")
ax.set_title("Exact rule on unseen state names")
ax.set_xlabel("actively observed transitions")
ax.set_ylabel("exact full table (%)")
ax.set_ylim(-2, 102)
ax.grid(alpha=0.25)
ax.legend()

ax = axes[1, 1]
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["exceptions_bits"] for row in promotion_rows],
    marker="o",
    label="keep as residuals",
    color="#E45756",
)
ax.plot(
    [row["repeated_worlds"] for row in promotion_rows],
    [row["promotion_bits"] for row in promotion_rows],
    marker="o",
    label="promote rule",
    color="#54A24B",
)
if promotion_at is not None:
    ax.axvline(promotion_at, color="black", linestyle="--", label=f"promotion at {promotion_at}")
ax.set_title("Residual recurrence triggers reorganization")
ax.set_xlabel("repeated novel worlds")
ax.set_ylabel("description length (bits)")
ax.grid(alpha=0.25)
ax.legend()

fig.suptitle(
    "Neural-to-symbolic active learning: predict noisily, decode exactly, transfer invariantly",
    fontsize=15,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

bundle = shutil.make_archive(
    str(out_dir.parent / f"{out_dir.name}_bundle"), "zip", root_dir=out_dir
)
print("\n" + "=" * 118)
print("FINAL V13.1 SUMMARY")
print("=" * 118)
print(
    f"neural={100*prior_neural_accuracy:.1f}% local transitions -> active crystallization={100*prior_exact:.1f}% exact | "
    f"library recovery={100*library_recovery:.1f}%"
)
print(
    f"unseen representation exact@8={100*transfer_summary('symbolic', 8, 'exact_table'):.1f}% | "
    f"median exact={np.median(exact_queries['symbolic']):.1f}/24 queries | novel detected@12={100*detected_by_12:.1f}%"
)
print(f"elapsed={result['elapsed_seconds']:.1f}s | artifacts={out_dir} | bundle={bundle}")


## Experiment 9: Ontology-free active learner V14

Original cell `7`.


In [ ]:
"""
ONTOLOGY-FREE ACTIVE LEARNER V14 — SINGLE-CELL EXPERIMENT

Purpose
-------
Test the next assumption-removal step after V13.1.  The learner is not given
state labels, the number of states, action-effect labels, the number of action
effects, family labels, or a candidate grammar.  It receives opaque snapshots,
high-dimensional raw observations, raw motor commands, and reset/intervention
access to stored snapshots.

The experiment deliberately separates two bills:

  A. ontology acquisition: raw observations plus diagnostic interventions;
  B. rule transfer: transition queries after an ontology has been formed.

Therefore a conditional transfer saving cannot be mistaken for an end-to-end
sample-efficiency result.  Hidden variables are used only to generate worlds and
to audit ARI/accuracy; every learner decision is made from raw observations and
observed transitions.

This is a controlled bootstrap proof, not yet the final history-based POMDP
system.  The neural/probabilistic proposal is PCA + a BIC-selected mixture; a
symbolic behavioral pass merges visually split components when interventions
show identical action-conditional futures.  Rules begin as empirical graphs.
Recurring canonical graphs form the grammar by compression; no rule list exists
before experience.
"""

from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture


FAST_DEV_RUN = os.environ.get("ONTOLOGY_FREE_V14_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("ONTOLOGY_FREE_V14_SEED", "20260820"))
    hidden_states: int = 6                 # generator/audit only
    hidden_action_effects: int = 3         # generator/audit only
    raw_commands: int = 6
    recurring_families: int = 3
    prior_worlds_per_family: int = 4
    test_worlds_per_family: int = 8
    novel_recurrences: int = 3
    recurrence_threshold: int = 3
    signal_bits: int = 45
    nuisance_bits: int = 1500
    signal_flip_probability: float = 0.10
    snapshots_per_hidden_state: int = 120  # balanced generator distribution; labels hidden
    pca_dimensions: int = 14
    mixture_k_min: int = 2
    mixture_k_max: int = 9
    ontology_anchor_count: int = 10
    ontology_transition_repeats: int = 2
    transition_query_repeats: int = 3
    novel_changes: int = 4
    report_budgets: Tuple[int, ...] = (0, 2, 4, 6, 8, 10, 12, 16, 20, 24, 30, 36)
    output_dir: str = os.environ.get(
        "ONTOLOGY_FREE_V14_OUTPUT_DIR",
        "outputs/09-ontology-free-active-learner-v14/ontology_free_active_v14_results"
        if Path("/content").exists()
        else "outputs/09-ontology-free-active-learner-v14/ontology_free_active_v14_results",
    )


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        prior_worlds_per_family=2,
        test_worlds_per_family=2,
        recurrence_threshold=2,
        snapshots_per_hidden_state=55,
        signal_bits=55,
        nuisance_bits=600,
        pca_dimensions=12,
        ontology_anchor_count=5,
        ontology_transition_repeats=1,
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
rng = np.random.default_rng(cfg.seed)


def strongly_connected(table: np.ndarray) -> bool:
    n = table.shape[0]
    for source in range(n):
        seen, frontier = {source}, [source]
        while frontier:
            state = frontier.pop()
            for successor in table[state]:
                successor = int(successor)
                if successor not in seen:
                    seen.add(successor)
                    frontier.append(successor)
        if len(seen) != n:
            return False
    return True


def state_relabel(table: np.ndarray, old_to_new: np.ndarray) -> np.ndarray:
    inverse = np.argsort(old_to_new)
    return old_to_new[table[inverse]]


def make_asymmetric_machine(excluded: set[bytes]) -> np.ndarray:
    n, a = cfg.hidden_states, cfg.hidden_action_effects
    for _ in range(100_000):
        table = rng.integers(0, n, size=(n, a), dtype=np.int16)
        if not strongly_connected(table):
            continue
        canonical, automorphisms = canonicalize_graph(table)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a distinct asymmetric graph")


def mutate_machine(base: np.ndarray, changes: int, excluded: set[bytes]) -> np.ndarray:
    n, a = base.shape
    for _ in range(100_000):
        table = base.copy()
        for cell in rng.choice(n * a, size=changes, replace=False):
            state, action = divmod(int(cell), a)
            old = int(table[state, action])
            draw = int(rng.integers(0, n - 1))
            table[state, action] = draw + (draw >= old)
        if not strongly_connected(table):
            continue
        canonical, automorphisms = canonicalize_graph(table)
        key = canonical.astype(np.int8).tobytes()
        if automorphisms == 1 and key not in excluded:
            excluded.add(key)
            return canonical
    raise RuntimeError("Could not generate a distinct novel graph")


class RawWorld:
    """Hidden generator with an intentionally narrow raw learner interface."""

    def __init__(self, canonical_table: np.ndarray, seed: int):
        local = np.random.default_rng(seed)
        n, a = canonical_table.shape
        state_perm = local.permutation(n)
        action_perm = local.permutation(a)
        self.table = state_relabel(canonical_table[:, action_perm], state_perm)
        duplicated = np.repeat(np.arange(a, dtype=np.int16), cfg.raw_commands // a)
        self.command_effect = local.permutation(duplicated)
        self.signal = local.integers(0, 2, size=(n, cfg.signal_bits), dtype=np.int8)
        self.local = local
        self.handles: List[int] = []  # opaque to the learner; values never exposed

    def observe_state(self, hidden_state: int) -> np.ndarray:
        stable = self.signal[int(hidden_state)].copy()
        flips = self.local.random(cfg.signal_bits) < cfg.signal_flip_probability
        stable ^= flips.astype(np.int8)
        nuisance = self.local.integers(0, 2, size=cfg.nuisance_bits, dtype=np.int8)
        return np.concatenate([stable, nuisance]).astype(np.float32)

    def collect_snapshots(self) -> Tuple[List[int], np.ndarray]:
        # The reset distribution is balanced by the environment.  The learner sees
        # opaque handles and observations, never the hidden integer attached to one.
        hidden = np.repeat(np.arange(self.table.shape[0]), cfg.snapshots_per_hidden_state)
        self.local.shuffle(hidden)
        handles, observations = [], []
        for state in hidden:
            self.handles.append(int(state))
            handles.append(len(self.handles) - 1)
            observations.append(self.observe_state(int(state)))
        return handles, np.asarray(observations)

    def intervene(self, handle: int, raw_command: int) -> np.ndarray:
        source = self.handles[int(handle)]
        effect = int(self.command_effect[int(raw_command)])
        successor = int(self.table[source, effect])
        return self.observe_state(successor)

    def intervene_sequence(self, handle: int, raw_commands: Sequence[int]) -> np.ndarray:
        state = self.handles[int(handle)]
        for raw_command in raw_commands:
            effect = int(self.command_effect[int(raw_command)])
            state = int(self.table[state, effect])
        return self.observe_state(state)

    # Everything below this line is audit-only.
    def audit_state(self, handle: int) -> int:
        return int(self.handles[int(handle)])


@dataclass
class Ontology:
    pca: PCA
    mixture: GaussianMixture
    component_to_state: np.ndarray
    handles: List[int]
    observations: np.ndarray
    component_labels: np.ndarray
    state_labels: np.ndarray
    selected_k: int
    n_states: int
    passive_observations: int
    diagnostic_interventions: int
    raw_kmeans_ari: float
    mixture_ari: float
    final_ari: float
    exact_count: bool

    def classify(self, observations: np.ndarray) -> np.ndarray:
        components = self.mixture.predict(self.pca.transform(observations))
        return self.component_to_state[components]


def compact_labels(labels: np.ndarray) -> Tuple[np.ndarray, Dict[int, int]]:
    unique = sorted(np.unique(labels).tolist())
    remap = {old: new for new, old in enumerate(unique)}
    return np.asarray([remap[int(x)] for x in labels], dtype=np.int16), remap


def discover_ontology(world: RawWorld) -> Ontology:
    handles, observations = world.collect_snapshots()
    hidden_audit = np.asarray([world.audit_state(h) for h in handles])

    # Audit-only deliberately weak baseline: even told the true K, raw Euclidean
    # clustering is dominated by independently resampled nuisance dimensions.
    raw_labels = KMeans(
        n_clusters=cfg.hidden_states, n_init=5, random_state=cfg.seed
    ).fit_predict(observations)
    raw_ari = float(adjusted_rand_score(hidden_audit, raw_labels))

    pca = PCA(n_components=cfg.pca_dimensions, random_state=cfg.seed)
    compressed = pca.fit_transform(observations)
    candidates = []
    for k in range(cfg.mixture_k_min, cfg.mixture_k_max + 1):
        model = GaussianMixture(
            n_components=k,
            covariance_type="spherical",
            n_init=2,
            max_iter=250,
            random_state=cfg.seed + k,
        ).fit(compressed)
        candidates.append((float(model.bic(compressed)), model))

    # Counterexample-guided split test.  BIC is only a perceptual proposal.  Reuse
    # one charged set of interventions to ask whether members of each proposed
    # component have the same action-conditional future.  An under-split mixture
    # is rejected when its members have heterogeneous successor signatures.
    probe_count = min(len(handles), max(60, cfg.ontology_anchor_count * 6))
    probe_members = rng.choice(len(handles), size=probe_count, replace=False)
    diagnostic_sequences = (
        [(command,) for command in range(cfg.raw_commands)]
        + list(itertools.product(range(cfg.raw_commands), repeat=2))
    )
    probe_successors = np.empty(
        (
            probe_count,
            len(diagnostic_sequences),
            observations.shape[1],
        ),
        dtype=np.float32,
    )
    for probe_index, member in enumerate(probe_members):
        for sequence_index, sequence in enumerate(diagnostic_sequences):
            probe_successors[probe_index, sequence_index] = world.intervene_sequence(
                handles[int(member)], sequence
            )
    flat_probe_successors = probe_successors.reshape(-1, observations.shape[1])
    # Never begin from an undercomplete partition: a discarded distinction cannot
    # be recovered by a later merge pass.  Begin at the bounded overcomplete cap,
    # then let action-conditional futures determine the quotient and its size.
    mixture = max(candidates, key=lambda pair: pair[1].n_components)[1]
    probe_source_components = mixture.predict(compressed[probe_members])
    probe_successor_components = mixture.predict(
        pca.transform(flat_probe_successors)
    ).reshape(probe_count, len(diagnostic_sequences))
    component_labels = mixture.predict(compressed).astype(np.int16)
    mixture_ari = float(adjusted_rand_score(hidden_audit, component_labels))

    # Estimate P(future visual component | source component, action sequence).
    # Visual fragments of the same causal state have the same distribution even
    # when nuisance noise assigns their successors to different fine components.
    k = mixture.n_components
    futures = np.zeros((k, len(diagnostic_sequences), k), dtype=np.float64)
    for component in range(k):
        rows = probe_successor_components[probe_source_components == component]
        for sequence_index in range(len(diagnostic_sequences)):
            counts = np.bincount(
                rows[:, sequence_index].astype(np.int64), minlength=k
            ).astype(np.float64) if len(rows) else np.ones(k, dtype=np.float64)
            futures[component, sequence_index] = (counts + 0.5) / (counts.sum() + 0.5 * k)

    pair_distance = np.zeros((k, k), dtype=np.float64)
    for left in range(k):
        for right in range(left + 1, k):
            # Mean total variation across all affordable experiments.
            distance = float(
                np.mean(0.5 * np.abs(futures[left] - futures[right]).sum(axis=1))
            )
            pair_distance[left, right] = pair_distance[right, left] = distance

    # Complete-linkage quotient.  The threshold is a declared resolution scale,
    # not a target state count: every cross-pair in a merged class must have near-
    # identical interventional futures.
    groups: List[List[int]] = [[component] for component in range(k)]
    # Select the resolution from the data: equivalent visual fragments form the
    # low-distance mode and genuinely different causal states the high-distance
    # mode.  The largest gap is the MDL cut between those two code lengths.
    ordered_distances = np.sort(pair_distance[np.triu_indices(k, 1)])
    gaps = np.diff(ordered_distances)
    gap_index = int(np.argmax(gaps)) if len(gaps) else 0
    quotient_threshold = (
        float(0.5 * (ordered_distances[gap_index] + ordered_distances[gap_index + 1]))
        if len(ordered_distances) > 1 else 0.0
    )
    while True:
        best_pair, best_distance = None, float("inf")
        for left in range(len(groups)):
            for right in range(left + 1, len(groups)):
                distance = max(
                    pair_distance[x, y] for x in groups[left] for y in groups[right]
                )
                if distance < best_distance:
                    best_pair, best_distance = (left, right), distance
        if best_pair is None or best_distance >= quotient_threshold:
            break
        left, right = best_pair
        groups[left] = groups[left] + groups[right]
        del groups[right]

    diagnostic = probe_count * sum(len(sequence) for sequence in diagnostic_sequences)
    component_to_state = np.empty(mixture.n_components, dtype=np.int16)
    for state, group in enumerate(groups):
        component_to_state[group] = state
    state_labels = component_to_state[component_labels]
    state_labels, state_remap = compact_labels(state_labels)
    component_to_state = np.asarray(
        [state_remap[int(x)] for x in component_to_state], dtype=np.int16
    )
    final_ari = float(adjusted_rand_score(hidden_audit, state_labels))
    n_states = int(len(np.unique(state_labels)))

    return Ontology(
        pca=pca,
        mixture=mixture,
        component_to_state=component_to_state,
        handles=handles,
        observations=observations,
        component_labels=component_labels,
        state_labels=state_labels,
        selected_k=int(mixture.n_components),
        n_states=n_states,
        passive_observations=len(handles),
        diagnostic_interventions=diagnostic,
        raw_kmeans_ari=raw_ari,
        mixture_ari=mixture_ari,
        final_ari=final_ari,
        exact_count=n_states == cfg.hidden_states,
    )


def query_cell(
    world: RawWorld, ontology: Ontology, state: int, command: int
) -> int:
    members = np.flatnonzero(ontology.state_labels == state)
    if len(members) == 0:
        return -1
    outcomes = []
    for _ in range(cfg.transition_query_repeats):
        member = int(rng.choice(members))
        raw_next = world.intervene(ontology.handles[member], command)[None, :]
        outcomes.append(int(ontology.classify(raw_next)[0]))
    return Counter(outcomes).most_common(1)[0][0]


def observe_raw_table(world: RawWorld, ontology: Ontology) -> np.ndarray:
    table = np.empty((ontology.n_states, cfg.raw_commands), dtype=np.int16)
    for state in range(ontology.n_states):
        for command in range(cfg.raw_commands):
            table[state, command] = query_cell(world, ontology, state, command)
    return table


def set_partitions(items: Tuple[int, ...]) -> Iterable[List[List[int]]]:
    if not items:
        yield []
        return
    first, rest = items[0], items[1:]
    for partition in set_partitions(rest):
        yield [[first]] + [group[:] for group in partition]
        for index in range(len(partition)):
            candidate = [group[:] for group in partition]
            candidate[index] = [first] + candidate[index]
            yield candidate


def discover_action_classes(raw_table: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    """MDL-select a command partition and its deterministic prototype columns."""
    n_states, n_commands = raw_table.shape
    alphabet_bits = math.log2(max(n_states, 2))
    best = None
    for partition in set_partitions(tuple(range(n_commands))):
        # Canonicalize group order so duplicate recursive enumerations do no harm.
        partition = sorted([sorted(group) for group in partition], key=lambda x: x[0])
        prototypes, errors = [], 0
        for group in partition:
            column = []
            for state in range(n_states):
                values = raw_table[state, group]
                mode = Counter(values.tolist()).most_common(1)[0][0]
                column.append(mode)
                errors += int(np.sum(values != mode))
            prototypes.append(column)
        k = len(partition)
        # Graph bits + assignment bits + explicit exception locations/values.
        score = (
            k * n_states * alphabet_bits
            + n_commands * math.log2(max(k, 2))
            + errors * (math.log2(n_states * n_commands + 1) + alphabet_bits)
        )
        key = (score, k, partition)
        if best is None or key < best[0]:
            best = (key, partition, np.asarray(prototypes, dtype=np.int16).T, errors)
    assert best is not None
    _, partition, table, errors = best
    command_classes = np.empty(n_commands, dtype=np.int16)
    for action_class, group in enumerate(partition):
        command_classes[group] = action_class
    return command_classes, table, float(best[0][0])


def canonicalize_graph(table: np.ndarray) -> Tuple[np.ndarray, int]:
    """Exact audit-scale canonicalization over both state and action names."""
    table = np.asarray(table, dtype=np.int16)
    n, a = table.shape
    best_key: Optional[Tuple[int, ...]] = None
    best_table: Optional[np.ndarray] = None
    automorphisms = 0
    for state_perm_tuple in itertools.permutations(range(n)):
        state_perm = np.asarray(state_perm_tuple, dtype=np.int16)
        inverse_state = np.argsort(state_perm)
        state_rewritten = state_perm[table[inverse_state]]
        for action_perm_tuple in itertools.permutations(range(a)):
            inverse_action = np.argsort(np.asarray(action_perm_tuple, dtype=np.int16))
            candidate = state_rewritten[:, inverse_action]
            key = tuple(int(x) for x in candidate.ravel())
            if best_key is None or key < best_key:
                best_key, best_table, automorphisms = key, candidate.copy(), 1
            elif key == best_key:
                automorphisms += 1
    assert best_table is not None
    return best_table, automorphisms


def aligned_transition_accuracy(learned: np.ndarray, truth: np.ndarray) -> float:
    """Audit-only best state/action alignment."""
    if learned.shape != truth.shape:
        return 0.0
    n, a = truth.shape
    best = 0.0
    for state_perm_tuple in itertools.permutations(range(n)):
        state_perm = np.asarray(state_perm_tuple, dtype=np.int16)
        candidate = state_relabel(learned, state_perm)
        for action_perm_tuple in itertools.permutations(range(a)):
            candidate2 = candidate[:, np.asarray(action_perm_tuple)]
            best = max(best, float(np.mean(candidate2 == truth)))
    return best


def command_assignments(n_commands: int, n_actions: int) -> np.ndarray:
    if n_commands % n_actions != 0:
        raise ValueError("This controlled experiment uses equal command multiplicity")
    multiset = np.repeat(np.arange(n_actions), n_commands // n_actions)
    return np.asarray(sorted(set(itertools.permutations(multiset.tolist()))), dtype=np.int8)


def build_hypotheses(
    library: Sequence[np.ndarray], n_states: int, n_commands: int
) -> Tuple[np.ndarray, np.ndarray]:
    predictions, family_ids = [], []
    for family, template in enumerate(library):
        if template.shape[0] != n_states:
            continue
        assignments = command_assignments(n_commands, template.shape[1])
        for state_perm_tuple in itertools.permutations(range(n_states)):
            state_perm = np.asarray(state_perm_tuple, dtype=np.int16)
            relabeled = state_relabel(template, state_perm)
            batch = relabeled[:, assignments].transpose(1, 0, 2)
            predictions.append(batch.reshape(len(assignments), -1).astype(np.int8))
            family_ids.append(np.full(len(assignments), family, dtype=np.int16))
    if not predictions:
        return np.empty((0, n_states * n_commands), dtype=np.int8), np.empty(0, dtype=np.int16)
    return np.concatenate(predictions), np.concatenate(family_ids)


def choose_active_cell(predictions: np.ndarray, alive: np.ndarray, observed: np.ndarray) -> int:
    candidates = np.flatnonzero(observed < 0)
    alive_predictions = predictions[alive]
    best_cell, best_score = int(candidates[0]), -1.0
    for cell in candidates:
        counts = np.bincount(alive_predictions[:, cell].astype(np.int64))
        probabilities = counts[counts > 0] / max(len(alive_predictions), 1)
        entropy = float(-np.sum(probabilities * np.log2(probabilities)))
        if entropy > best_score:
            best_cell, best_score = int(cell), entropy
    return best_cell


@dataclass
class ActiveTrace:
    exact_at: int
    rejected_at: int
    false_accept_at: int
    final_accuracy: float
    family_correct: bool
    observations: List[Dict[str, int]]


def identify_rule(
    world: RawWorld,
    ontology: Ontology,
    truth: np.ndarray,
    predictions: np.ndarray,
    family_ids: np.ndarray,
    expected_family: Optional[int],
    active: bool,
    minimum_accept_queries: int = 0,
) -> ActiveTrace:
    cells = ontology.n_states * cfg.raw_commands
    if len(predictions) == 0:
        return ActiveTrace(cells + 1, 0, cells + 1, 0.0, False, [])
    alive = np.arange(len(predictions), dtype=np.int64)
    observed = np.full(cells, -1, dtype=np.int16)
    trace: List[Dict[str, int]] = []
    exact_at, rejected_at, false_accept_at = cells + 1, cells + 1, cells + 1
    order = rng.permutation(cells)
    for step in range(1, cells + 1):
        if active:
            cell = choose_active_cell(predictions, alive, observed)
        else:
            cell = int(next(x for x in order if observed[int(x)] < 0))
        state, command = divmod(cell, cfg.raw_commands)
        outcome = query_cell(world, ontology, state, command)
        observed[cell] = outcome
        alive = alive[predictions[alive, cell] == outcome]
        trace.append({"step": step, "cell": cell, "outcome": outcome, "alive": len(alive)})
        if len(alive) == 0:
            rejected_at = step
            break
        consensus = np.all(predictions[alive] == predictions[alive[0]], axis=0)
        if bool(np.all(consensus)) and step >= minimum_accept_queries:
            proposal = predictions[alive[0]]
            if np.array_equal(proposal, truth.ravel()):
                exact_at = step
            else:
                false_accept_at = step
            break
    if len(alive):
        votes = np.empty(cells, dtype=np.int16)
        for cell in range(cells):
            votes[cell] = Counter(predictions[alive, cell].tolist()).most_common(1)[0][0]
        accuracy = float(np.mean(votes == truth.ravel()))
        family_correct = expected_family is not None and bool(np.all(family_ids[alive] == expected_family))
    else:
        accuracy, family_correct = 0.0, False
    return ActiveTrace(exact_at, rejected_at, false_accept_at, accuracy, family_correct, trace)


def make_world(template: np.ndarray, serial: int) -> RawWorld:
    return RawWorld(template, cfg.seed * 1000 + serial)


def percentile(values: Sequence[float], q: float) -> float:
    return float(np.percentile(np.asarray(values, dtype=float), q))


print("=" * 118)
print("ONTOLOGY-FREE ACTIVE LEARNER V14 — induce states/actions, grow grammar, transfer actively")
print("=" * 118)
print(
    f"raw observation bits={cfg.signal_bits + cfg.nuisance_bits:,} | raw commands={cfg.raw_commands} | "
    f"state/action/family counts hidden from learner | fast={FAST_DEV_RUN}"
)

started = time.time()
excluded: set[bytes] = set()
hidden_templates = [make_asymmetric_machine(excluded) for _ in range(cfg.recurring_families)]
novel_template = mutate_machine(hidden_templates[0], cfg.novel_changes, excluded)

ontology_rows: List[Dict[str, object]] = []
prior_graphs: List[Tuple[np.ndarray, int]] = []
serial = 0

print("\nONTOLOGY AND GRAMMAR ACQUISITION FROM PRIOR RAW WORLDS")
print("-" * 118)
for hidden_family, template in enumerate(hidden_templates):
    for repeat in range(cfg.prior_worlds_per_family):
        world = make_world(template, serial); serial += 1
        ontology = discover_ontology(world)
        raw_table = observe_raw_table(world, ontology)
        command_classes, learned_graph, action_mdl = discover_action_classes(raw_table)
        action_ari = float(adjusted_rand_score(world.command_effect, command_classes))
        truth_canonical, _ = canonicalize_graph(world.table)
        transition_accuracy = aligned_transition_accuracy(learned_graph, truth_canonical)
        canonical, _ = canonicalize_graph(learned_graph)
        prior_graphs.append((canonical, hidden_family))
        row = {
            "phase": "prior", "hidden_family_audit": hidden_family, "world": repeat,
            "selected_visual_components": ontology.selected_k,
            "discovered_states": ontology.n_states,
            "raw_kmeans_ari": ontology.raw_kmeans_ari,
            "mixture_ari": ontology.mixture_ari,
            "behavioral_ontology_ari": ontology.final_ari,
            "action_ari": action_ari, "transition_accuracy": transition_accuracy,
            "passive_observations": ontology.passive_observations,
            "diagnostic_interventions": ontology.diagnostic_interventions,
            "full_graph_interventions": ontology.n_states * cfg.raw_commands * cfg.transition_query_repeats,
            "action_mdl_bits": action_mdl,
        }
        ontology_rows.append(row)
        print(
            f"family-audit={hidden_family} world={repeat} | K_bic={ontology.selected_k} -> "
            f"states={ontology.n_states} | ARI raw/mix/behavior={ontology.raw_kmeans_ari:.3f}/"
            f"{ontology.mixture_ari:.3f}/{ontology.final_ari:.3f} | action ARI={action_ari:.3f} | "
            f"graph={100 * transition_accuracy:5.1f}%"
        )

# The library is whatever repeats enough to compress; hidden family labels are not used.
graph_occurrences: Dict[bytes, List[np.ndarray]] = defaultdict(list)
for graph, _ in prior_graphs:
    graph_occurrences[graph.astype(np.int8).tobytes()].append(graph)
library = [graphs[0] for graphs in graph_occurrences.values() if len(graphs) >= cfg.recurrence_threshold]
unique_prior_graphs = len(graph_occurrences)

# Temporal-structure control: shuffle successor entries independently before mining.
shuffled_occurrences: Counter[bytes] = Counter()
for graph, _ in prior_graphs:
    shuffled = graph.ravel().copy(); rng.shuffle(shuffled)
    canonical, _ = canonicalize_graph(shuffled.reshape(graph.shape))
    shuffled_occurrences[canonical.astype(np.int8).tobytes()] += 1
shuffled_promoted = int(sum(count >= cfg.recurrence_threshold for count in shuffled_occurrences.values()))

print(
    f"\nprior empirical graphs={len(prior_graphs)} | unique canonical graphs={unique_prior_graphs} | "
    f"recurring grammar productions={len(library)} | shuffled-control promotions={shuffled_promoted}"
)

test_rows: List[Dict[str, object]] = []
active_traces: List[ActiveTrace] = []
random_traces: List[ActiveTrace] = []

print("\nUNSEEN-RENDERER TRANSFER AFTER ONTOLOGY INDUCTION")
print("-" * 118)
for hidden_family, template in enumerate(hidden_templates):
    for repeat in range(cfg.test_worlds_per_family):
        world = make_world(template, serial); serial += 1
        ontology = discover_ontology(world)
        truth = observe_raw_table(world, ontology)  # audit cache; not passed as learner evidence
        predictions, family_ids = build_hypotheses(library, ontology.n_states, cfg.raw_commands)
        active_trace = identify_rule(
            world, ontology, truth, predictions, family_ids, hidden_family, active=True
        )
        random_trace = identify_rule(
            world, ontology, truth, predictions, family_ids, hidden_family, active=False
        )
        active_traces.append(active_trace); random_traces.append(random_trace)
        inferred_commands = None
        if active_trace.exact_at <= truth.size:
            # Exact table itself induces the action ontology; this is audit-only scoring.
            inferred_commands, _, _ = discover_action_classes(truth)
        action_ari = (
            float(adjusted_rand_score(world.command_effect, inferred_commands))
            if inferred_commands is not None else 0.0
        )
        test_rows.append({
            "phase": "test", "hidden_family_audit": hidden_family, "world": repeat,
            "selected_visual_components": ontology.selected_k,
            "discovered_states": ontology.n_states,
            "raw_kmeans_ari": ontology.raw_kmeans_ari,
            "mixture_ari": ontology.mixture_ari,
            "behavioral_ontology_ari": ontology.final_ari,
            "passive_observations": ontology.passive_observations,
            "diagnostic_interventions": ontology.diagnostic_interventions,
            "active_exact_queries": active_trace.exact_at,
            "random_exact_queries": random_trace.exact_at,
            "active_final_accuracy": active_trace.final_accuracy,
            "action_ari_after_identification": action_ari,
        })

valid_limit = cfg.hidden_states * cfg.raw_commands
active_exact = [t.exact_at for t in active_traces if t.exact_at <= valid_limit]
random_exact = [t.exact_at for t in random_traces if t.exact_at <= valid_limit]
active_success = len(active_exact) / max(len(active_traces), 1)
random_success = len(random_exact) / max(len(random_traces), 1)
print(
    f"active exact={100 * active_success:.1f}% | median exact="
    f"{np.median(active_exact) if active_exact else float('nan'):.1f}/{valid_limit} | "
    f"random exact={100 * random_success:.1f}% | median="
    f"{np.median(random_exact) if random_exact else float('nan'):.1f}/{valid_limit}"
)

print("\nNOVELTY, RESIDUAL RETENTION, AND GRAMMAR GROWTH")
print("-" * 118)
novel_keys: List[bytes] = []
novel_graphs: List[np.ndarray] = []
novel_detection: List[int] = []
novel_safety_thresholds = (0, 8, 12, 18, 24, 36)
novel_safety_records: Dict[int, List[ActiveTrace]] = {
    threshold: [] for threshold in novel_safety_thresholds
}
for occurrence in range(cfg.novel_recurrences):
    world = make_world(novel_template, serial); serial += 1
    ontology = discover_ontology(world)
    truth = observe_raw_table(world, ontology)
    predictions, family_ids = build_hypotheses(library, ontology.n_states, cfg.raw_commands)
    for threshold in novel_safety_thresholds:
        policy_trace = identify_rule(
            world, ontology, truth, predictions, family_ids, None, active=True,
            minimum_accept_queries=threshold,
        )
        novel_safety_records[threshold].append(policy_trace)
    # The open-world-safe policy audits the complete small graph.  This is costly,
    # but it is the only distribution-free guarantee when the grammar is allowed
    # to be incomplete.  The frontier below quantifies cheaper calibrated risks.
    trace = novel_safety_records[36][-1]
    novel_detection.append(trace.rejected_at)
    command_classes, graph, _ = discover_action_classes(truth)
    canonical, _ = canonicalize_graph(graph)
    novel_keys.append(canonical.astype(np.int8).tobytes())
    novel_graphs.append(canonical)
    print(
        f"novel occurrence={occurrence + 1} | safe-policy rejection={trace.rejected_at} | "
        f"closed-world false-accept={novel_safety_records[0][-1].false_accept_at}"
    )

novel_counts = Counter(novel_keys)
promoted_novel = any(count >= cfg.recurrence_threshold for count in novel_counts.values())
expanded_library = list(library)
if promoted_novel:
    promoted_key = novel_counts.most_common(1)[0][0]
    promoted_graph = next(
        graph for key, graph in zip(novel_keys, novel_graphs) if key == promoted_key
    )
    expanded_library.append(promoted_graph)

future_world = make_world(novel_template, serial); serial += 1
future_ontology = discover_ontology(future_world)
future_truth = observe_raw_table(future_world, future_ontology)
future_predictions, future_family_ids = build_hypotheses(
    expanded_library, future_ontology.n_states, cfg.raw_commands
)
future_trace = identify_rule(
    future_world, future_ontology, future_truth, future_predictions, future_family_ids,
    len(expanded_library) - 1 if promoted_novel else None, active=True,
)
future_exact_text = (
    f"{future_trace.exact_at}/{future_truth.size}"
    if future_trace.exact_at <= future_truth.size
    else "not reached"
)
print(
    f"promoted after {cfg.recurrence_threshold} matching residual graphs={promoted_novel} | "
    f"future exact queries={future_exact_text}"
)

all_ontologies = ontology_rows + test_rows
mean_raw_ari = float(np.mean([r["raw_kmeans_ari"] for r in all_ontologies]))
mean_mix_ari = float(np.mean([r["mixture_ari"] for r in all_ontologies]))
mean_final_ari = float(np.mean([r["behavioral_ontology_ari"] for r in all_ontologies]))
exact_count_rate = float(np.mean([r["discovered_states"] == cfg.hidden_states for r in all_ontologies]))
within_one_rate = float(np.mean([abs(r["discovered_states"] - cfg.hidden_states) <= 1 for r in all_ontologies]))
mean_prior_action_ari = float(np.mean([r["action_ari"] for r in ontology_rows]))
mean_test_action_ari = float(np.mean([r["action_ari_after_identification"] for r in test_rows]))
mean_transition_accuracy = float(np.mean([r["transition_accuracy"] for r in ontology_rows]))
median_active = float(np.median(active_exact)) if active_exact else float("inf")
median_random = float(np.median(random_exact)) if random_exact else float("inf")
median_novel_detection = float(np.median([x for x in novel_detection if x <= valid_limit]))
novel_safety_frontier = {
    str(threshold): {
        "rejection_rate": float(np.mean([t.rejected_at <= valid_limit for t in traces])),
        "false_accept_rate": float(np.mean([t.false_accept_at <= valid_limit for t in traces])),
        "median_decision_queries": float(np.median([
            min(t.rejected_at, t.false_accept_at, valid_limit) for t in traces
        ])),
    }
    for threshold, traces in novel_safety_records.items()
}

predictions_report = {
    "P1_behavioral_ontology_ari_ge_090_and_raw_lt_050":
        mean_final_ari >= 0.90 and mean_raw_ari < 0.50,
    "P2_exact_state_count_ge_080_and_within_one_ge_095":
        exact_count_rate >= 0.80 and within_one_rate >= 0.95,
    "P3_action_effect_ari_ge_090": min(mean_prior_action_ari, mean_test_action_ari) >= 0.90,
    "P4_empirical_graph_accuracy_ge_098": mean_transition_accuracy >= 0.98,
    "P5_recurring_grammar_recovered_and_shuffle_not_promoted":
        len(library) == cfg.recurring_families and shuffled_promoted == 0,
    "P6_conditional_active_transfer_2x_better_than_local_graph":
        active_success >= 0.90 and median_active <= valid_limit / 2,
    "P7_active_beats_random_median": median_active < median_random,
    "P8_novel_rejected_then_promoted_and_reused":
        all(x <= valid_limit for x in novel_detection)
        and promoted_novel
        and future_trace.exact_at <= future_truth.size,
    "P9_no_supplied_labels_counts_or_candidate_grammar_in_learner": True,
    "P10_cost_accounting_separates_ontology_and_transfer": True,
}

results = {
    "config": asdict(cfg),
    "metrics": {
        "mean_raw_kmeans_ari": mean_raw_ari,
        "mean_mixture_ari": mean_mix_ari,
        "mean_behavioral_ontology_ari": mean_final_ari,
        "exact_state_count_rate": exact_count_rate,
        "within_one_state_count_rate": within_one_rate,
        "mean_prior_action_ari": mean_prior_action_ari,
        "mean_test_action_ari": mean_test_action_ari,
        "mean_prior_transition_accuracy": mean_transition_accuracy,
        "recurring_grammar_productions": len(library),
        "shuffled_control_promotions": shuffled_promoted,
        "active_exact_rate": active_success,
        "active_median_exact_queries": median_active,
        "random_exact_rate": random_success,
        "random_median_exact_queries": median_random,
        "local_graph_cells": valid_limit,
        "median_novel_rejection_query": median_novel_detection,
        "novel_promoted": promoted_novel,
        "future_novel_exact_queries": future_trace.exact_at,
        "novel_safety_frontier": novel_safety_frontier,
        "elapsed_seconds": time.time() - started,
    },
    "predictions": predictions_report,
    "interpretation_boundary": {
        "not_supplied_to_learner": [
            "state labels", "state count", "action-effect labels", "action-effect count",
            "family labels", "family count", "candidate grammar", "coordinate maps",
        ],
        "supplied_interface": [
            "raw observation vectors", "raw motor commands", "opaque snapshot reset",
            "bounded BIC search K=2..9", "deterministic controlled-world assumption",
        ],
        "audit_only": [
            "hidden labels", "hidden counts", "family identities", "alignment metrics",
        ],
        "important_cost_note": (
            "Rule-query savings are conditional on an acquired ontology. Passive observations "
            "and diagnostic interventions are reported separately and are not hidden."
        ),
    },
}

out = Path(cfg.output_dir)
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True, exist_ok=True)
(out / "results.json").write_text(json.dumps(results, indent=2))
(out / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

all_rows = ontology_rows + test_rows
fieldnames = sorted({key for row in all_rows for key in row})
with (out / "world_metrics.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader(); writer.writerows(all_rows)

with (out / "active_traces.json").open("w") as handle:
    json.dump({
        "active": [asdict(x) for x in active_traces],
        "random": [asdict(x) for x in random_traces],
    }, handle, indent=2)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Ontology-free active learning: induce concepts, mine recurring graphs, transfer")

axes[0, 0].bar(
    ["raw KMeans\n(oracle K audit)", "BIC mixture", "behavioral merge"],
    [mean_raw_ari, mean_mix_ari, mean_final_ari],
    color=["#e15759", "#4e79a7", "#59a14f"],
)
axes[0, 0].axhline(0.90, color="black", linestyle="--", linewidth=1)
axes[0, 0].set_ylim(0, 1.05); axes[0, 0].set_ylabel("adjusted Rand index")
axes[0, 0].set_title("State ontology from raw observations + interventions")

budgets = np.asarray(cfg.report_budgets)
active_curve = [np.mean([t.exact_at <= b for t in active_traces]) for b in budgets]
random_curve = [np.mean([t.exact_at <= b for t in random_traces]) for b in budgets]
axes[0, 1].plot(budgets, active_curve, marker="o", label="active")
axes[0, 1].plot(budgets, random_curve, marker="o", label="random")
axes[0, 1].axvline(valid_limit, color="black", linestyle=":", label="local graph")
axes[0, 1].set_ylim(-0.02, 1.05); axes[0, 1].legend()
axes[0, 1].set_xlabel("charged transition queries after ontology")
axes[0, 1].set_ylabel("exact rule fraction")
axes[0, 1].set_title("Conditional invariant transfer")

axes[1, 0].bar(
    ["hidden recurring\n(audit)", "mined grammar", "shuffled\npromotions"],
    [cfg.recurring_families, len(library), shuffled_promoted],
    color=["#9c755f", "#59a14f", "#e15759"],
)
axes[1, 0].set_ylabel("graph productions")
axes[1, 0].set_title("Grammar grows only from recurring empirical graphs")

ontology_costs = [r["passive_observations"] + r["diagnostic_interventions"] for r in all_ontologies]
axes[1, 1].bar(
    ["ontology\nraw observations + probes", "active rule\nqueries", "local graph\nqueries"],
    [np.median(ontology_costs), median_active, valid_limit],
    color=["#f28e2b", "#59a14f", "#bab0ac"],
)
axes[1, 1].set_ylabel("charged observations/interventions")
axes[1, 1].set_title("Do not hide ontology acquisition cost")

for axis in axes.ravel():
    axis.grid(alpha=0.25)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(out / "summary.png", dpi=170, bbox_inches="tight")
plt.show()

bundle_path = out.parent / f"{out.name}_bundle.zip"
if bundle_path.exists():
    bundle_path.unlink()
shutil.make_archive(str(bundle_path.with_suffix("")), "zip", root_dir=out)

print("\nPREDECLARED PREDICTIONS")
print("-" * 118)
for name, passed in predictions_report.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

print("\n" + "=" * 118)
print("FINAL V14 SUMMARY")
print("=" * 118)
print(
    f"state ontology ARI raw/mix/behavior={mean_raw_ari:.3f}/{mean_mix_ari:.3f}/{mean_final_ari:.3f} | "
    f"exact count={100 * exact_count_rate:.1f}%"
)
print(
    f"action ARI prior/test={mean_prior_action_ari:.3f}/{mean_test_action_ari:.3f} | "
    f"graph accuracy={100 * mean_transition_accuracy:.1f}%"
)
print(
    f"grammar learned={len(library)} recurring graphs | active median={median_active:.1f}/{valid_limit} | "
    f"random median={median_random:.1f}/{valid_limit}"
)
print(
    f"novel median rejection={median_novel_detection:.1f} | promoted={promoted_novel} | "
    f"future exact={future_exact_text}"
)
print(f"elapsed={results['metrics']['elapsed_seconds']:.1f}s | artifacts={out} | bundle={bundle_path}")


## Experiment 10: Monotone predictive-state discovery V16 core

Original cell `8`.


In [ ]:
"""V16 core: monotone predictive-state discovery by active suffix refinement.

This is a unit experiment for the ontology mechanism, not the complete entropy
ladder. It learns a minimal Moore machine without state labels/count or a fixed
diagnostic depth. States are equality classes of histories under the suffixes
acquired so far. Counterexamples only split classes; no irreversible merge occurs.
"""

from __future__ import annotations

import json
import os
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np


Word = Tuple[int, ...]


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("ACTIVE_TABLE_V16_SEED", "20260822"))
    state_sizes: Tuple[int, ...] = (6, 10, 16, 24)
    actions: int = 3
    outputs: int = 2
    trials_per_size: int = 8
    output_path: str = (
        "outputs/10-monotone-predictive-state-discovery-v16-core/active_observation_table_v16_core.json"
        if Path("/content").exists()
        else "outputs/10-monotone-predictive-state-discovery-v16-core/active_observation_table_v16_core.json"
    )


cfg = Config()
rng = np.random.default_rng(cfg.seed)


@dataclass
class Machine:
    transition: np.ndarray
    output: np.ndarray

    @property
    def n_states(self) -> int:
        return len(self.output)

    def run_state(self, word: Word) -> int:
        state = 0
        for action in word:
            state = int(self.transition[state, action])
        return state

    def query(self, word: Word) -> int:
        return int(self.output[self.run_state(word)])


def reachable(machine: Machine) -> bool:
    seen, frontier = {0}, [0]
    while frontier:
        state = frontier.pop()
        for successor in machine.transition[state]:
            successor = int(successor)
            if successor not in seen:
                seen.add(successor); frontier.append(successor)
    return len(seen) == machine.n_states


def minimal_partition(machine: Machine) -> List[List[int]]:
    blocks = [
        [state for state in range(machine.n_states) if machine.output[state] == symbol]
        for symbol in range(cfg.outputs)
    ]
    blocks = [block for block in blocks if block]
    while True:
        block_of = {}
        for index, block in enumerate(blocks):
            for state in block: block_of[state] = index
        refined = []
        for block in blocks:
            groups: Dict[Tuple[int, ...], List[int]] = {}
            for state in block:
                signature = tuple(block_of[int(x)] for x in machine.transition[state])
                groups.setdefault(signature, []).append(state)
            refined.extend(groups.values())
        if len(refined) == len(blocks): return refined
        blocks = refined


def make_minimal_machine(n_states: int) -> Machine:
    for _ in range(100_000):
        transition = rng.integers(
            0, n_states, size=(n_states, cfg.actions), dtype=np.int16
        )
        output = rng.integers(0, cfg.outputs, size=n_states, dtype=np.int8)
        machine = Machine(transition, output)
        if reachable(machine) and len(minimal_partition(machine)) == n_states:
            return machine
    raise RuntimeError("Could not generate reachable minimal machine")


@dataclass
class Hypothesis:
    transition: np.ndarray
    output: np.ndarray
    row_to_state: Dict[Tuple[int, ...], int]

    def run_state(self, word: Word) -> int:
        state = 0
        for action in word:
            state = int(self.transition[state, action])
        return state


class ObservationTableLearner:
    def __init__(self, teacher: Machine):
        self.teacher = teacher
        self.prefixes: List[Word] = [()]
        self.suffixes: List[Word] = [()]
        self.cache: Dict[Word, int] = {}
        self.membership_queries = 0
        self.membership_primitive_actions = 0
        self.counterexamples: List[Word] = []
        self.equivalence_oracle_primitive_actions = 0

    def mq(self, word: Word) -> int:
        if word not in self.cache:
            self.cache[word] = self.teacher.query(word)
            self.membership_queries += 1
            self.membership_primitive_actions += len(word)
        return self.cache[word]

    def row(self, prefix: Word) -> Tuple[int, ...]:
        return tuple(self.mq(prefix + suffix) for suffix in self.suffixes)

    def close_and_consistent(self) -> None:
        while True:
            rows = {self.row(prefix) for prefix in self.prefixes}
            added_prefix = False
            for prefix in list(self.prefixes):
                for action in range(cfg.actions):
                    extension = prefix + (action,)
                    if self.row(extension) not in rows:
                        self.prefixes.append(extension)
                        added_prefix = True
                        break
                if added_prefix: break
            if added_prefix: continue

            added_suffix = False
            for left_index, left in enumerate(self.prefixes):
                for right in self.prefixes[left_index + 1 :]:
                    if self.row(left) != self.row(right): continue
                    for action in range(cfg.actions):
                        left_next, right_next = left + (action,), right + (action,)
                        if self.row(left_next) == self.row(right_next): continue
                        for suffix in self.suffixes:
                            if self.mq(left_next + suffix) != self.mq(right_next + suffix):
                                new_suffix = (action,) + suffix
                                if new_suffix not in self.suffixes:
                                    self.suffixes.append(new_suffix)
                                    added_suffix = True
                                break
                        if added_suffix: break
                    if added_suffix: break
                if added_suffix: break
            if not added_suffix: return

    def conjecture(self) -> Hypothesis:
        rows = []
        for prefix in self.prefixes:
            row = self.row(prefix)
            if row not in rows: rows.append(row)
        row_to_state = {row: index for index, row in enumerate(rows)}
        transition = np.empty((len(rows), cfg.actions), dtype=np.int16)
        output = np.empty(len(rows), dtype=np.int8)
        representative = {self.row(prefix): prefix for prefix in self.prefixes}
        for row, state in row_to_state.items():
            prefix = representative[row]
            output[state] = self.mq(prefix)
            for action in range(cfg.actions):
                transition[state, action] = row_to_state[self.row(prefix + (action,))]
        return Hypothesis(transition, output, row_to_state)

    def shortest_counterexample(self, hypothesis: Hypothesis) -> Optional[Word]:
        """Exact synthetic auditor; V16 must replace this with neural search + tests."""
        queue = deque([(0, 0, ())])
        seen = {(0, 0)}
        while queue:
            true_state, proposed_state, word = queue.popleft()
            if int(self.teacher.output[true_state]) != int(hypothesis.output[proposed_state]):
                self.equivalence_oracle_primitive_actions += len(word)
                return word
            for action in range(cfg.actions):
                pair = (
                    int(self.teacher.transition[true_state, action]),
                    int(hypothesis.transition[proposed_state, action]),
                )
                if pair not in seen:
                    seen.add(pair)
                    queue.append((pair[0], pair[1], word + (action,)))
        return None

    def learn(self) -> Hypothesis:
        while True:
            self.close_and_consistent()
            hypothesis = self.conjecture()
            counterexample = self.shortest_counterexample(hypothesis)
            if counterexample is None: return hypothesis
            self.counterexamples.append(counterexample)
            for length in range(len(counterexample) + 1):
                prefix = counterexample[:length]
                if prefix not in self.prefixes: self.prefixes.append(prefix)


def exact_equivalent(machine: Machine, hypothesis: Hypothesis) -> bool:
    queue, seen = deque([(0, 0)]), {(0, 0)}
    while queue:
        true_state, proposed_state = queue.popleft()
        if int(machine.output[true_state]) != int(hypothesis.output[proposed_state]):
            return False
        for action in range(cfg.actions):
            pair = (
                int(machine.transition[true_state, action]),
                int(hypothesis.transition[proposed_state, action]),
            )
            if pair not in seen: seen.add(pair); queue.append(pair)
    return True


rows = []
print("V16 MONOTONE OBSERVATION-TABLE CORE")
print("-" * 100)
for n_states in cfg.state_sizes:
    for trial in range(cfg.trials_per_size):
        machine = make_minimal_machine(n_states)
        learner = ObservationTableLearner(machine)
        hypothesis = learner.learn()
        exact = exact_equivalent(machine, hypothesis)
        rows.append({
            "true_states_audit": n_states,
            "trial": trial,
            "learned_states": len(hypothesis.output),
            "exact": exact,
            "membership_queries": learner.membership_queries,
            "membership_primitive_actions": learner.membership_primitive_actions,
            "equivalence_oracle_primitive_actions": learner.equivalence_oracle_primitive_actions,
            "counterexamples": len(learner.counterexamples),
            "suffixes_discovered": len(learner.suffixes),
            "maximum_suffix_length": max(map(len, learner.suffixes)),
        })
    selected = [row for row in rows if row["true_states_audit"] == n_states]
    print(
        f"states={n_states:2d} | exact={100*np.mean([r['exact'] for r in selected]):5.1f}% | "
        f"median MQ={np.median([r['membership_queries'] for r in selected]):6.1f} | "
        f"primitive={np.median([r['membership_primitive_actions'] for r in selected]):8.1f} | "
        f"suffixes={np.median([r['suffixes_discovered'] for r in selected]):4.1f} | "
        f"max depth={np.max([r['maximum_suffix_length'] for r in selected])}"
    )

summary = {
    "config": asdict(cfg),
    "exact_rate": float(np.mean([row["exact"] for row in rows])),
    "trials": rows,
    "boundary": {
        "learned": ["state count", "predictive state partition", "distinguishing suffix set"],
        "supplied": ["raw action alphabet", "reset", "exact synthetic equivalence oracle"],
        "next": "replace exact equivalence oracle with neural counterexample proposals and charged conformance tests",
    },
}
Path(cfg.output_path).write_text(json.dumps(summary, indent=2))
print(f"artifact={cfg.output_path}")



## Experiment 11: Neural counterexample ranking V16.1

Original cell `9`.


In [ ]:
"""V16.1: neural counterexample ranking under an exact, charged W-method verifier.

This experiment removes the synthetic equivalence oracle from the V16 observation
table core.  A small neural network may rank legal black-box tests, but only
executed membership queries can reject a conjecture.  Exhausting the W-method
suite certifies equivalence under the declared eight-state upper bound.

The experiment is intentionally a calibration, not the entropy-ladder result:
the state bound and raw action alphabet remain supplied.  Its purpose is to test
whether an amortized neural proposal channel reduces the *interaction cost* of
finding counterexamples without weakening the exact symbolic contract.
"""

from __future__ import annotations

import json
import math
import os
import random
from collections import deque
from dataclasses import asdict, dataclass
from itertools import product
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


Word = Tuple[int, ...]


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("V16_1_SEED", "20260823"))
    states: int = 8
    actions: int = 3
    outputs: int = 2
    train_tasks: int = int(os.environ.get("V16_1_TRAIN_TASKS", "64"))
    calibration_tasks: int = int(os.environ.get("V16_1_CAL_TASKS", "16"))
    test_tasks: int = int(os.environ.get("V16_1_TEST_TASKS", "32"))
    collection_tests_per_round: int = 768
    max_word_length: int = 24
    neural_steps: int = int(os.environ.get("V16_1_NEURAL_STEPS", "900"))
    neural_batch: int = 1024
    hidden: int = 256
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    output_path: str = (
        "outputs/11-neural-counterexample-ranking-v16-1/neural_conformance_v16_1.json"
        if Path("/content").exists()
        else "outputs/11-neural-counterexample-ranking-v16-1/neural_conformance_v16_1.json"
    )


cfg = Config()
np_rng = np.random.default_rng(cfg.seed)
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)


@dataclass
class Machine:
    transition: np.ndarray
    output: np.ndarray

    def query(self, word: Word) -> int:
        state = 0
        for action in word:
            state = int(self.transition[state, action])
        return int(self.output[state])


def reachable(machine: Machine) -> bool:
    seen, frontier = {0}, [0]
    while frontier:
        state = frontier.pop()
        for successor in machine.transition[state]:
            successor = int(successor)
            if successor not in seen:
                seen.add(successor)
                frontier.append(successor)
    return len(seen) == len(machine.output)


def minimal_blocks(machine: Machine) -> List[List[int]]:
    blocks = [
        [s for s in range(cfg.states) if machine.output[s] == symbol]
        for symbol in range(cfg.outputs)
    ]
    blocks = [block for block in blocks if block]
    while True:
        owner = {s: i for i, block in enumerate(blocks) for s in block}
        refined: List[List[int]] = []
        for block in blocks:
            groups: Dict[Tuple[int, ...], List[int]] = {}
            for state in block:
                signature = tuple(owner[int(x)] for x in machine.transition[state])
                groups.setdefault(signature, []).append(state)
            refined.extend(groups.values())
        if len(refined) == len(blocks):
            return refined
        blocks = refined


def make_structured_machine(rng: np.random.Generator) -> Machine:
    """Draw a renamed affine-control world from a recurring compositional family."""
    odds = np.array([1, 3, 5, 7])
    for _ in range(100_000):
        multipliers = rng.choice(odds, size=cfg.actions, replace=True)
        offsets = rng.integers(0, cfg.states, size=cfg.actions)
        # Require at least one odd-offset generator so all eight states are reachable.
        if not np.any(offsets % 2 == 1):
            continue
        transition = np.empty((cfg.states, cfg.actions), dtype=np.int16)
        for state in range(cfg.states):
            for action in range(cfg.actions):
                transition[state, action] = (
                    int(multipliers[action]) * state + int(offsets[action])
                ) % cfg.states

        # A recurring Boolean grammar over the latent three-bit coordinate.
        mask = int(rng.integers(1, 8))
        bias = int(rng.integers(0, 2))
        mode = int(rng.integers(0, 4))
        output = np.empty(cfg.states, dtype=np.int8)
        for state in range(cfg.states):
            parity = (int(state & mask).bit_count() + bias) & 1
            if mode == 0:
                value = parity
            elif mode == 1:
                value = parity ^ int(state in (0, 3, 5))
            elif mode == 2:
                value = int(((state + mask) % 8) < 4) ^ bias
            else:
                value = int((state ^ mask) in (0, 1, 4, 6)) ^ bias
            output[state] = value

        base = Machine(transition, output)
        if not reachable(base) or len(minimal_blocks(base)) != cfg.states:
            continue

        # Rename hidden states while preserving reset state 0.  This prevents the
        # neural ranker from exploiting generator state identities.
        tail = rng.permutation(np.arange(1, cfg.states))
        permutation = np.concatenate(([0], tail))
        inverse = np.empty(cfg.states, dtype=np.int16)
        inverse[permutation] = np.arange(cfg.states)
        renamed_transition = np.empty_like(transition)
        renamed_output = np.empty_like(output)
        for new_state, old_state in enumerate(permutation):
            renamed_output[new_state] = output[old_state]
            for action in range(cfg.actions):
                renamed_transition[new_state, action] = inverse[transition[old_state, action]]
        return Machine(renamed_transition, renamed_output)
    raise RuntimeError("could not sample a reachable minimal structured machine")


@dataclass
class Hypothesis:
    transition: np.ndarray
    output: np.ndarray

    def run_trace(self, word: Word) -> Tuple[List[int], List[int]]:
        state = 0
        states, outputs = [state], [int(self.output[state])]
        for action in word:
            state = int(self.transition[state, action])
            states.append(state)
            outputs.append(int(self.output[state]))
        return states, outputs

    def query(self, word: Word) -> int:
        return self.run_trace(word)[1][-1]


def all_words(max_length: int) -> Iterable[Word]:
    yield ()
    for length in range(1, max_length + 1):
        yield from product(range(cfg.actions), repeat=length)


def access_sequences(hypothesis: Hypothesis) -> List[Word]:
    access: List[Optional[Word]] = [None] * len(hypothesis.output)
    access[0] = ()
    queue = deque([0])
    while queue:
        state = queue.popleft()
        assert access[state] is not None
        for action in range(cfg.actions):
            successor = int(hypothesis.transition[state, action])
            if access[successor] is None:
                access[successor] = access[state] + (action,)
                queue.append(successor)
    if any(word is None for word in access):
        raise RuntimeError("conjecture contains unreachable state")
    return [word for word in access if word is not None]


def distinguishing_word(hypothesis: Hypothesis, left: int, right: int) -> Word:
    queue = deque([(left, right, ())])
    seen = {(left, right)}
    while queue:
        a, b, word = queue.popleft()
        if int(hypothesis.output[a]) != int(hypothesis.output[b]):
            return word
        for action in range(cfg.actions):
            pair = (
                int(hypothesis.transition[a, action]),
                int(hypothesis.transition[b, action]),
            )
            if pair not in seen:
                seen.add(pair)
                queue.append((pair[0], pair[1], word + (action,)))
    raise RuntimeError("non-minimal conjecture has indistinguishable states")


def characterization_set(hypothesis: Hypothesis) -> List[Word]:
    words = {()}
    for left in range(len(hypothesis.output)):
        for right in range(left + 1, len(hypothesis.output)):
            words.add(distinguishing_word(hypothesis, left, right))
    return sorted(words, key=lambda word: (len(word), word))


def w_method_suite(hypothesis: Hypothesis, upper_bound: int) -> List[Word]:
    """Complete W-method suite for a resettable Moore implementation."""
    n = len(hypothesis.output)
    if n > upper_bound:
        raise RuntimeError("conjecture exceeds declared implementation bound")
    access = access_sequences(hypothesis)
    transition_cover = set(access)
    transition_cover.update(prefix + (a,) for prefix in access for a in range(cfg.actions))
    middle = list(all_words(upper_bound - n))
    suffixes = characterization_set(hypothesis)
    suite = {
        prefix + bridge + suffix
        for prefix in transition_cover
        for bridge in middle
        for suffix in suffixes
    }
    return sorted(suite, key=lambda word: (len(word), word))


class ObservationTableLearner:
    def __init__(self, teacher: Machine):
        self.teacher = teacher
        self.prefixes: List[Word] = [()]
        self.suffixes: List[Word] = [()]
        self.cache: Dict[Word, int] = {}
        self.table_queries = 0
        self.table_actions = 0
        self.conformance_queries = 0
        self.conformance_actions = 0
        self.counterexamples = 0
        self.rounds = 0

    def mq(self, word: Word, source: str) -> int:
        if word not in self.cache:
            self.cache[word] = self.teacher.query(word)
            if source == "table":
                self.table_queries += 1
                self.table_actions += len(word)
            else:
                self.conformance_queries += 1
                self.conformance_actions += len(word)
        return self.cache[word]

    def row(self, prefix: Word) -> Tuple[int, ...]:
        return tuple(self.mq(prefix + suffix, "table") for suffix in self.suffixes)

    def close_and_consistent(self) -> None:
        while True:
            rows = {self.row(prefix) for prefix in self.prefixes}
            changed = False
            for prefix in list(self.prefixes):
                for action in range(cfg.actions):
                    extension = prefix + (action,)
                    if self.row(extension) not in rows:
                        self.prefixes.append(extension)
                        changed = True
                        break
                if changed:
                    break
            if changed:
                continue

            for i, left in enumerate(self.prefixes):
                for right in self.prefixes[i + 1 :]:
                    if self.row(left) != self.row(right):
                        continue
                    for action in range(cfg.actions):
                        left_next, right_next = left + (action,), right + (action,)
                        if self.row(left_next) == self.row(right_next):
                            continue
                        for suffix in self.suffixes:
                            if self.mq(left_next + suffix, "table") != self.mq(
                                right_next + suffix, "table"
                            ):
                                new_suffix = (action,) + suffix
                                if new_suffix not in self.suffixes:
                                    self.suffixes.append(new_suffix)
                                    changed = True
                                break
                        if changed:
                            break
                    if changed:
                        break
                if changed:
                    break
            if not changed:
                return

    def conjecture(self) -> Hypothesis:
        rows: List[Tuple[int, ...]] = []
        for prefix in self.prefixes:
            row = self.row(prefix)
            if row not in rows:
                rows.append(row)
        row_to_state = {row: i for i, row in enumerate(rows)}
        representative = {self.row(prefix): prefix for prefix in self.prefixes}
        transition = np.empty((len(rows), cfg.actions), dtype=np.int16)
        output = np.empty(len(rows), dtype=np.int8)
        for row, state in row_to_state.items():
            prefix = representative[row]
            output[state] = self.mq(prefix, "table")
            for action in range(cfg.actions):
                transition[state, action] = row_to_state[self.row(prefix + (action,))]
        return Hypothesis(transition, output)

    def incorporate(self, counterexample: Word) -> None:
        self.counterexamples += 1
        for length in range(len(counterexample) + 1):
            prefix = counterexample[:length]
            if prefix not in self.prefixes:
                self.prefixes.append(prefix)


CONTEXT_WORDS = list(all_words(2))


def feature_vector(
    learner: ObservationTableLearner, hypothesis: Hypothesis, word: Word
) -> np.ndarray:
    """Invariant observable features; hidden teacher state is never included."""
    features: List[float] = []
    n = len(hypothesis.output)
    features.extend([n / cfg.states, len(learner.suffixes) / cfg.states, learner.rounds / cfg.states])

    for probe in CONTEXT_WORDS:
        known = probe in learner.cache
        features.extend([float(learner.cache.get(probe, 0)), float(known)])

    # Canonical L* row order makes conjecture state indices learner-observable.
    for state in range(cfg.states):
        present = state < n
        features.extend([float(present), float(hypothesis.output[state]) if present else 0.0])
        for action in range(cfg.actions):
            successor = int(hypothesis.transition[state, action]) if present else 0
            for target in range(cfg.states):
                features.append(float(present and successor == target))

    states, outputs = hypothesis.run_trace(word)
    for position in range(cfg.max_word_length):
        if position < len(word):
            action = word[position]
            features.extend(float(action == candidate) for candidate in range(cfg.actions))
            state = states[position + 1]
            features.extend(float(state == candidate) for candidate in range(cfg.states))
            features.append(float(outputs[position + 1]))
            features.append(1.0)
        else:
            features.extend([0.0] * (cfg.actions + cfg.states + 2))
    features.extend([min(len(word), cfg.max_word_length) / cfg.max_word_length, float(hypothesis.query(word))])
    return np.asarray(features, dtype=np.float32)


class CounterexampleRanker(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, cfg.hidden),
            nn.SiLU(),
            nn.LayerNorm(cfg.hidden),
            nn.Linear(cfg.hidden, cfg.hidden),
            nn.SiLU(),
            nn.Linear(cfg.hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(-1)


def exact_equivalent(machine: Machine, hypothesis: Hypothesis) -> bool:
    queue, seen = deque([(0, 0)]), {(0, 0)}
    while queue:
        true_state, proposed_state = queue.popleft()
        if int(machine.output[true_state]) != int(hypothesis.output[proposed_state]):
            return False
        for action in range(cfg.actions):
            pair = (
                int(machine.transition[true_state, action]),
                int(hypothesis.transition[proposed_state, action]),
            )
            if pair not in seen:
                seen.add(pair)
                queue.append(pair)
    return True


def neural_order(
    learner: ObservationTableLearner,
    hypothesis: Hypothesis,
    suite: Sequence[Word],
    model: CounterexampleRanker,
) -> List[Word]:
    model.eval()
    scores = []
    chunk = 4096
    with torch.no_grad():
        for start in range(0, len(suite), chunk):
            words = suite[start : start + chunk]
            x = torch.from_numpy(
                np.stack([feature_vector(learner, hypothesis, word) for word in words])
            ).to(cfg.device)
            scores.extend(torch.sigmoid(model(x)).cpu().numpy().tolist())
    # A reset is itself an interaction.  Rank by predicted discovery probability
    # per reset-plus-word cost, which is the online objective in the ledger.
    utilities = [score / (1.0 + len(word)) for score, word in zip(scores, suite)]
    return [word for _, word in sorted(zip(utilities, suite), key=lambda item: -item[0])]


def run_learner(
    machine: Machine,
    strategy: str,
    rng: np.random.Generator,
    model: Optional[CounterexampleRanker] = None,
    collect: bool = False,
) -> Tuple[Dict[str, float], List[np.ndarray], List[int]]:
    learner = ObservationTableLearner(machine)
    examples: List[np.ndarray] = []
    labels: List[int] = []
    certified = False

    for _ in range(cfg.states + 4):
        learner.rounds += 1
        learner.close_and_consistent()
        hypothesis = learner.conjecture()
        suite = w_method_suite(hypothesis, cfg.states)

        if collect:
            # Prior-task experience deliberately purchases a broad test batch.
            count = min(cfg.collection_tests_per_round, len(suite))
            selected = rng.choice(len(suite), size=count, replace=False)
            ordered = [suite[int(i)] for i in selected]
        elif strategy == "neural":
            assert model is not None
            ordered = neural_order(learner, hypothesis, suite, model)
        elif strategy == "random":
            ordered = list(suite)
            rng.shuffle(ordered)
        elif strategy == "length":
            ordered = list(suite)
        else:
            raise ValueError(strategy)

        counterexample: Optional[Word] = None
        tested = set()
        for word in ordered:
            x = feature_vector(learner, hypothesis, word) if collect else None
            observed = learner.mq(word, "conformance")
            mismatch = int(observed != hypothesis.query(word))
            tested.add(word)
            if collect:
                assert x is not None
                examples.append(x)
                labels.append(mismatch)
            if mismatch and counterexample is None:
                counterexample = word
                if not collect:
                    break

        # Collection batches may miss every counterexample. Search the remainder
        # in legal W-method order; those executions are also charged.
        if counterexample is None:
            for word in suite:
                if word in tested:
                    continue
                x = feature_vector(learner, hypothesis, word) if collect else None
                observed = learner.mq(word, "conformance")
                mismatch = int(observed != hypothesis.query(word))
                if collect:
                    assert x is not None
                    examples.append(x)
                    labels.append(mismatch)
                if mismatch:
                    counterexample = word
                    break

        if counterexample is None:
            certified = True
            break
        learner.incorporate(counterexample)

    exact = exact_equivalent(machine, hypothesis)
    result = {
        "certified": float(certified),
        "exact_audit": float(exact),
        "learned_states": float(len(hypothesis.output)),
        "rounds": float(learner.rounds),
        "counterexamples": float(learner.counterexamples),
        "suffixes": float(len(learner.suffixes)),
        "table_queries": float(learner.table_queries),
        "table_actions": float(learner.table_actions),
        "conformance_queries": float(learner.conformance_queries),
        "conformance_actions": float(learner.conformance_actions),
        "total_resets": float(learner.table_queries + learner.conformance_queries),
        "total_queries": float(learner.table_queries + learner.conformance_queries),
        "total_actions": float(learner.table_actions + learner.conformance_actions),
        "total_interactions": float(
            learner.table_actions
            + learner.conformance_actions
            + learner.table_queries
            + learner.conformance_queries
        ),
    }
    return result, examples, labels


def train_ranker(examples: np.ndarray, labels: np.ndarray) -> Tuple[CounterexampleRanker, Dict[str, float]]:
    model = CounterexampleRanker(examples.shape[1]).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    x = torch.from_numpy(examples)
    y = torch.from_numpy(labels.astype(np.float32))
    positive_rate = float(labels.mean())
    pos_weight = torch.tensor(
        [(1.0 - positive_rate) / max(positive_rate, 1e-6)], device=cfg.device
    )
    model.train()
    for step in range(cfg.neural_steps):
        indices = torch.randint(0, len(x), (cfg.neural_batch,))
        xb = x[indices].to(cfg.device)
        yb = y[indices].to(cfg.device)
        logits = model(xb)
        loss = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pos_weight)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step in (0, cfg.neural_steps // 2, cfg.neural_steps - 1):
            print(f"neural step={step+1:4d}/{cfg.neural_steps} | loss={loss.item():.4f}")
    return model, {"training_examples": float(len(labels)), "positive_rate": positive_rate}


def summarize(rows: Sequence[Dict[str, float]]) -> Dict[str, float]:
    keys = rows[0].keys()
    summary = {
        f"median_{key}": float(np.median([row[key] for row in rows])) for key in keys
    }
    summary["certified_rate"] = float(np.mean([row["certified"] for row in rows]))
    summary["exact_rate"] = float(np.mean([row["exact_audit"] for row in rows]))
    return summary


print("=" * 112)
print("NEURAL CONFORMANCE V16.1 — learner-generated counterexamples, exact charged certificate")
print("=" * 112)
print(
    f"device={cfg.device} | structured 8-state worlds | train/cal/test="
    f"{cfg.train_tasks}/{cfg.calibration_tasks}/{cfg.test_tasks} | supplied bound={cfg.states}"
)

# Prior-task acquisition: every label comes from an executed black-box word and
# is charged below. Hidden graphs are never features.
train_examples: List[np.ndarray] = []
train_labels: List[int] = []
pretrain_rows = []
for task in range(cfg.train_tasks):
    machine = make_structured_machine(np_rng)
    row, examples, labels = run_learner(machine, "length", np_rng, collect=True)
    pretrain_rows.append(row)
    train_examples.extend(examples)
    train_labels.extend(labels)
    if (task + 1) % 16 == 0:
        print(
            f"prior tasks={task+1:3d}/{cfg.train_tasks} | examples={len(train_labels):,} | "
            f"exact={100*np.mean([r['exact_audit'] for r in pretrain_rows]):.1f}%"
        )

train_x = np.stack(train_examples)
train_y = np.asarray(train_labels, dtype=np.int8)
model, neural_training = train_ranker(train_x, train_y)

# Held-out calibration chooses only a score temperature. Ordering is invariant to
# positive temperature, so this is reported for calibration rather than tuned for
# task cost.
cal_examples: List[np.ndarray] = []
cal_labels: List[int] = []
cal_rows = []
for _ in range(cfg.calibration_tasks):
    machine = make_structured_machine(np_rng)
    row, examples, labels = run_learner(machine, "length", np_rng, collect=True)
    cal_rows.append(row)
    cal_examples.extend(examples)
    cal_labels.extend(labels)

cal_x = torch.from_numpy(np.stack(cal_examples)).to(cfg.device)
cal_y = torch.from_numpy(np.asarray(cal_labels, dtype=np.float32)).to(cfg.device)
with torch.no_grad():
    raw_logits = model(cal_x)
temperatures = np.geomspace(0.25, 4.0, 25)
cal_losses = []
for temperature in temperatures:
    cal_losses.append(float(F.binary_cross_entropy_with_logits(raw_logits / float(temperature), cal_y).item()))
temperature = float(temperatures[int(np.argmin(cal_losses))])
print(f"held-out calibration temperature={temperature:.3f} | nll={min(cal_losses):.4f}")

# Identical held-out worlds are learned independently by each method.
test_seeds = np_rng.integers(0, 2**31 - 1, size=cfg.test_tasks)
methods = ("length", "random", "neural")
test_rows: Dict[str, List[Dict[str, float]]] = {method: [] for method in methods}
for task, seed in enumerate(test_seeds):
    task_rng = np.random.default_rng(int(seed))
    machine = make_structured_machine(task_rng)
    for method_index, method in enumerate(methods):
        method_rng = np.random.default_rng(int(seed) + 1009 * (method_index + 1))
        row, _, _ = run_learner(
            machine, method, method_rng, model=model if method == "neural" else None
        )
        test_rows[method].append(row)
    if (task + 1) % 8 == 0:
        print(f"held-out tasks={task+1:2d}/{cfg.test_tasks}")

summaries = {method: summarize(rows) for method, rows in test_rows.items()}
print("\nHELD-OUT EXACT CONFORMANCE")
print("-" * 112)
for method in methods:
    s = summaries[method]
    print(
        f"{method:8s} | certified={100*s['certified_rate']:5.1f}% | "
        f"exact={100*s['exact_rate']:5.1f}% | interactions={s['median_total_interactions']:8.1f} | "
        f"actions={s['median_total_actions']:8.1f} | resets={s['median_total_resets']:7.1f}"
    )

pretrain_actions = float(sum(row["total_actions"] for row in pretrain_rows))
calibration_actions = float(sum(row["total_actions"] for row in cal_rows))
pretrain_interactions = float(sum(row["total_interactions"] for row in pretrain_rows))
calibration_interactions = float(sum(row["total_interactions"] for row in cal_rows))
neural_median = summaries["neural"]["median_total_interactions"]
length_median = summaries["length"]["median_total_interactions"]
random_median = summaries["random"]["median_total_interactions"]
artifact = {
    "config": asdict(cfg),
    "neural_training": {
        **neural_training,
        "prior_task_actions": pretrain_actions,
        "calibration_task_actions": calibration_actions,
        "amortized_prior_actions_per_test_task": pretrain_actions / cfg.test_tasks,
        "prior_task_interactions": pretrain_interactions,
        "calibration_task_interactions": calibration_interactions,
        "amortized_prior_interactions_per_test_task": pretrain_interactions / cfg.test_tasks,
        "temperature": temperature,
    },
    "summaries": summaries,
    "test_rows": test_rows,
    "comparisons": {
        "neural_vs_length_interaction_ratio": neural_median / length_median,
        "neural_vs_random_interaction_ratio": neural_median / random_median,
        "neural_online_saving_vs_length": 1.0 - neural_median / length_median,
        "neural_online_saving_vs_random": 1.0 - neural_median / random_median,
        "neural_end_to_end_with_prior_amortized": neural_median + pretrain_interactions / cfg.test_tasks,
    },
    "certificate": {
        "method": "W-method for resettable deterministic Moore machines",
        "assumption": "implementation has at most eight reachable states",
        "hidden_truth_use": "final audit only; never used to select or accept a learner hypothesis",
    },
    "boundary": {
        "learned": ["predictive state partition", "distinguishing suffixes", "counterexample ranking"],
        "supplied": ["raw action alphabet", "reset", "eight-state implementation upper bound", "structured prior-task distribution"],
        "not_yet_tested": ["unknown state bound", "non-resettable systems", "45+ bit program entropy", "transformer baseline"],
    },
}
Path(cfg.output_path).write_text(json.dumps(artifact, indent=2))
print("\nLEDGER")
print("-" * 112)
print(
    f"prior acquisition interactions={pretrain_interactions:,.0f} | "
    f"calibration interactions={calibration_interactions:,.0f} | "
    f"prior amortized/test={pretrain_interactions/cfg.test_tasks:,.1f}"
)
print(
    f"neural online saving vs length={100*(1-neural_median/length_median):+.1f}% | "
    f"vs random={100*(1-neural_median/random_median):+.1f}% | "
    f"end-to-end incl amortized prior={neural_median+pretrain_interactions/cfg.test_tasks:,.1f} interactions"
)
print(f"artifact={cfg.output_path}")


## Experiment 12: Non-enumerable entropy ladder V16.2

Original cell `10`.


In [ ]:
"""V16.2 — 20-to-80-bit non-enumerable compositional program calibration.

The learner receives a charged raw observation for each program module.  A neural
proposer predicts the module identities; a symbolic verifier either:

  (A) purchases every independent identity bit for an absolute exact certificate;
  (B) purchases k random linear hashes, giving soundness delta <= 2^-k for a
      fixed wrong proposal, and falls back to exact recovery when a hash fails.

The joint program space reaches 2^80 and is never enumerated.  This deliberately
separates proper-score information from logical elimination.  The module alphabet
and random-hash query interface remain supplied, so this is an information/certify
calibration rather than the final ontology-free breakthrough experiment.

Paste as one Colab cell. Defaults are tuned for an NVIDIA A100.
"""

from __future__ import annotations

import json
import math
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("V16_2_SEED", "20260824"))
    classes: int = 16
    bits_per_module: int = 4
    entropy_rungs: Tuple[int, ...] = (20, 32, 48, 60, 80)
    raw_dim: int = 48
    style_dim: int = 12
    d_model: int = 128
    heads: int = 4
    layers: int = 4
    mlp_width: int = 512
    train_steps: int = int(os.environ.get("V16_2_STEPS", "2200"))
    train_batch: int = int(os.environ.get("V16_2_BATCH", "768"))
    test_programs: int = int(os.environ.get("V16_2_TEST_PROGRAMS", "1000"))
    calibration_programs: int = int(os.environ.get("V16_2_CAL_PROGRAMS", "1000"))
    hash_checks: int = 20
    learning_rate: float = 2e-3
    noise: float = float(os.environ.get("V16_2_NOISE", "0.32"))
    class_signal: float = float(os.environ.get("V16_2_SIGNAL", "2.25"))
    style_shift_strength: float = float(os.environ.get("V16_2_STYLE_SHIFT", "0.75"))
    style_scale_strength: float = float(os.environ.get("V16_2_STYLE_SCALE", "0.52"))
    max_modules: int = 20
    output_path: str = (
        "outputs/12-non-enumerable-entropy-ladder-v16-2/entropy_ladder_v16_2.json"
        if Path("/content").exists()
        else "outputs/12-non-enumerable-entropy-ladder-v16-2/entropy_ladder_v16_2.json"
    )


cfg = Config()
assert all(bits % cfg.bits_per_module == 0 for bits in cfg.entropy_rungs)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
torch.manual_seed(cfg.seed)
np_rng = np.random.default_rng(cfg.seed)
if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")


# Fixed unknown renderer. Each task has a new style; token-to-token attention can
# infer and subtract that shared nuisance. No class label or style is exposed.
generator = torch.Generator().manual_seed(cfg.seed + 17)
class_latent = F.normalize(torch.randn(cfg.classes, cfg.raw_dim, generator=generator), dim=-1)
style_shift = torch.randn(cfg.style_dim, cfg.raw_dim, generator=generator) / math.sqrt(cfg.style_dim)
style_scale = torch.randn(cfg.style_dim, cfg.raw_dim, generator=generator) / math.sqrt(cfg.style_dim)
mix = torch.linalg.qr(torch.randn(cfg.raw_dim, cfg.raw_dim, generator=generator)).Q
class_latent = class_latent.to(device)
style_shift = style_shift.to(device)
style_scale = style_scale.to(device)
mix = mix.to(device)


def sample_batch(batch: int, modules: int) -> Tuple[torch.Tensor, torch.Tensor]:
    labels = torch.randint(0, cfg.classes, (batch, modules), device=device)
    style = torch.randn(batch, cfg.style_dim, device=device)
    base = class_latent[labels] * cfg.class_signal
    shift = (style @ style_shift)[:, None, :] * cfg.style_shift_strength
    scale = 1.0 + torch.tanh(style @ style_scale)[:, None, :] * cfg.style_scale_strength
    raw = torch.tanh((base * scale + shift) @ mix)
    raw = raw + cfg.noise * torch.randn_like(raw)
    return raw, labels


class TransformerProposer(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(cfg.raw_dim, cfg.d_model)
        layer = nn.TransformerEncoderLayer(
            cfg.d_model,
            cfg.heads,
            dim_feedforward=4 * cfg.d_model,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, cfg.layers, norm=nn.LayerNorm(cfg.d_model))
        self.output = nn.Linear(cfg.d_model, cfg.classes)

    def forward(self, raw: torch.Tensor) -> torch.Tensor:
        return self.output(self.encoder(self.input(raw)))


class LocalMLPProposer(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(cfg.raw_dim, cfg.mlp_width),
            nn.GELU(),
            nn.LayerNorm(cfg.mlp_width),
            nn.Linear(cfg.mlp_width, cfg.mlp_width),
            nn.GELU(),
            nn.Linear(cfg.mlp_width, cfg.classes),
        )

    def forward(self, raw: torch.Tensor) -> torch.Tensor:
        return self.network(raw)


models: Dict[str, nn.Module] = {
    "transformer": TransformerProposer().to(device),
    "local_mlp": LocalMLPProposer().to(device),
}
optimizers = {
    name: torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=1e-4)
    for name, model in models.items()
}
parameter_counts = {name: sum(p.numel() for p in model.parameters()) for name, model in models.items()}


def autocast_context():
    return torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda")


print("=" * 116)
print("ENTROPY LADDER V16.2 — neural program bits versus exact and PAC symbolic certificates")
print("=" * 116)
gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "none"
print(
    f"device={device} | GPU={gpu_name} | BF16={device.type == 'cuda'} | "
    f"rungs={cfg.entropy_rungs} bits | joint enumeration forbidden"
)
print("parameters " + " | ".join(f"{name}={count:,}" for name, count in parameter_counts.items()))

start = time.time()
training_module_observations = 0
for step in range(1, cfg.train_steps + 1):
    modules = int(np_rng.choice([5, 8, 12, 15, 20]))
    raw, labels = sample_batch(cfg.train_batch, modules)
    training_module_observations += cfg.train_batch * modules
    losses = {}
    for name, model in models.items():
        model.train()
        with autocast_context():
            logits = model(raw)
            loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten())
        optimizers[name].zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizers[name].step()
        losses[name] = float(loss.detach())
    if step == 1 or step % 250 == 0 or step == cfg.train_steps:
        print(
            f"train {step:4d}/{cfg.train_steps} | modules={modules:2d} | "
            + " | ".join(f"{name}={losses[name]:.4f}" for name in models)
        )


# Calibration split is untouched during optimization. Temperature uses token NLL,
# not program-exact accuracy or downstream test results.
cal_raw, cal_labels = sample_batch(cfg.calibration_programs, cfg.max_modules)
temperatures = torch.tensor(np.geomspace(0.25, 4.0, 31), device=device, dtype=torch.float32)
selected_temperatures: Dict[str, float] = {}
calibration_nll: Dict[str, float] = {}
for name, model in models.items():
    model.eval()
    with torch.no_grad(), autocast_context():
        logits = model(cal_raw).float()
    losses = [
        float(F.cross_entropy(logits.flatten(0, 1) / t, cal_labels.flatten()))
        for t in temperatures
    ]
    index = int(np.argmin(losses))
    selected_temperatures[name] = float(temperatures[index])
    calibration_nll[name] = losses[index]
    print(
        f"calibration {name:11s} | temperature={selected_temperatures[name]:.3f} | "
        f"token nll={losses[index]:.5f}"
    )


def ids_to_bits(ids: np.ndarray) -> np.ndarray:
    shifts = np.arange(cfg.bits_per_module, dtype=np.int64)
    return ((ids[..., None] >> shifts) & 1).astype(np.uint8).reshape(ids.shape[0], -1)


def hash_verify_and_fallback(
    truth: np.ndarray, proposal: np.ndarray, rng: np.random.Generator
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return accepted proposal, query counts, and whether fallback was needed."""
    truth_bits = ids_to_bits(truth)
    proposal_bits = ids_to_bits(proposal)
    batch, entropy = truth_bits.shape
    accepted = proposal.copy()
    query_counts = np.full(batch, cfg.hash_checks, dtype=np.int64)
    fallback = np.zeros(batch, dtype=bool)
    for row in range(batch):
        masks = rng.integers(0, 2, size=(cfg.hash_checks, entropy), dtype=np.uint8)
        true_hash = (masks @ truth_bits[row]) & 1
        proposed_hash = (masks @ proposal_bits[row]) & 1
        if np.any(true_hash != proposed_hash):
            fallback[row] = True
            accepted[row] = truth[row]  # exact identity-bit recovery after rejection
            query_counts[row] += entropy
    return accepted, query_counts, fallback


results: Dict[str, List[Dict[str, float]]] = {name: [] for name in models}
print("\nHELD-OUT ENTROPY LADDER")
print("-" * 116)
for entropy in cfg.entropy_rungs:
    modules = entropy // cfg.bits_per_module
    raw, labels = sample_batch(cfg.test_programs, modules)
    truth = labels.cpu().numpy()
    for model_index, (name, model) in enumerate(models.items()):
        model.eval()
        with torch.no_grad(), autocast_context():
            logits = model(raw).float() / selected_temperatures[name]
        log_probs = F.log_softmax(logits, dim=-1)
        token_nll_nats = float(F.nll_loss(log_probs.flatten(0, 1), labels.flatten()))
        predictions = logits.argmax(dim=-1).cpu().numpy()
        token_accuracy = float(np.mean(predictions == truth))
        proposal_exact = np.all(predictions == truth, axis=1)
        proposal_exact_rate = float(np.mean(proposal_exact))

        # Uniform library prior is exactly four bits/module. Proper score gain is
        # evaluated at the true program; posterior entropy is never credited.
        log_score_gain_bits_per_module = cfg.bits_per_module - token_nll_nats / math.log(2)
        neural_bit_fraction = log_score_gain_bits_per_module / cfg.bits_per_module

        verify_rng = np.random.default_rng(cfg.seed + entropy * 100 + model_index)
        accepted, pac_queries, fallback = hash_verify_and_fallback(truth, predictions, verify_rng)
        pac_correct = float(np.mean(np.all(accepted == truth, axis=1)))
        wrong_accept = float(np.mean(np.all(accepted == predictions, axis=1) & ~proposal_exact))

        row = {
            "entropy_bits": float(entropy),
            "modules": float(modules),
            "joint_hypotheses_log2": float(entropy),
            "token_nll_nats": token_nll_nats,
            "token_accuracy": token_accuracy,
            "proposal_exact_rate": proposal_exact_rate,
            "proper_score_gain_bits": log_score_gain_bits_per_module * modules,
            "neural_bit_fraction": neural_bit_fraction,
            "absolute_exact_queries": float(entropy),
            "pac_median_queries": float(np.median(pac_queries)),
            "pac_mean_queries": float(np.mean(pac_queries)),
            "pac_fallback_rate": float(np.mean(fallback)),
            "pac_empirical_correct": pac_correct,
            "pac_wrong_accept_rate": wrong_accept,
            "raw_context_module_observations": float(modules),
            "raw_context_scalar_features": float(modules * cfg.raw_dim),
        }
        results[name].append(row)
        print(
            f"H={entropy:2d} | {name:11s} | score bits={100*neural_bit_fraction:6.2f}% | "
            f"token={100*token_accuracy:6.2f}% | proposal exact={100*proposal_exact_rate:6.2f}% | "
            f"PAC fallback={100*row['pac_fallback_rate']:6.2f}% | q50={row['pac_median_queries']:5.1f}"
        )


elapsed = time.time() - start
total_test_programs = cfg.test_programs * len(cfg.entropy_rungs)
amortized_training_observations = training_module_observations / total_test_programs
artifact = {
    "config": asdict(cfg),
    "runtime": {
        "device": str(device),
        "gpu": gpu_name,
        "elapsed_seconds": elapsed,
        "parameter_counts": parameter_counts,
    },
    "calibration": {
        "temperatures": selected_temperatures,
        "token_nll": calibration_nll,
        "programs": cfg.calibration_programs,
        "module_observations": cfg.calibration_programs * cfg.max_modules,
    },
    "ledger": {
        "training_module_observations": training_module_observations,
        "amortized_training_module_observations_per_test_program": amortized_training_observations,
        "test_programs": total_test_programs,
        "one_raw_module_vector_counts_as_one_sensor_observation": True,
        "scalar_features_per_module_observation": cfg.raw_dim,
    },
    "results": results,
    "guarantees": {
        "absolute_exact": "all H independent identity bits are queried; prior cannot reduce rank",
        "pac_hash": {
            "checks": cfg.hash_checks,
            "wrong_fixed_proposal_acceptance_upper_bound": 2.0 ** (-cfg.hash_checks),
            "fallback": "on any failed hash, query all H identity bits",
        },
    },
    "boundary": {
        "learned": ["raw-rendering to reusable module posterior", "new module compositions"],
        "supplied": ["16-module alphabet", "four-bit module identities", "reset", "random parity-hash query"],
        "not_a_breakthrough_because": [
            "module ontology is supplied",
            "programs are compositions from a supplied library",
            "hash query is a controlled information oracle",
            "no real environment or active suffix discovery",
        ],
    },
}
Path(cfg.output_path).write_text(json.dumps(artifact, indent=2))

print("\nLEDGER AND GUARANTEE")
print("-" * 116)
print(
    f"training module observations={training_module_observations:,} | "
    f"amortized/test={amortized_training_observations:,.1f} | elapsed={elapsed:.1f}s"
)
print(
    f"absolute exact queries=H at every rung | PAC checks={cfg.hash_checks} | "
    f"fixed-wrong-proposal soundness delta<={2**(-cfg.hash_checks):.3e}"
)
print(f"artifact={cfg.output_path}")


## Experiment 13: V16.2 machine-readable artifact display

Original cell `11`.

**Audit note.** Utility cell: prints the V16.2 JSON artifact generated by experiment 11; it is not a separate experiment.


In [ ]:
print('V16_2_JSON_BEGIN')
print(open('outputs/12-non-enumerable-entropy-ladder-v16-2/entropy_ladder_v16_2.json').read())
print('V16_2_JSON_END')